# Module 1

In [1]:
"""
01_00_metadata_config.txt
Phase 3 -- Module 01: Metadata Encoder -- Configuration

Single place for every path, category list, and layout constant the metadata
encoder uses. Nothing else in Module 01 hardcodes these -- change them here.

WHAT THE 20-VALUE VECTOR LOOKS LIKE (fixed order, never reshuffled):
  index 0        age_z            age rescaled with the TRAINING fold's own mean/std
  index 1        age_recorded     1 = age was recorded, 0 = age was missing
  index 2-4      sex_*            one-hot: male, female, unknown
  index 5-19     site_*           one-hot: 15 anatomical sites, alphabetical

SITE SOURCE (locked, Option A): the `localization` column of
cv_fold_assignments.csv -- one site per lesion, the same value the Phase 1
splits were stratified on. NOT the raw per-image site in the metadata file.

NAMING NOTE: variables here use distinct names (P1_ARTIFACTS_DIR,
PHASE3_ARTIFACTS_DIR) on purpose, so they can never collide with an
ARTIFACTS_DIR defined in another cell of the same notebook kernel -- the same
silent-overwrite trap Phase 2 hit and fixed.
"""

from pathlib import Path

# --- Paths -----------------------------------------------------------------
BASE_DIR = Path(r"T:\Thesis - Development")

P1_ARTIFACTS_DIR = BASE_DIR / "Resources" / "Artifacts" / "Phase 1 - P2"
FOLD_CSV_PATH = P1_ARTIFACTS_DIR / "cv_fold_assignments.csv"
HOLDOUT_CSV_PATH = P1_ARTIFACTS_DIR / "holdout_split.csv"
METADATA_TAB_PATH = BASE_DIR / "Resources" / "Dataset" / "HAM10000" / "HAM10000_metadata.tab"

# Placeholder folder name -- rename freely, everything below follows this variable.
PHASE3_ARTIFACTS_DIR = BASE_DIR / "Resources" / "Artifacts" / "Phase 3 - P1"
PHASE3_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
ENCODER_STATS_PATH = PHASE3_ARTIFACTS_DIR / "metadata_encoder_stats.json"

# --- Layout ----------------------------------------------------------------
AGE_FEATURE_NAMES = ["age_z", "age_recorded"]

# Values exactly as they appear in HAM10000_metadata.tab (lowercase).
SEX_CATEGORIES = ["male", "female", "unknown"]

# The 15 HAM10000 sites, alphabetical, exactly as spelled in the data.
SITE_CATEGORIES = [
    "abdomen", "acral", "back", "chest", "ear", "face", "foot", "genital",
    "hand", "lower extremity", "neck", "scalp", "trunk", "unknown",
    "upper extremity",
]

SEX_OFFSET = len(AGE_FEATURE_NAMES)                 # 2
SITE_OFFSET = SEX_OFFSET + len(SEX_CATEGORIES)      # 5

FEATURE_NAMES = (
    AGE_FEATURE_NAMES
    + [f"sex_{c}" for c in SEX_CATEGORIES]
    + [f"site_{c}" for c in SITE_CATEGORIES]
)
METADATA_DIM = len(FEATURE_NAMES)
assert METADATA_DIM == 20, f"Expected a 20-value metadata vector, got {METADATA_DIM}"

# --- Encoding rules --------------------------------------------------------
AGE_STD_DDOF = 0   # population std (divide by n) -- the usual z-score convention

# --- Known dataset facts, used only by the verification cell ---------------
EXPECTED_POOL_ROWS = 8583   # CV working pool from Phase 1 (10,015 minus the 1,432 holdout)
N_FOLDS = 5

print("Metadata encoder config loaded.")
print(f"  Vector width      : {METADATA_DIM}")
print(f"  Site source       : fold CSV 'localization' (Option A)")
print(f"  Fold CSV          : {FOLD_CSV_PATH}")
print(f"  Encoder stats out : {ENCODER_STATS_PATH}")

Metadata encoder config loaded.
  Vector width      : 20
  Site source       : fold CSV 'localization' (Option A)
  Fold CSV          : T:\Thesis - Development\Resources\Artifacts\Phase 1 - P2\cv_fold_assignments.csv
  Encoder stats out : T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\metadata_encoder_stats.json


In [2]:
"""
01_01_metadata_encoder.txt
Phase 3 -- Module 01: Metadata Encoder -- Functions (reusable, not one-time)

Requires 01_00_metadata_config.txt already run in the same notebook.

WHAT THIS DOES
  1. load_fold_table_with_metadata()  -- fold table + age + sex, joined on image_id
  2. compute_fold_age_stats()         -- a fold's age mean/std from TRAINING rows only
  3. encode_metadata()                -- turns rows into the 20-value vector
  4. build / save / load encoder stats -- the small JSON file that lets Phase 5
     rescale BCN20000 ages with the TRAINING fold's mean/std (never BCN20000's own)
  5. get_fold_split_with_metadata()   -- convenience for the Dataset (Module 02):
     a fold's train or val rows plus their encoded vectors, row-aligned

KEY RULE: the validation fold is encoded with the TRAINING folds' age mean/std.
Age statistics never see validation rows or the Phase 1 holdout.

JSON STRUCTURE (metadata_encoder_stats.json) -- rename any key here if you want:
  {
    "description": "...",
    "feature_names": [20 names, in vector order],
    "metadata_dim": 20,
    "sex_categories": [...],
    "site_categories": [...],
    "age_std_ddof": 0,
    "age_missing_rule": "...",
    "source_fold_csv": "<path>",
    "folds": { "0": {"mean": ..., "std": ..., "n_recorded": ..., "n_train_images": ...}, ... "4": {...} }
  }
"""

import json
import os
import tempfile

import numpy as np
import pandas as pd

_SEX_INDEX = {c: i for i, c in enumerate(SEX_CATEGORIES)}
_SITE_INDEX = {c: i for i, c in enumerate(SITE_CATEGORIES)}


def load_fold_table_with_metadata() -> pd.DataFrame:
    """Fold table (image_id, lesion_id, dx, localization, strat_label, cv_fold)
    plus `age` and `sex` from the metadata file, matched on image_id.

    Row order is exactly the fold CSV's order (a left join keeps it), so any
    later filter on cv_fold matches how Phase 2's Dataset ordered its rows.
    Raises if a row is lost, duplicated, or has an unrecognised sex/site value."""
    fold_df = pd.read_csv(FOLD_CSV_PATH)
    meta_df = pd.read_csv(METADATA_TAB_PATH, sep="\t")[["image_id", "age", "sex"]]

    n_before = len(fold_df)
    merged = fold_df.merge(meta_df, on="image_id", how="left", validate="one_to_one")

    if len(merged) != n_before:
        raise ValueError(f"Join changed the row count: {n_before} -> {len(merged)}")
    if merged["sex"].isna().any():
        raise ValueError(f"{int(merged['sex'].isna().sum())} fold-table image_id(s) "
                         f"have no match in the metadata file")

    bad_sex = set(merged["sex"].unique()) - set(SEX_CATEGORIES)
    if bad_sex:
        raise ValueError(f"Unrecognised sex value(s): {sorted(bad_sex)}")
    bad_site = set(merged["localization"].unique()) - set(SITE_CATEGORIES)
    if bad_site:
        raise ValueError(f"Unrecognised site value(s): {sorted(bad_site)}")

    return merged


def compute_fold_age_stats(df: pd.DataFrame, fold_num: int) -> dict:
    """Age mean/std for one fold, from the TRAINING rows only
    (every fold except fold_num) and only where age was recorded.
    Computed over images, since each image is one training sample."""
    is_train = df["cv_fold"] != fold_num
    train_ages = df.loc[is_train, "age"].dropna().to_numpy(dtype=np.float64)
    if len(train_ages) == 0:
        raise ValueError(f"Fold {fold_num}: no recorded ages in the training rows")

    mean = float(train_ages.mean())
    std = float(train_ages.std(ddof=AGE_STD_DDOF))
    if not std > 0:
        raise ValueError(f"Fold {fold_num}: age std is {std}, cannot z-score")

    return {
        "mean": mean,
        "std": std,
        "n_recorded": int(len(train_ages)),
        "n_train_images": int(is_train.sum()),
    }


def encode_metadata(df: pd.DataFrame, age_mean: float, age_std: float) -> np.ndarray:
    """Rows -> float32 array of shape (len(df), 20), row-aligned with df.

    Missing age: age_z = 0.0 (the training mean after z-scoring), age_recorded = 0.
    The mean/std passed in must come from the TRAINING folds (see
    compute_fold_age_stats) -- for validation rows too."""
    n = len(df)
    out = np.zeros((n, METADATA_DIM), dtype=np.float32)

    ages = df["age"].to_numpy(dtype=np.float64)
    recorded = ~np.isnan(ages)
    out[recorded, 0] = (ages[recorded] - age_mean) / age_std
    out[:, 1] = recorded.astype(np.float32)

    sex_idx = df["sex"].map(_SEX_INDEX)
    if sex_idx.isna().any():
        raise ValueError("encode_metadata: unrecognised sex value")
    out[np.arange(n), SEX_OFFSET + sex_idx.to_numpy(dtype=np.int64)] = 1.0

    site_idx = df["localization"].map(_SITE_INDEX)
    if site_idx.isna().any():
        raise ValueError("encode_metadata: unrecognised site value")
    out[np.arange(n), SITE_OFFSET + site_idx.to_numpy(dtype=np.int64)] = 1.0

    return out


def build_encoder_stats_record(df: pd.DataFrame) -> dict:
    """All five folds' age stats plus the layout, ready to save as JSON."""
    folds = {}
    for fold_num in sorted(int(f) for f in df["cv_fold"].unique()):
        folds[str(fold_num)] = compute_fold_age_stats(df, fold_num)

    return {
        "description": ("Phase 3 metadata encoder statistics. Age mean/std come from each "
                        "fold's TRAINING rows only. Use these -- not the new dataset's own "
                        "statistics -- when encoding any other dataset (e.g. BCN20000)."),
        "feature_names": list(FEATURE_NAMES),
        "metadata_dim": METADATA_DIM,
        "sex_categories": list(SEX_CATEGORIES),
        "site_categories": list(SITE_CATEGORIES),
        "age_std_ddof": AGE_STD_DDOF,
        "age_missing_rule": ("age_z set to 0.0 (the training mean after z-scoring) "
                             "and age_recorded set to 0"),
        "source_fold_csv": str(FOLD_CSV_PATH),
        "folds": folds,
    }


def save_encoder_stats(record: dict, path=None):
    """Atomic write (tmp file + os.replace), same pattern as Phase 2's
    run_config writer, so a power cut can't leave a half-written JSON."""
    path = path or ENCODER_STATS_PATH
    out_dir = os.path.dirname(str(path)) or "."
    os.makedirs(out_dir, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=out_dir, suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_path, path)
    except Exception:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise
    print(f"Encoder stats saved: {path}")


def load_encoder_stats(path=None) -> dict:
    """Loads the JSON and refuses to continue if its layout no longer matches
    01_00's layout (guards against loading stats built for a different vector)."""
    path = path or ENCODER_STATS_PATH
    with open(path, "r", encoding="utf-8") as f:
        record = json.load(f)
    if record["feature_names"] != list(FEATURE_NAMES):
        raise ValueError("Saved encoder stats use a different feature layout than 01_00")
    return record


def get_fold_age_stats(record: dict, fold_num: int):
    """(mean, std) for a fold, read from a loaded stats record."""
    fold = record["folds"][str(int(fold_num))]
    return fold["mean"], fold["std"]


def get_fold_split_with_metadata(df: pd.DataFrame, fold_num: int, split: str, record: dict):
    """A fold's train or val rows and their encoded vectors, row-aligned.
    Both splits are encoded with the TRAINING folds' age mean/std.
    Row order matches Phase 2's Dataset: filter the fold table, reset the index."""
    assert split in ("train", "val")
    mean, std = get_fold_age_stats(record, fold_num)
    if split == "train":
        sub = df[df["cv_fold"] != fold_num].reset_index(drop=True)
    else:
        sub = df[df["cv_fold"] == fold_num].reset_index(drop=True)
    return sub, encode_metadata(sub, mean, std)

In [3]:
"""
01_02_metadata_verification.txt
Phase 3 -- Module 01: Metadata Encoder -- Verification (run once)

Requires 01_00 and 01_01 already run in the same notebook.

All seven checks run as a full set -- every failure is collected and reported
together, not stopped at the first one -- and the cell raises RuntimeError at
the end if anything failed. Same convention as Phase 1's verification cells
and Phase 2's preflight (00_04).

Checks:
  1. Join integrity   -- row count matches Phase 1's pool, image_ids unique,
                         no Phase 1 holdout image is in the fold table
  2. Missing values   -- INFORMATIONAL: age nulls and unknown-sex counts, and
                         whether they are the same records (closes the open
                         "57 and 57" flag from the Phase 1 log)
  3. Leakage test     -- each fold's age stats recomputed by an independent
                         route, then again after the fold's own validation ages
                         are replaced with an absurd value; stats must not move
  4. Vector structure -- shape, dtype, finite values, one-hot blocks sum to 1,
                         flag matches age presence, missing age encodes as 0
  5. Z-score sanity   -- recorded ages in a fold's training rows come out with
                         mean ~0 and std ~1
  6. Round trip       -- decoding the one-hot blocks gives back the original
                         sex and site for every row
  7. JSON round trip  -- saved stats reload identical, and the layout guard works

If run with save_stats=True (default) the JSON is written to ENCODER_STATS_PATH.
"""

import numpy as np
import pandas as pd


def run_metadata_verification(save_stats: bool = True):
    failures = []
    print("=" * 60)
    print("METADATA ENCODER VERIFICATION -- Phase 3, Module 01")
    print("=" * 60)

    df = load_fold_table_with_metadata()

    # --- 1. Join integrity ---
    print("\n[1/7] Join integrity...")
    ok = True
    if len(df) != EXPECTED_POOL_ROWS:
        failures.append(f"Join integrity: expected {EXPECTED_POOL_ROWS} rows, got {len(df)}")
        ok = False
    if df["image_id"].duplicated().any():
        failures.append("Join integrity: duplicate image_id values in the fold table")
        ok = False
    fold_values = set(int(f) for f in df["cv_fold"].unique())
    if fold_values != set(range(N_FOLDS)):
        failures.append(f"Join integrity: cv_fold values are {sorted(fold_values)}, expected 0-{N_FOLDS - 1}")
        ok = False
    holdout_ids = set(pd.read_csv(HOLDOUT_CSV_PATH)["image_id"])
    overlap = holdout_ids & set(df["image_id"])
    if overlap:
        failures.append(f"Join integrity: {len(overlap)} holdout image(s) are inside the fold table")
        ok = False
    if ok:
        print(f"  OK -- {len(df)} rows, unique image_ids, folds 0-{N_FOLDS - 1}, "
              f"none of the {len(holdout_ids)} holdout images present")
    else:
        print("  FAILED -- see summary")

    # --- 2. Missing values (informational) ---
    print("\n[2/7] Missing values (informational)...")
    pool_age_null = df["age"].isna()
    pool_sex_unknown = df["sex"] == "unknown"
    print(f"  Fold table: age null = {int(pool_age_null.sum())}, sex unknown = {int(pool_sex_unknown.sum())}, "
          f"both = {int((pool_age_null & pool_sex_unknown).sum())}")
    full_meta = pd.read_csv(METADATA_TAB_PATH, sep="\t")
    full_age_null = full_meta["age"].isna()
    full_sex_unknown = full_meta["sex"] == "unknown"
    same_records = set(full_meta.loc[full_age_null, "image_id"]) == set(full_meta.loc[full_sex_unknown, "image_id"])
    print(f"  Full metadata file: age null = {int(full_age_null.sum())}, sex unknown = {int(full_sex_unknown.sum())}, "
          f"both = {int((full_age_null & full_sex_unknown).sum())}")
    print(f"  Are the age-null and sex-unknown records the SAME records? {same_records}")

    # --- 3. Leakage test ---
    print("\n[3/7] Leakage test...")
    leak_ok = True
    for fold_num in range(N_FOLDS):
        stats = compute_fold_age_stats(df, fold_num)

        # independent route: plain numpy over raw arrays, no pandas filtering helper
        ages_all = df["age"].to_numpy(dtype=np.float64)
        folds_all = df["cv_fold"].to_numpy()
        train_mask = (folds_all != fold_num) & ~np.isnan(ages_all)
        ind_mean = ages_all[train_mask].mean()
        ind_std = ages_all[train_mask].std(ddof=AGE_STD_DDOF)
        if not (np.isclose(stats["mean"], ind_mean, rtol=0, atol=1e-9)
                and np.isclose(stats["std"], ind_std, rtol=0, atol=1e-9)):
            failures.append(f"Leakage test: fold {fold_num} stats differ from the independent recomputation")
            leak_ok = False

        # poison test: make this fold's own validation ages absurd, stats must not change
        poisoned = df.copy()
        poisoned.loc[poisoned["cv_fold"] == fold_num, "age"] = 1.0e6
        poisoned_stats = compute_fold_age_stats(poisoned, fold_num)
        if poisoned_stats["mean"] != stats["mean"] or poisoned_stats["std"] != stats["std"]:
            failures.append(f"Leakage test: fold {fold_num} stats changed when its validation ages were altered")
            leak_ok = False
    if leak_ok:
        print(f"  OK -- all {N_FOLDS} folds match the independent recomputation and ignore their own validation ages")
    else:
        print("  FAILED -- see summary")

    # --- 4 to 6. Vector structure, z-score sanity, round trip ---
    print("\n[4-6/7] Vector structure, z-score sanity, round trip...")
    print(f"  {'fold':>4} {'age mean':>9} {'age std':>8} {'n recorded':>11} {'train z mean':>13} {'train z std':>12}")
    struct_ok = True
    for fold_num in range(N_FOLDS):
        stats = compute_fold_age_stats(df, fold_num)
        X = encode_metadata(df, stats["mean"], stats["std"])

        # 4. structure
        if X.shape != (len(df), METADATA_DIM) or X.dtype != np.float32:
            failures.append(f"Vector structure: fold {fold_num} shape/dtype is {X.shape}/{X.dtype}")
            struct_ok = False
        if not np.isfinite(X).all():
            failures.append(f"Vector structure: fold {fold_num} contains NaN or inf")
            struct_ok = False
        sex_block = X[:, SEX_OFFSET:SITE_OFFSET]
        site_block = X[:, SITE_OFFSET:]
        if not (np.all(sex_block.sum(axis=1) == 1.0) and np.all(site_block.sum(axis=1) == 1.0)):
            failures.append(f"Vector structure: fold {fold_num} has a sex or site block that does not sum to 1")
            struct_ok = False
        flag = X[:, 1]
        recorded = df["age"].notna().to_numpy()
        if not np.array_equal(flag, recorded.astype(np.float32)):
            failures.append(f"Vector structure: fold {fold_num} age_recorded flag does not match age presence")
            struct_ok = False
        if not np.all(X[~recorded, 0] == 0.0):
            failures.append(f"Vector structure: fold {fold_num} has a missing age that did not encode as 0")
            struct_ok = False

        # 5. z-score sanity on the TRAINING rows of this fold
        train_rows = (df["cv_fold"].to_numpy() != fold_num) & recorded
        z = X[train_rows, 0].astype(np.float64)
        z_mean, z_std = z.mean(), z.std(ddof=AGE_STD_DDOF)
        if not (abs(z_mean) < 1e-4 and abs(z_std - 1.0) < 1e-3):
            failures.append(f"Z-score sanity: fold {fold_num} training z has mean {z_mean:.6f}, std {z_std:.6f}")
            struct_ok = False
        print(f"  {fold_num:>4} {stats['mean']:>9.3f} {stats['std']:>8.3f} {stats['n_recorded']:>11} "
              f"{z_mean:>13.6f} {z_std:>12.6f}")

        # 6. round trip: decode the one-hot blocks back to labels
        decoded_sex = [SEX_CATEGORIES[i] for i in sex_block.argmax(axis=1)]
        decoded_site = [SITE_CATEGORIES[i] for i in site_block.argmax(axis=1)]
        if decoded_sex != df["sex"].tolist() or decoded_site != df["localization"].tolist():
            failures.append(f"Round trip: fold {fold_num} decoded sex/site do not match the original rows")
            struct_ok = False
    if struct_ok:
        print(f"  OK -- {METADATA_DIM}-value float32 vectors, one-hots valid, missing ages encode as 0, "
              f"z-scores centred, labels decode back exactly")
    else:
        print("  FAILED -- see summary")

    # --- 7. JSON round trip ---
    print("\n[7/7] JSON round trip...")
    json_ok = True
    record = build_encoder_stats_record(df)
    if save_stats:
        save_encoder_stats(record)
        reloaded = load_encoder_stats()
    else:
        import tempfile
        tmp_path = os.path.join(tempfile.mkdtemp(), "metadata_encoder_stats_test.json")
        save_encoder_stats(record, tmp_path)
        reloaded = load_encoder_stats(tmp_path)
    if reloaded != record:
        failures.append("JSON round trip: reloaded stats differ from the in-memory stats")
        json_ok = False
    if json_ok:
        print(f"  OK -- reloaded stats identical ({len(record['folds'])} folds, "
              f"{record['metadata_dim']}-value layout)")
    else:
        print("  FAILED -- see summary")

    print("\n" + "=" * 60)
    if failures:
        print(f"VERIFICATION FAILED -- {len(failures)} issue(s):")
        for msg in failures:
            print(f"  - {msg}")
        print("=" * 60)
        raise RuntimeError(f"Metadata encoder verification failed with {len(failures)} issue(s).")
    print("VERIFICATION PASSED -- all checks OK. Module 01 is safe to build on.")
    print("=" * 60)
    return record


# --- Usage ---
# encoder_record = run_metadata_verification()

In [4]:
encoder_record = run_metadata_verification()

METADATA ENCODER VERIFICATION -- Phase 3, Module 01

[1/7] Join integrity...
  OK -- 8583 rows, unique image_ids, folds 0-4, none of the 1432 holdout images present

[2/7] Missing values (informational)...
  Fold table: age null = 48, sex unknown = 49, both = 39
  Full metadata file: age null = 57, sex unknown = 57, both = 47
  Are the age-null and sex-unknown records the SAME records? False

[3/7] Leakage test...
  OK -- all 5 folds match the independent recomputation and ignore their own validation ages

[4-6/7] Vector structure, z-score sanity, round trip...
  fold  age mean  age std  n recorded  train z mean  train z std
     0    51.894   17.147        6825     -0.000000     1.000000
     1    51.946   16.905        6832      0.000000     1.000000
     2    51.688   16.999        6827     -0.000000     1.000000
     3    51.763   16.982        6829      0.000000     1.000000
     4    51.728   16.948        6827      0.000000     1.000000
  OK -- 20-value float32 vectors, one-hots

# Module 2

In [5]:
"""
02_00_image_helpers.txt
Phase 3 -- Module 02: Dataset -- Image helpers and constants

Requires 01_00_metadata_config.txt already run (uses P1_ARTIFACTS_DIR).

WHAT THIS HOLDS
  - FINAL_IMAGES_DIR, CLASSES, CLASS_TO_IDX
  - normalize_image() and augment_image() with their configs

The block marked VERBATIM below is copied character-for-character from Phase 2
(Phase_2_P3.ipynb, cell 01_00), itself reused verbatim from Phase 1 (08_06 and
08_07). Do not edit it: Phase 3 must feed the model exactly the same pixels
Phase 2 did. Notably, augmentation fills rotated/zoomed-out corners by
reflection (BORDER_REFLECT_101), not black.

CLASS_TO_IDX is alphabetical and fixed -- it is the same mapping that the
loss-weight tensor (Mechanism A check) and the model's output order use.
"""

import cv2
import numpy as np

FINAL_IMAGES_DIR = P1_ARTIFACTS_DIR / "Final Output Phase 1" / "images"

CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]  # alphabetical, fixed
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# ===================== VERBATIM from Phase 2 (do not edit) =====================
# --- Reused verbatim from Phase 1 (08_06_normalization.txt) ---
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def normalize_image(image_bgr):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_float = image_rgb.astype(np.float32) / 255.0
    normalized = (image_float - IMAGENET_MEAN) / IMAGENET_STD
    return normalized


# --- Reused verbatim from Phase 1 (08_07_augmentation.txt) ---
AUGMENTATION_CONFIG = {
    "hflip_prob": 0.5,
    "vflip_prob": 0.5,
    "rotation_max_degrees": 180,
    "zoom_range": (0.9, 1.1),
    "brightness_range": (0.8, 1.2),
}


def augment_image(image, config=AUGMENTATION_CONFIG):
    h, w = image.shape[:2]
    out = image.copy()

    if np.random.random() < config["hflip_prob"]:
        out = cv2.flip(out, 1)
    if np.random.random() < config["vflip_prob"]:
        out = cv2.flip(out, 0)

    angle = np.random.uniform(-config["rotation_max_degrees"], config["rotation_max_degrees"])
    center = (w / 2, h / 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    out = cv2.warpAffine(out, rotation_matrix, (w, h), borderMode=cv2.BORDER_REFLECT_101)

    zoom_factor = np.random.uniform(*config["zoom_range"])
    new_h, new_w = int(round(h * zoom_factor)), int(round(w * zoom_factor))
    scaled = cv2.resize(out, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    if zoom_factor >= 1.0:
        # Zoomed in — centre-crop back down to original size
        row_start = (new_h - h) // 2
        col_start = (new_w - w) // 2
        out = scaled[row_start:row_start + h, col_start:col_start + w]
    else:
        # Zoomed out — pad back up to original size
        pad_h = h - new_h
        pad_w = w - new_w
        top, bottom = pad_h // 2, pad_h - pad_h // 2
        left, right = pad_w // 2, pad_w - pad_w // 2
        out = cv2.copyMakeBorder(scaled, top, bottom, left, right, borderType=cv2.BORDER_REFLECT_101)

    # Brightness
    brightness_factor = np.random.uniform(*config["brightness_range"])
    out = np.clip(out.astype(np.float32) * brightness_factor, 0, 255).astype(np.uint8)

    return out

# ================================ END VERBATIM ================================

print("Image helpers loaded.")
print(f"  Images folder : {FINAL_IMAGES_DIR}")
print(f"  Classes       : {CLASSES}")

Image helpers loaded.
  Images folder : T:\Thesis - Development\Resources\Artifacts\Phase 1 - P2\Final Output Phase 1\images
  Classes       : ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [6]:
"""
02_01_multimodal_dataset.txt
Phase 3 -- Module 02: Dataset -- Fold-aware multimodal dataset (reusable)

Requires 01_00, 01_01 (Module 01) and 02_00 already run in the same notebook.

WHAT THIS DOES
  One example = (image tensor, metadata tensor, label).
    image     float32, shape (3, 300, 300)  -- BGR->RGB, /255, ImageNet mean/std
    metadata  float32, shape (20,)          -- Module 01's vector
    label     int, 0-6 (CLASS_TO_IDX order)

ONE DATASET FOR BOTH MODELS (locked, Option A): the multimodal model uses all
three items; the matched image-only ablation simply ignores the metadata.
Both then see identical images, in identical order, with identical
augmentation draws -- so any difference comes from the metadata alone.

WHAT IS DELIBERATELY IDENTICAL TO PHASE 2's HAM10000FoldDataset
  - Rows: the fold table filtered on cv_fold, then index reset (same order, so
    the balanced sampler's seeded draws land on the same images)
  - Image loading: cv2.imread, BGR, from the Phase 1 final images folder
  - Training images get augment_image() then normalize_image(); validation
    images get normalize_image() only
  - Augmentation uses NumPy's global RNG, so keep num_workers=0 as in Phase 2
    (so the resume checkpoint can restore the augmentation sequence exactly)

WHAT IS NEW
  - metadata comes from Module 01, encoded with the TRAINING folds' age
    mean/std for both splits. It is never augmented.
  - no resize option: the multimodal model and its ablation are B3 at 300x300.
"""

import torch
from torch.utils.data import Dataset


class HAM10000MultimodalFoldDataset(Dataset):
    def __init__(self, fold_num: int, split: str, encoder_record: dict, fold_table=None):
        """
        fold_num: 0-4, which fold is held out as validation.
        split: "train" (folds != fold_num) or "val" (fold == fold_num).
        encoder_record: the dict from build_encoder_stats_record() or
                        load_encoder_stats() (Module 01).
        fold_table: optional pre-loaded load_fold_table_with_metadata() result,
                    so building 10 datasets doesn't re-read the CSVs 10 times.
        """
        assert split in ("train", "val")
        self.fold_num = fold_num
        self.split = split

        if fold_table is None:
            fold_table = load_fold_table_with_metadata()

        self.df, metadata = get_fold_split_with_metadata(fold_table, fold_num, split, encoder_record)
        self.metadata = torch.from_numpy(metadata)          # (n, 20) float32, row-aligned with self.df
        self.image_ids = self.df["image_id"].tolist()
        self.labels = self.df["dx"].map(CLASS_TO_IDX).to_numpy()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        label = int(self.labels[idx])

        img_path = FINAL_IMAGES_DIR / f"{image_id}.jpg"
        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f"Failed to load image: {img_path}")

        if self.split == "train":
            image = augment_image(image)  # training-only

        normalized = normalize_image(image)  # both splits
        tensor = torch.from_numpy(normalized.transpose(2, 0, 1)).float()  # HWC -> CHW

        return tensor, self.metadata[idx], label

    def get_labels(self):
        """Fast label access without loading images -- used by the balanced sampler."""
        return self.labels

In [7]:
"""
02_02_dataset_verification.txt
Phase 3 -- Module 02: Dataset -- Verification (run once)

Requires 01_00, 01_01, 01_02 (Module 01, with `encoder_record` produced by
run_metadata_verification()) and 02_00, 02_01 already run.

All seven checks run as a full set (every failure collected, then one
RuntimeError at the end), same convention as Module 01.

Checks:
  1. Fold sizes       -- train + val = whole pool for every fold, no image in
                         both, val sizes match the raw fold CSV
  2. Row order        -- image order equals an independent filter of the raw
                         fold CSV (i.e. the same order Phase 2's Dataset used)
  3. Metadata match   -- 25 random rows per fold/split: the dataset's vector
                         equals one built BY HAND from the raw metadata file and
                         the saved age statistics (image i really gets vector i)
  4. Item structure   -- image / metadata / label shapes, dtypes, finite values,
                         label agrees with the row's dx, get_labels() agrees
  5. Augmentation     -- validation is deterministic and un-augmented; training
                         is augmented, reproducible under a fixed NumPy seed,
                         and never changes the metadata
  6. Batching         -- a DataLoader batch has the expected three tensors
  7. Fold isolation   -- validation rows are encoded with TRAINING-fold age
                         stats (checked on the age column directly)

Uses local seeds for the augmentation checks (np.random.seed) so it does not
depend on set_all_seeds() being defined. Loads only a few dozen images.
"""

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader


def _hand_encode(image_id, raw_meta, site, age_mean, age_std):
    """Builds one metadata vector explicitly, without using encode_metadata()."""
    age, sex = raw_meta[image_id]
    vec = [0.0] * 20
    if age == age:               # not NaN
        vec[0] = (age - age_mean) / age_std
        vec[1] = 1.0
    sex_pos = {"male": 2, "female": 3, "unknown": 4}[sex]
    vec[sex_pos] = 1.0
    site_names = ["abdomen", "acral", "back", "chest", "ear", "face", "foot", "genital",
                  "hand", "lower extremity", "neck", "scalp", "trunk", "unknown",
                  "upper extremity"]
    vec[5 + site_names.index(site)] = 1.0
    return np.array(vec, dtype=np.float64)


def run_dataset_verification(encoder_record: dict, n_sample_rows: int = 25):
    failures = []
    print("=" * 60)
    print("DATASET VERIFICATION -- Phase 3, Module 02")
    print("=" * 60)

    fold_table = load_fold_table_with_metadata()
    raw_fold = pd.read_csv(FOLD_CSV_PATH)
    raw_meta_df = pd.read_csv(METADATA_TAB_PATH, sep="\t")
    raw_meta = {r.image_id: (r.age, r.sex) for r in raw_meta_df.itertuples(index=False)}
    raw_site = dict(zip(raw_fold["image_id"], raw_fold["localization"]))

    datasets = {}
    for fold_num in range(N_FOLDS):
        for split in ("train", "val"):
            datasets[(fold_num, split)] = HAM10000MultimodalFoldDataset(
                fold_num, split, encoder_record, fold_table=fold_table)

    # --- 1. Fold sizes ---
    print("\n[1/7] Fold sizes...")
    ok = True
    val_counts = raw_fold["cv_fold"].value_counts().sort_index()
    print(f"  {'fold':>4} {'train':>7} {'val':>6}")
    for fold_num in range(N_FOLDS):
        tr, va = datasets[(fold_num, "train")], datasets[(fold_num, "val")]
        print(f"  {fold_num:>4} {len(tr):>7} {len(va):>6}")
        if len(tr) + len(va) != EXPECTED_POOL_ROWS:
            failures.append(f"Fold sizes: fold {fold_num} train+val = {len(tr) + len(va)}, expected {EXPECTED_POOL_ROWS}")
            ok = False
        if set(tr.image_ids) & set(va.image_ids):
            failures.append(f"Fold sizes: fold {fold_num} has images in both train and val")
            ok = False
        if len(va) != int(val_counts[fold_num]):
            failures.append(f"Fold sizes: fold {fold_num} val size {len(va)} != raw CSV count {int(val_counts[fold_num])}")
            ok = False
    print("  OK" if ok else "  FAILED -- see summary")

    # --- 2. Row order ---
    print("\n[2/7] Row order matches an independent filter of the raw fold CSV...")
    ok = True
    for fold_num in range(N_FOLDS):
        exp_train = raw_fold.loc[raw_fold["cv_fold"] != fold_num, "image_id"].tolist()
        exp_val = raw_fold.loc[raw_fold["cv_fold"] == fold_num, "image_id"].tolist()
        if datasets[(fold_num, "train")].image_ids != exp_train or datasets[(fold_num, "val")].image_ids != exp_val:
            failures.append(f"Row order: fold {fold_num} image order differs from the raw fold CSV order")
            ok = False
    print("  OK -- all folds, both splits" if ok else "  FAILED -- see summary")

    # --- 3. Metadata match ---
    print(f"\n[3/7] Metadata match ({n_sample_rows} random rows per fold/split, built by hand)...")
    ok = True
    rng = np.random.RandomState(0)
    for (fold_num, split), ds in datasets.items():
        mean, std = encoder_record["folds"][str(fold_num)]["mean"], encoder_record["folds"][str(fold_num)]["std"]
        for idx in rng.choice(len(ds), size=n_sample_rows, replace=False):
            image_id = ds.image_ids[idx]
            expected = _hand_encode(image_id, raw_meta, raw_site[image_id], mean, std)
            got = ds.metadata[idx].numpy().astype(np.float64)
            if not np.allclose(got, expected, atol=1e-5):
                failures.append(f"Metadata match: fold {fold_num} {split} row {idx} ({image_id}) differs from the hand-built vector")
                ok = False
                break
    print("  OK -- image i gets vector i in every fold/split sampled" if ok else "  FAILED -- see summary")

    # --- 4. Item structure ---
    print("\n[4/7] Item structure...")
    ok = True
    for split in ("train", "val"):
        ds = datasets[(0, split)]
        image, meta, label = ds[0]
        if not (isinstance(image, torch.Tensor) and tuple(image.shape) == (3, 300, 300) and image.dtype == torch.float32):
            failures.append(f"Item structure: {split} image is {getattr(image, 'shape', None)} / {getattr(image, 'dtype', None)}")
            ok = False
        elif not torch.isfinite(image).all():
            failures.append(f"Item structure: {split} image has NaN or inf")
            ok = False
        if not (tuple(meta.shape) == (METADATA_DIM,) and meta.dtype == torch.float32 and torch.isfinite(meta).all()):
            failures.append(f"Item structure: {split} metadata is {tuple(meta.shape)} / {meta.dtype}")
            ok = False
        if not (isinstance(label, int) and label == CLASS_TO_IDX[ds.df.loc[0, 'dx']]):
            failures.append(f"Item structure: {split} label {label} does not match the row's dx")
            ok = False
        expected_labels = np.array([CLASS_TO_IDX[d] for d in ds.df["dx"]])
        if not np.array_equal(ds.get_labels(), expected_labels):
            failures.append(f"Item structure: {split} get_labels() differs from the dx column")
            ok = False
    print("  OK -- image (3,300,300) float32, metadata (20,) float32, label int, get_labels() consistent"
          if ok else "  FAILED -- see summary")

    # --- 5. Augmentation behaviour ---
    print("\n[5/7] Augmentation behaviour...")
    ok = True
    val_ds, train_ds = datasets[(0, "val")], datasets[(0, "train")]

    v1, vm1, _ = val_ds[0]
    v2, vm2, _ = val_ds[0]
    if not (torch.equal(v1, v2) and torch.equal(vm1, vm2)):
        failures.append("Augmentation: validation item is not deterministic")
        ok = False
    manual = cv2.imread(str(FINAL_IMAGES_DIR / f"{val_ds.image_ids[0]}.jpg"))
    manual = torch.from_numpy(normalize_image(manual).transpose(2, 0, 1)).float()
    if not torch.equal(v1, manual):
        failures.append("Augmentation: validation image differs from load -> normalize (something augmented it)")
        ok = False

    np.random.seed(0); a, am, _ = train_ds[0]
    np.random.seed(0); b, bm, _ = train_ds[0]
    np.random.seed(1); c, cm, _ = train_ds[0]
    if not torch.equal(a, b):
        failures.append("Augmentation: training item is not reproducible under the same NumPy seed")
        ok = False
    if torch.equal(a, c):
        failures.append("Augmentation: training item did not change under a different seed (augmentation not applied?)")
        ok = False
    if not (torch.equal(am, bm) and torch.equal(am, cm)):
        failures.append("Augmentation: metadata changed between training fetches")
        ok = False
    print("  OK -- val deterministic and un-augmented; train augmented and seed-reproducible; metadata untouched"
          if ok else "  FAILED -- see summary")

    # --- 6. Batching ---
    print("\n[6/7] Batching...")
    ok = True
    loader = DataLoader(train_ds, batch_size=8, shuffle=False, num_workers=0)
    images, metas, labels = next(iter(loader))
    if tuple(images.shape) != (8, 3, 300, 300) or images.dtype != torch.float32:
        failures.append(f"Batching: images batch is {tuple(images.shape)} / {images.dtype}")
        ok = False
    if tuple(metas.shape) != (8, METADATA_DIM) or metas.dtype != torch.float32:
        failures.append(f"Batching: metadata batch is {tuple(metas.shape)} / {metas.dtype}")
        ok = False
    if tuple(labels.shape) != (8,) or labels.dtype != torch.int64:
        failures.append(f"Batching: labels batch is {tuple(labels.shape)} / {labels.dtype}")
        ok = False
    print("  OK -- (8,3,300,300) float32, (8,20) float32, (8,) int64" if ok else "  FAILED -- see summary")

    # --- 7. Fold isolation ---
    print("\n[7/7] Fold isolation (validation encoded with TRAINING-fold age stats)...")
    ok = True
    for fold_num in range(N_FOLDS):
        val_ds_k = datasets[(fold_num, "val")]
        mean, std = encoder_record["folds"][str(fold_num)]["mean"], encoder_record["folds"][str(fold_num)]["std"]
        ages = val_ds_k.df["age"].to_numpy(dtype=np.float64)
        rec = ~np.isnan(ages)
        expected_z = ((ages[rec] - mean) / std).astype(np.float32)
        got_z = val_ds_k.metadata[:, 0].numpy()[rec]
        if not np.allclose(got_z, expected_z, atol=1e-5):
            failures.append(f"Fold isolation: fold {fold_num} validation age_z is not (age - train mean) / train std")
            ok = False
    print("  OK -- all 5 folds" if ok else "  FAILED -- see summary")

    print("\n" + "=" * 60)
    if failures:
        print(f"VERIFICATION FAILED -- {len(failures)} issue(s):")
        for msg in failures:
            print(f"  - {msg}")
        print("=" * 60)
        raise RuntimeError(f"Dataset verification failed with {len(failures)} issue(s).")
    print("VERIFICATION PASSED -- all checks OK. Module 02 is safe to build on.")
    print("=" * 60)


# --- Usage (encoder_record comes from Module 01's run_metadata_verification()) ---
# run_dataset_verification(encoder_record)

In [8]:
run_dataset_verification(encoder_record)

DATASET VERIFICATION -- Phase 3, Module 02

[1/7] Fold sizes...
  fold   train    val
     0    6866   1717
     1    6867   1716
     2    6867   1716
     3    6865   1718
     4    6867   1716
  OK

[2/7] Row order matches an independent filter of the raw fold CSV...
  OK -- all folds, both splits

[3/7] Metadata match (25 random rows per fold/split, built by hand)...
  OK -- image i gets vector i in every fold/split sampled

[4/7] Item structure...
  OK -- image (3,300,300) float32, metadata (20,) float32, label int, get_labels() consistent

[5/7] Augmentation behaviour...
  OK -- val deterministic and un-augmented; train augmented and seed-reproducible; metadata untouched

[6/7] Batching...
  OK -- (8,3,300,300) float32, (8,20) float32, (8,) int64

[7/7] Fold isolation (validation encoded with TRAINING-fold age stats)...
  OK -- all 5 folds

VERIFICATION PASSED -- all checks OK. Module 02 is safe to build on.


# Module 3

In [9]:
"""
03_00_model_config.txt
Phase 3 -- Module 03: Model Builders -- Configuration

Requires 01_00 (METADATA_DIM) and 02_00 (CLASSES) already run in the same notebook.

Every architecture number lives here. Nothing else in Module 03 hardcodes them.

  FEATURE_CHANNELS  1,536 -- EfficientNet-B3's final convolutional layer width
  META_HIDDEN       32    -- metadata branch output (Methodology 4.3)
  META_DROPOUT      0.3   -- metadata branch dropout (Methodology 4.5)
  HEAD_DROPOUT      0.5   -- classification head dropout (Methodology 4.4)
  NUM_CLASSES       7     -- HAM10000 taxonomy, CLASSES order from 02_00

Initialisation (Phase-3_Planning_Updated.md, Sections 2-4):
  - metadata branch Dense(32): PyTorch default (Kaiming-uniform) -- left untouched
  - MetaBlock f_b and g_b linear layers: Xavier/Glorot (they feed tanh / sigmoid)
  - classification head linear layer: Xavier/Glorot
  Xavier here = xavier_uniform_ on the weights, zeros on the biases.
"""

import torch
import torch.nn as nn
import torchvision.models as tvm

FEATURE_CHANNELS = 1536
META_HIDDEN = 32
META_DROPOUT = 0.3
HEAD_DROPOUT = 0.5
NUM_CLASSES = 7

assert NUM_CLASSES == len(CLASSES), "NUM_CLASSES must equal len(CLASSES) from 02_00"

print("Model config loaded.")
print(f"  Backbone      : EfficientNet-B3 features, {FEATURE_CHANNELS} channels, ImageNet1K_V1 weights")
print(f"  Metadata      : {METADATA_DIM} -> Dense({META_HIDDEN}) -> ReLU -> Dropout({META_DROPOUT})")
print(f"  Head          : Dropout({HEAD_DROPOUT}) -> Linear({FEATURE_CHANNELS}, {NUM_CLASSES})")

Model config loaded.
  Backbone      : EfficientNet-B3 features, 1536 channels, ImageNet1K_V1 weights
  Metadata      : 20 -> Dense(32) -> ReLU -> Dropout(0.3)
  Head          : Dropout(0.5) -> Linear(1536, 7)


In [10]:
"""
03_01_model_components.txt
Phase 3 -- Module 03: Model Builders -- Shared components (reusable)

Requires 03_00 already run.

THREE PARTS, shared by both models so they cannot drift apart:

  build_classification_head()  Dropout(0.5) -> Linear(1536, 7), Xavier init.
                               Built by ONE function for the multimodal model
                               AND the matched ablation.
  MetadataBranch               Dense(32) -> ReLU -> Dropout(0.3).
  MetaBlock                    The fusion mechanism of Pacheco & Krohling (2021).

METABLOCK (as specified in Phase-3_Planning_Updated.md, Section 3, Equations 3-7
of the source paper). Compare this forward() line by line against the paper:

    f = BN( W_f . m + b_f )        one value per image channel   (f_b)
    g = BN( W_g . m + b_g )        one value per image channel   (g_b)
    out = sigmoid( tanh( f * F ) + g )

  F = the image feature map, shape (B, 1536, H, W), NOT yet pooled
  m = the 32-value metadata feature vector
  f and g are broadcast across the H x W grid (one value per channel).
  Output has exactly F's shape -- a reweighting, not a concatenation.
  Batch normalisation is applied to f_b and g_b themselves, not to F.
  Global average pooling happens AFTER this block (see 03_02).

BatchNorm1d note: in training mode it needs at least 2 samples per batch. The
last batch of each fold's training epoch is 17-19 images, so this is fine; keep
it in mind if batch size or drop_last ever changes.
"""


def init_xavier_linear(layer: nn.Linear) -> nn.Linear:
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)
    return layer


def build_classification_head(in_features: int = FEATURE_CHANNELS,
                              num_classes: int = NUM_CLASSES) -> nn.Sequential:
    """Dropout(0.5) -> Linear(in_features, num_classes). Returns logits;
    the softmax lives inside CrossEntropyLoss, as in Phase 2."""
    return nn.Sequential(
        nn.Dropout(HEAD_DROPOUT),
        init_xavier_linear(nn.Linear(in_features, num_classes)),
    )


class MetadataBranch(nn.Module):
    """Dense(32) -> ReLU -> Dropout(0.3). PyTorch-default (Kaiming-uniform) init."""

    def __init__(self, in_dim: int = METADATA_DIM, hidden: int = META_HIDDEN,
                 dropout: float = META_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, metadata: torch.Tensor) -> torch.Tensor:
        return self.net(metadata)


class MetaBlock(nn.Module):
    """Metadata-conditioned reweighting of an UNPOOLED image feature map."""

    def __init__(self, meta_dim: int = META_HIDDEN, n_channels: int = FEATURE_CHANNELS):
        super().__init__()
        self.f_b = nn.Sequential(
            init_xavier_linear(nn.Linear(meta_dim, n_channels)),
            nn.BatchNorm1d(n_channels),
        )
        self.g_b = nn.Sequential(
            init_xavier_linear(nn.Linear(meta_dim, n_channels)),
            nn.BatchNorm1d(n_channels),
        )

    def forward(self, feature_map: torch.Tensor, meta_features: torch.Tensor) -> torch.Tensor:
        if feature_map.dim() != 4:
            raise ValueError(f"MetaBlock needs an unpooled (B, C, H, W) feature map, got {tuple(feature_map.shape)}")
        f = self.f_b(meta_features).unsqueeze(-1).unsqueeze(-1)   # (B, C, 1, 1)
        g = self.g_b(meta_features).unsqueeze(-1).unsqueeze(-1)   # (B, C, 1, 1)
        return torch.sigmoid(torch.tanh(f * feature_map) + g)

In [11]:
"""
03_02_model_builders.txt
Phase 3 -- Module 03: Model Builders -- The two models (reusable)

Requires 03_00 and 03_01 already run.

  MultimodalNet   image + metadata:
        backbone -> feature map (B,1536,10,10)
        metadata -> MetadataBranch -> 32 values
        MetaBlock(feature map, 32 values) -> reweighted map (B,1536,10,10)
        global average pooling  (AFTER the fusion)
        shared head -> 7 logits

  ImageOnlyNet    the matched ablation:
        backbone -> feature map -> global average pooling -> shared head -> 7 logits

The two differ ONLY in the metadata branch and MetaBlock. Same backbone, same
pooling, same head function, same dropout, same initialisation.

CALLING CONVENTION (locked, Option A): both are called model(image, metadata).
The ablation ignores `metadata`, so one training loop serves both.

BACKBONE: torchvision EfficientNet-B3 `.features` only. The stock avgpool and
classifier are NOT used -- this removes the stock Dropout(0.3) that the Phase 2
ablation carried in front of its own Dropout(0.5). Here each model has exactly
the dropouts it is meant to have.

GRAD-CAM (for Phase 4): the primary layer is model.backbone[-1] for BOTH models
-- the backbone's final convolutional output, before any fusion or pooling.

PARAMETER GROUPS (for Module 05): backbone_parameters() and head_parameters()
together cover every parameter exactly once. "Head" here means everything that
is not the backbone: metadata branch, MetaBlock, classification head.

SEEDING: the builders do not seed. The training script must call
set_all_seeds() BEFORE building a model so the new layers' initial weights are
captured by the seed.
"""


def _build_backbone(pretrained: bool = True) -> nn.Module:
    weights = tvm.EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None
    return tvm.efficientnet_b3(weights=weights).features


class ImageOnlyNet(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.backbone = _build_backbone(pretrained)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = build_classification_head()

    def forward(self, image: torch.Tensor, metadata: torch.Tensor) -> torch.Tensor:
        feature_map = self.backbone(image)                 # (B, 1536, 10, 10)
        pooled = self.pool(feature_map).flatten(1)         # (B, 1536)
        return self.classifier(pooled)                     # (B, 7); metadata deliberately unused

    def backbone_parameters(self):
        return list(self.backbone.parameters())

    def head_parameters(self):
        return list(self.classifier.parameters())


class MultimodalNet(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.backbone = _build_backbone(pretrained)
        self.metadata_branch = MetadataBranch()
        self.metablock = MetaBlock()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = build_classification_head()

    def forward(self, image: torch.Tensor, metadata: torch.Tensor) -> torch.Tensor:
        feature_map = self.backbone(image)                       # (B, 1536, 10, 10)  not yet pooled
        meta_features = self.metadata_branch(metadata)           # (B, 32)
        fused = self.metablock(feature_map, meta_features)       # (B, 1536, 10, 10)
        pooled = self.pool(fused).flatten(1)                     # (B, 1536)  pooling AFTER fusion
        return self.classifier(pooled)                           # (B, 7)

    def backbone_parameters(self):
        return list(self.backbone.parameters())

    def head_parameters(self):
        return (list(self.metadata_branch.parameters())
                + list(self.metablock.parameters())
                + list(self.classifier.parameters()))


def build_multimodal_model(pretrained: bool = True) -> nn.Module:
    return MultimodalNet(pretrained=pretrained)


def build_image_only_model(pretrained: bool = True) -> nn.Module:
    return ImageOnlyNet(pretrained=pretrained)

In [12]:
"""
03_03_model_verification.txt
Phase 3 -- Module 03: Model Builders -- Verification (run once)

Requires 03_00, 03_01, 03_02 already run. Runs on CPU by default (small dummy
batches), so it does not touch the GPU or interfere with anything else.

All nine checks run as a full set (failures collected, one RuntimeError at the
end), same convention as Modules 01 and 02.

Checks:
  1. Shapes            -- both models: (4,3,300,300) + (4,20) -> (4,7), finite
  2. Pooling order     -- hooks prove the feature map is still an unpooled
                          (4,1536,10,10) grid when it ENTERS MetaBlock, and that
                          pooling consumes MetaBlock's OUTPUT (not the backbone's)
  3. MetaBlock maths   -- output equals sigmoid(tanh(f*F)+g) recomputed by hand
                          from the layers' raw weights and BatchNorm statistics;
                          output stays in (0,1)
  4. Metadata effect   -- changing one sample's metadata changes only that sample
                          in the multimodal model; the ablation ignores metadata
                          exactly; gradients reach the metadata branch and MetaBlock
  5. Shared head       -- both heads are Dropout(0.5) -> Linear(1536,7); the
                          ablation has exactly ONE dropout in the whole network
                          (no hidden stock Dropout(0.3)), the multimodal exactly two
  6. Initialisation    -- head and MetaBlock linears are Xavier (bounds, spread,
                          zero bias); the metadata Dense keeps PyTorch's default
  7. Seeding           -- same seed -> identical new-layer weights; a different
                          seed -> different weights
  8. Parameter groups  -- backbone + head groups cover every parameter exactly
                          once; both models share an identical backbone
  9. Pretrained weights -- both backbones equal torchvision's ImageNet1K_V1 features,
                          compared right after construction, before any training-mode
                          pass changes BatchNorm statistics (skipped when pretrained=False)
"""

import math

import torch
import torch.nn as nn
import torchvision.models as tvm


def _cpu_batch(seed=0, n=4):
    g = torch.Generator().manual_seed(seed)
    image = torch.randn(n, 3, 300, 300, generator=g)
    meta = torch.randn(n, METADATA_DIM, generator=g)
    return image, meta


def _build_seeded(builder, seed, pretrained):
    torch.manual_seed(seed)
    return builder(pretrained=pretrained)


def _xavier_ok(layer: nn.Linear, tag: str, failures: list):
    fan_out, fan_in = layer.weight.shape
    bound = math.sqrt(6.0 / (fan_in + fan_out))
    w = layer.weight.detach()
    ok = True
    if w.abs().max().item() > bound * 1.0001:
        failures.append(f"Initialisation: {tag} has a weight beyond the Xavier bound {bound:.4f}")
        ok = False
    if abs(w.std().item() - bound / math.sqrt(3.0)) > 0.05 * (bound / math.sqrt(3.0)):
        failures.append(f"Initialisation: {tag} weight spread is not Xavier-like")
        ok = False
    if layer.bias.detach().abs().max().item() != 0.0:
        failures.append(f"Initialisation: {tag} bias is not zero")
        ok = False
    return ok


def run_model_verification(pretrained: bool = True, device: str = "cpu"):
    failures = []
    print("=" * 60)
    print("MODEL VERIFICATION -- Phase 3, Module 03")
    print("=" * 60)
    print(f"  pretrained={pretrained}, device={device}")

    image, meta = _cpu_batch()
    image, meta = image.to(device), meta.to(device)

    torch.manual_seed(42)
    multimodal = build_multimodal_model(pretrained=pretrained).to(device)
    torch.manual_seed(42)
    ablation = build_image_only_model(pretrained=pretrained).to(device)
    multimodal.eval(); ablation.eval()
    # Snapshot taken right after construction (later checks run training-mode passes that
    # legitimately update BatchNorm statistics, so this comparison must happen first).
    backbones_match_at_construction = all(
        torch.equal(p, q) for p, q in zip(multimodal.backbone.state_dict().values(),
                                          ablation.backbone.state_dict().values()))
    if pretrained:
        _reference = tvm.efficientnet_b3(weights=tvm.EfficientNet_B3_Weights.IMAGENET1K_V1).features
        _ref_sd = _reference.state_dict()
        pretrained_match_at_construction = True
        for _model in (multimodal, ablation):
            _sd = _model.backbone.state_dict()
            if _sd.keys() != _ref_sd.keys() or not all(torch.equal(_ref_sd[k].to(device), _sd[k]) for k in _ref_sd):
                pretrained_match_at_construction = False
        del _reference, _ref_sd

    # --- 1. Shapes ---
    print("\n[1/9] Shapes...")
    ok = True
    with torch.no_grad():
        for name, model in (("multimodal", multimodal), ("ablation", ablation)):
            out = model(image, meta)
            if tuple(out.shape) != (4, NUM_CLASSES) or not torch.isfinite(out).all():
                failures.append(f"Shapes: {name} output is {tuple(out.shape)} or non-finite")
                ok = False
    print(f"  OK -- both models: image (4,3,300,300) + metadata (4,{METADATA_DIM}) -> logits (4,{NUM_CLASSES})"
          if ok else "  FAILED -- see summary")

    # --- 2. Pooling order ---
    print("\n[2/9] Pooling order (hooks)...")
    ok = True
    seen = {}
    h1 = multimodal.metablock.register_forward_hook(
        lambda m, inp, out: seen.update(mb_in=inp[0].detach().clone(), mb_meta=inp[1].detach().clone(), mb_out=out.detach().clone()))
    h2 = multimodal.pool.register_forward_hook(lambda m, inp, out: seen.update(pool_in=inp[0].detach().clone()))
    h3 = ablation.pool.register_forward_hook(lambda m, inp, out: seen.update(abl_pool_in=inp[0].detach().clone()))
    with torch.no_grad():
        multimodal(image, meta)
        ablation(image, meta)
    for h in (h1, h2, h3):
        h.remove()
    grid = tuple(seen["mb_in"].shape)
    print(f"  Feature map entering MetaBlock: {grid}")
    if grid[1] != FEATURE_CHANNELS or grid[2] * grid[3] <= 1:
        failures.append(f"Pooling order: the map entering MetaBlock is {grid}, not an unpooled ({FEATURE_CHANNELS}, H, W) grid")
        ok = False
    if not torch.equal(seen["pool_in"], seen["mb_out"]):
        failures.append("Pooling order: pooling does not consume MetaBlock's output")
        ok = False
    if tuple(seen["mb_out"].shape) != grid:
        failures.append("Pooling order: MetaBlock changed the feature map's shape")
        ok = False
    if tuple(seen["abl_pool_in"].shape) != grid:
        failures.append("Pooling order: the ablation pools a different-shaped map than the multimodal backbone produces")
        ok = False
    print("  OK -- MetaBlock sees the unpooled grid, pooling runs on MetaBlock's output" if ok else "  FAILED -- see summary")

    # --- 3. MetaBlock maths ---
    print("\n[3/9] MetaBlock maths (recomputed by hand)...")
    ok = True
    block = multimodal.metablock
    block.train()
    with torch.no_grad():
        block(seen["mb_in"], torch.randn(4, META_HIDDEN) * 3.0 + 1.0)   # move BatchNorm statistics off their defaults
    block.eval()
    with torch.no_grad():
        got = block(seen["mb_in"], seen["mb_meta"])

        def _hand_bn_linear(seq, m):
            lin, bn = seq[0], seq[1]
            z = m @ lin.weight.T + lin.bias
            return (z - bn.running_mean) / torch.sqrt(bn.running_var + bn.eps) * bn.weight + bn.bias

        f = _hand_bn_linear(block.f_b, seen["mb_meta"])[:, :, None, None]
        g = _hand_bn_linear(block.g_b, seen["mb_meta"])[:, :, None, None]
        expected = torch.sigmoid(torch.tanh(f * seen["mb_in"]) + g)
    if not torch.allclose(got, expected, atol=1e-5):
        failures.append("MetaBlock maths: output differs from sigmoid(tanh(f*F)+g) computed by hand")
        ok = False
    if not (got.min().item() >= 0.0 and got.max().item() <= 1.0):
        failures.append("MetaBlock maths: output left the (0,1) range")
        ok = False
    print("  OK -- matches sigmoid(tanh(f*F)+g), output within (0,1)" if ok else "  FAILED -- see summary")

    # --- 4. Metadata effect ---
    print("\n[4/9] Metadata effect...")
    ok = True
    meta2 = meta.clone()
    meta2[0] = torch.randn(METADATA_DIM, generator=torch.Generator().manual_seed(99)).to(device)
    with torch.no_grad():
        a1, a2 = multimodal(image, meta), multimodal(image, meta2)
        b1, b2 = ablation(image, meta), ablation(image, meta2)
    if (a1[0] - a2[0]).abs().max().item() < 1e-6:
        failures.append("Metadata effect: changing a sample's metadata did not change the multimodal output")
        ok = False
    if not torch.allclose(a1[1:], a2[1:], atol=1e-5):
        failures.append("Metadata effect: changing sample 0's metadata altered OTHER samples' outputs")
        ok = False
    if not torch.equal(b1, b2):
        failures.append("Metadata effect: the ablation output depends on metadata")
        ok = False
    multimodal.train()
    multimodal.zero_grad()
    labels = torch.tensor([0, 1, 2, 3]).to(device)
    loss = nn.CrossEntropyLoss()(multimodal(image, meta), labels)
    loss.backward()
    multimodal.eval()
    for tag, p in (("metadata branch", multimodal.metadata_branch.net[0].weight),
                   ("MetaBlock f_b", multimodal.metablock.f_b[0].weight),
                   ("MetaBlock g_b", multimodal.metablock.g_b[0].weight),
                   ("backbone", next(iter(multimodal.backbone.parameters())))):
        if p.grad is None or not torch.isfinite(p.grad).all() or p.grad.abs().sum().item() == 0.0:
            failures.append(f"Metadata effect: no usable gradient reaches the {tag}")
            ok = False
    print("  OK -- metadata changes only its own sample; ablation ignores it exactly; gradients reach every part"
          if ok else "  FAILED -- see summary")

    # --- 5. Shared head ---
    print("\n[5/9] Shared head and dropout count...")
    ok = True
    for name, model in (("multimodal", multimodal), ("ablation", ablation)):
        layers = list(model.classifier)
        if [type(m).__name__ for m in layers] != ["Dropout", "Linear"]:
            failures.append(f"Shared head: {name} classifier is {[type(m).__name__ for m in layers]}")
            ok = False
        else:
            if layers[0].p != HEAD_DROPOUT:
                failures.append(f"Shared head: {name} dropout is {layers[0].p}, expected {HEAD_DROPOUT}")
                ok = False
            if (layers[1].in_features, layers[1].out_features) != (FEATURE_CHANNELS, NUM_CLASSES):
                failures.append(f"Shared head: {name} Linear is {layers[1].in_features}->{layers[1].out_features}")
                ok = False
    abl_drops = sorted(m.p for m in ablation.modules() if isinstance(m, nn.Dropout))
    mm_drops = sorted(m.p for m in multimodal.modules() if isinstance(m, nn.Dropout))
    print(f"  Dropout layers in ablation   : {abl_drops}")
    print(f"  Dropout layers in multimodal : {mm_drops}")
    if abl_drops != [HEAD_DROPOUT]:
        failures.append(f"Shared head: ablation dropouts are {abl_drops}, expected exactly [{HEAD_DROPOUT}]")
        ok = False
    if mm_drops != sorted([META_DROPOUT, HEAD_DROPOUT]):
        failures.append(f"Shared head: multimodal dropouts are {mm_drops}, expected [{META_DROPOUT}, {HEAD_DROPOUT}]")
        ok = False
    print("  OK -- identical Dropout(0.5) -> Linear(1536,7) head, no hidden extra dropout" if ok else "  FAILED -- see summary")

    # --- 6. Initialisation ---
    print("\n[6/9] Initialisation...")
    ok = True
    ok &= _xavier_ok(multimodal.classifier[1], "multimodal head Linear", failures)
    ok &= _xavier_ok(ablation.classifier[1], "ablation head Linear", failures)
    ok &= _xavier_ok(multimodal.metablock.f_b[0], "MetaBlock f_b Linear", failures)
    ok &= _xavier_ok(multimodal.metablock.g_b[0], "MetaBlock g_b Linear", failures)
    dense = multimodal.metadata_branch.net[0]
    default_bound = 1.0 / math.sqrt(dense.in_features)     # PyTorch's default Linear init bound
    dw = dense.weight.detach()
    if dw.abs().max().item() > default_bound * 1.0001 or abs(dw.std().item() - default_bound / math.sqrt(3.0)) > 0.15 * (default_bound / math.sqrt(3.0)):
        failures.append("Initialisation: metadata Dense does not look like PyTorch's default init")
        ok = False
    print("  OK -- Xavier on head, f_b and g_b (zero bias); PyTorch default on the metadata Dense"
          if ok else "  FAILED -- see summary")

    # --- 7. Seeding ---
    print("\n[7/9] Seeding...")
    ok = True
    m_a = _build_seeded(build_multimodal_model, 42, pretrained)
    m_b = _build_seeded(build_multimodal_model, 42, pretrained)
    m_c = _build_seeded(build_multimodal_model, 43, pretrained)
    same = all(torch.equal(p, q) for p, q in zip(m_a.head_parameters(), m_b.head_parameters()))
    diff = any(not torch.equal(p, q) for p, q in zip(m_a.head_parameters(), m_c.head_parameters()))
    if not same:
        failures.append("Seeding: the same seed produced different new-layer weights")
        ok = False
    if not diff:
        failures.append("Seeding: a different seed produced identical new-layer weights")
        ok = False
    a_a = _build_seeded(build_image_only_model, 42, pretrained)
    a_b = _build_seeded(build_image_only_model, 42, pretrained)
    if not all(torch.equal(p, q) for p, q in zip(a_a.head_parameters(), a_b.head_parameters())):
        failures.append("Seeding: the ablation's head differs between two builds with the same seed")
        ok = False
    print("  OK -- same seed gives identical new layers, a different seed does not" if ok else "  FAILED -- see summary")

    # --- 8. Parameter groups ---
    print("\n[8/9] Parameter groups and shared backbone...")
    ok = True
    for name, model in (("multimodal", multimodal), ("ablation", ablation)):
        bb, hd = model.backbone_parameters(), model.head_parameters()
        bb_ids, hd_ids = [id(p) for p in bb], [id(p) for p in hd]
        all_ids = {id(p) for p in model.parameters()}
        if len(set(bb_ids)) != len(bb_ids) or len(set(hd_ids)) != len(hd_ids) or set(bb_ids) & set(hd_ids):
            failures.append(f"Parameter groups: {name} groups overlap or repeat a parameter")
            ok = False
        if set(bb_ids) | set(hd_ids) != all_ids:
            failures.append(f"Parameter groups: {name} groups do not cover every parameter")
            ok = False
        n_bb = sum(p.numel() for p in bb)
        n_hd = sum(p.numel() for p in hd)
        n_bn2d = sum(isinstance(m, nn.BatchNorm2d) for m in model.backbone.modules())
        print(f"  {name:>10}: backbone {n_bb:,} params ({n_bn2d} BatchNorm2d layers), head {n_hd:,} params")
    n_bn1d = sum(isinstance(m, nn.BatchNorm1d) for m in multimodal.modules())
    if n_bn1d != 2:
        failures.append(f"Parameter groups: multimodal has {n_bn1d} BatchNorm1d layers, expected 2 (f_b and g_b)")
        ok = False
    if not backbones_match_at_construction:
        failures.append("Parameter groups: the two models' backbones differ at construction")
        ok = False
    print("  OK -- groups are disjoint and complete; both models start from an identical backbone" if ok else "  FAILED -- see summary")

    # --- 9. Pretrained weights ---
    print("\n[9/9] Pretrained weights...")
    if pretrained:
        ok = True
        if not pretrained_match_at_construction:
            failures.append("Pretrained weights: backbone differs from torchvision's ImageNet1K_V1 features")
            ok = False
        print("  OK -- both models' backbones equal torchvision's ImageNet1K_V1 features (parameters and BatchNorm statistics)"
              if ok else "  FAILED -- see summary")
    else:
        print("  SKIPPED -- run with pretrained=True to check this")

    print("\n" + "=" * 60)
    if failures:
        print(f"VERIFICATION FAILED -- {len(failures)} issue(s):")
        for msg in failures:
            print(f"  - {msg}")
        print("=" * 60)
        raise RuntimeError(f"Model verification failed with {len(failures)} issue(s).")
    print("VERIFICATION PASSED -- all checks OK. Module 03 is safe to build on.")
    print("=" * 60)


# --- Usage ---
# run_model_verification()                    # on your machine (pretrained weights)

In [13]:
run_model_verification()

MODEL VERIFICATION -- Phase 3, Module 03
  pretrained=True, device=cpu

[1/9] Shapes...
  OK -- both models: image (4,3,300,300) + metadata (4,20) -> logits (4,7)

[2/9] Pooling order (hooks)...
  Feature map entering MetaBlock: (4, 1536, 10, 10)
  OK -- MetaBlock sees the unpooled grid, pooling runs on MetaBlock's output

[3/9] MetaBlock maths (recomputed by hand)...
  OK -- matches sigmoid(tanh(f*F)+g), output within (0,1)

[4/9] Metadata effect...
  OK -- metadata changes only its own sample; ablation ignores it exactly; gradients reach every part

[5/9] Shared head and dropout count...
  Dropout layers in ablation   : [0.5]
  Dropout layers in multimodal : [0.3, 0.5]
  OK -- identical Dropout(0.5) -> Linear(1536,7) head, no hidden extra dropout

[6/9] Initialisation...
  OK -- Xavier on head, f_b and g_b (zero bias); PyTorch default on the metadata Dense

[7/9] Seeding...
  OK -- same seed gives identical new layers, a different seed does not

[8/9] Parameter groups and shared back

# Module 4

In [14]:
"""
04_00_phase_control.txt
Phase 3 -- Module 04: Phase Control -- freeze / unfreeze the backbone (reusable)

Requires 03_02 (MultimodalNet / ImageOnlyNet) already run. Works on both models,
because both expose .backbone, .backbone_parameters() and .head_parameters().

TWO PHASES (Phase-3 plan)
  Phase A  epochs 1-5   backbone FROZEN, only the "head" group learns
  Phase B  epochs 6+    everything trains (backbone at its lower learning rate,
                        set in Module 05)

"Head" means everything that is not the backbone: metadata branch, MetaBlock,
classification head (the same definition as head_parameters()).

WHAT "FROZEN" HAS TO MEAN -- two things, not one
  1. Weights: requires_grad = False on every backbone parameter.
  2. BatchNorm statistics: the backbone's 78 BatchNorm2d layers keep running
     averages that update on EVERY forward pass while the layer is in train
     mode, even when its weights are frozen. So the backbone must also be in
     eval mode.

THE TRAP THIS MODULE REMOVES
  model.train() switches EVERY layer back to train mode, including the frozen
  backbone's BatchNorm. If the training loop calls model.train() at the top of
  each epoch (it must, because validation calls model.eval()), the backbone
  would silently stop being frozen after epoch 1. So the training loop must
  call set_phase(model, phase) INSTEAD of model.train(), every epoch.

SIDE EFFECT (intended): backbone.eval() also switches off EfficientNet's
stochastic depth (random block dropping) during Phase A. It switches on again in
Phase B -- normal fine-tuning behaviour.

Phase A leaves the head group in train mode: Dropout active, and the MetaBlock's
own BatchNorm1d layers keep learning their statistics (they are new layers).

set_phase does not build the optimizer (Module 05) and does not run the loop
(Module 07). It is safe to call every epoch; calling it twice gives the same state.
"""

import torch.nn as nn

PHASE_A_EPOCHS = 5          # epochs 1..5 are Phase A; epoch 6 is the first Phase B epoch
PHASES = ("A", "B")


def set_phase(model: nn.Module, phase: str) -> None:
    """Put `model` into Phase A (backbone frozen) or Phase B (all trainable).

    Sets requires_grad AND train/eval modes together, so the loop cannot forget
    one of them. Call it at the start of every epoch in place of model.train().
    """
    if phase not in PHASES:
        raise ValueError(f"phase must be one of {PHASES}, got {phase!r}")
    for needed in ("backbone", "backbone_parameters", "head_parameters"):
        if not hasattr(model, needed):
            raise TypeError(f"model has no '{needed}' -- set_phase expects MultimodalNet or ImageOnlyNet")

    backbone = getattr(model, "backbone")
    head_params = getattr(model, "head_parameters")()

    # Head group: always trainable, in both phases.
    for p in head_params:
        p.requires_grad = True

    if phase == "A":
        for p in backbone.parameters():
            p.requires_grad = False        # weights frozen
        model.train()                      # everything to train mode first ...
        backbone.eval()                    # ... then the backbone back to eval:
                                           #     BatchNorm statistics locked, stochastic depth off
    else:
        for p in backbone.parameters():
            p.requires_grad = True
        model.train()                      # everything in train mode, backbone included

print("Phase control loaded.")
print(f"  Phase A : epochs 1-{PHASE_A_EPOCHS}  -- backbone frozen (weights + BatchNorm statistics), head group trains")
print(f"  Phase B : epoch {PHASE_A_EPOCHS + 1}+  -- everything trains")
print("  The training loop must call set_phase(model, phase) INSTEAD of model.train(), every epoch.")

Phase control loaded.
  Phase A : epochs 1-5  -- backbone frozen (weights + BatchNorm statistics), head group trains
  Phase B : epoch 6+  -- everything trains
  The training loop must call set_phase(model, phase) INSTEAD of model.train(), every epoch.


In [15]:
"""
04_01_phase_helpers.txt
Phase 3 -- Module 04: Phase Helpers (reusable)

Requires 04_00 already run.

  phase_for_epoch(epoch)   1..PHASE_A_EPOCHS -> "A", later epochs -> "B".
                           Epochs are numbered from 1 (as in the Phase 2 logs).
                           The training loop AND the resume logic both use this
                           one function, so the phase can never disagree between them.

  describe_phase(model)    prints a short state report (trainable parameter
                           counts, and how many BatchNorm layers are in train
                           vs eval mode) -- for logs and for eyeballing.
"""

import torch.nn as nn


def phase_for_epoch(epoch: int) -> str:
    if not isinstance(epoch, int) or isinstance(epoch, bool) or epoch < 1:
        raise ValueError(f"epoch must be an integer >= 1 (epochs are numbered from 1), got {epoch!r}")
    return "A" if epoch <= PHASE_A_EPOCHS else "B"


def describe_phase(model: nn.Module) -> None:
    backbone = getattr(model, "backbone")
    backbone_params = getattr(model, "backbone_parameters")()
    head_params = getattr(model, "head_parameters")()

    def _trainable(params):
        return sum(p.numel() for p in params if p.requires_grad)

    def _bn_modes(module, kind):
        layers = [m for m in module.modules() if isinstance(m, kind)]
        return sum(m.training for m in layers), sum(not m.training for m in layers)

    b_train, b_eval = _bn_modes(backbone, nn.BatchNorm2d)
    h_train, h_eval = _bn_modes(model, nn.BatchNorm1d)
    print(f"  backbone trainable params : {_trainable(backbone_params):,} of {sum(p.numel() for p in backbone_params):,}")
    print(f"  head-group trainable      : {_trainable(head_params):,} of {sum(p.numel() for p in head_params):,}")
    print(f"  backbone BatchNorm2d      : {b_train} train / {b_eval} eval")
    print(f"  head-group BatchNorm1d    : {h_train} train / {h_eval} eval")

In [16]:
"""
04_02_phase_verification.txt
Phase 3 -- Module 04: Phase Control -- Verification (run once)

Requires 03_00, 03_01, 03_02, 04_00, 04_01 already run. Runs on CPU by default,
on small dummy images (64x64) with randomly initialised backbones, so it needs no
pretrained-weight download and does not touch the GPU. Both models are tested.

All seven checks run as a full set (failures collected, one RuntimeError at the
end), same convention as Modules 01-03.

Checks:
  1. Phase A state      -- backbone: every parameter frozen, every layer (78
                           BatchNorm2d included) in eval mode. Head group: every
                           parameter trainable, layers in train mode, MetaBlock
                           BatchNorm1d in train mode.
  2. Phase A holds      -- 3 real optimizer steps, with model.train() called before
                           each step (the trap). The backbone's ENTIRE state
                           (weights, BatchNorm running statistics, batch counters)
                           is bit-identical afterwards; the head group learned; the
                           MetaBlock BatchNorm1d statistics kept updating.
  3. Control            -- proves check 2 has teeth: a naive loop (weights frozen
                           with requires_grad only, model.train() every step) DOES
                           drift the backbone's BatchNorm statistics.
  4. Phase B unfreezes  -- the same 3 steps in Phase B change the backbone's weights
                           AND its BatchNorm statistics.
  5. Stochastic depth   -- Phase A backbone forward is deterministic (stochastic
                           depth off); Phase B backbone forward is not (it is on).
  6. Repeatability      -- set_phase twice gives identical state; A -> B -> A
                           re-freezes; set_phase after model.eval() (what
                           validation leaves behind) restores the phase correctly.
  7. Epoch mapping      -- 1..5 -> "A", 6 and later -> "B"; epoch < 1 and a bad
                           phase name raise errors.
"""

import copy

import torch
import torch.nn as nn
import torch.nn.functional as F


def _snapshot(module):
    return {k: v.detach().clone() for k, v in module.state_dict().items()}


def _states_equal(a, b):
    return a.keys() == b.keys() and all(torch.equal(a[k], b[k]) for k in a)


def _any_changed(a, b, key_filter=None):
    return any(not torch.equal(a[k], b[k]) for k in a if key_filter is None or key_filter(k))


def _bn_running_changed(a, b):
    """True if any BatchNorm running statistic or batch counter differs."""
    return _any_changed(a, b, lambda k: ("running_mean" in k or "running_var" in k or "num_batches_tracked" in k))


def _make_optimizer(model):
    return torch.optim.AdamW(
        [{"params": getattr(model, "backbone_parameters")(), "lr": 1e-4},
         {"params": getattr(model, "head_parameters")(), "lr": 1e-3}],
        weight_decay=1e-5)


def _run_steps(model, optimizer, n_steps, image_size, device, phase):
    """n real optimizer steps. phase=None -> the naive loop (bare model.train())."""
    for _ in range(n_steps):
        if phase is None:
            model.train()
        else:
            set_phase(model, phase)
        image = torch.randn(4, 3, image_size, image_size, device=device)
        meta = torch.randn(4, METADATA_DIM, device=device)
        labels = torch.randint(0, NUM_CLASSES, (4,), device=device)
        loss = F.cross_entropy(model(image, meta), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


def _steps_or_error(model, optimizer, n_steps, image_size, device, phase):
    """Run the steps; return None on success or an error string (so one broken phase
    is reported as a failure instead of crashing the whole verification)."""
    try:
        _run_steps(model, optimizer, n_steps, image_size, device, phase)
        return None
    except RuntimeError as exc:
        return str(exc).splitlines()[0]


def _head_children(model):
    """The head-group sub-modules that exist on this model."""
    return [(name, getattr(model, name)) for name in ("metadata_branch", "metablock", "classifier")
            if hasattr(model, name)]


def run_phase_verification(image_size: int = 64, device: str = "cpu"):
    failures = []
    print("=" * 60)
    print("PHASE CONTROL VERIFICATION -- Phase 3, Module 04")
    print("=" * 60)
    print(f"  image_size={image_size}, device={device}, backbones randomly initialised (pretrained=False)")

    builders = (("multimodal", build_multimodal_model), ("ablation  ", build_image_only_model))

    def _fresh(builder):
        torch.manual_seed(42)
        return builder(pretrained=False).to(device)

    # --- 1. Phase A state ---
    print("\n[1/7] Phase A state...")
    ok = True
    for tag, builder in builders:
        model = _fresh(builder)
        set_phase(model, "A")
        backbone = getattr(model, "backbone")
        if any(p.requires_grad for p in backbone.parameters()):
            failures.append(f"Phase A state ({tag}): a backbone parameter is still trainable"); ok = False
        if any(m.training for m in backbone.modules()):
            failures.append(f"Phase A state ({tag}): a backbone layer is still in train mode"); ok = False
        n_bn2d = sum(isinstance(m, nn.BatchNorm2d) for m in backbone.modules())
        if n_bn2d != 78:
            failures.append(f"Phase A state ({tag}): expected 78 backbone BatchNorm2d layers, found {n_bn2d}"); ok = False
        if not all(p.requires_grad for p in getattr(model, "head_parameters")()):
            failures.append(f"Phase A state ({tag}): a head-group parameter is frozen"); ok = False
        for name, child in _head_children(model):
            if not all(m.training for m in child.modules()):
                failures.append(f"Phase A state ({tag}): part of '{name}' is not in train mode"); ok = False
        if not model.training:
            failures.append(f"Phase A state ({tag}): the top-level model is not in train mode"); ok = False
    print("  OK -- backbone fully frozen and in eval mode (78 BatchNorm2d); head group trainable and in train mode"
          if ok else "  FAILED -- see summary")

    # --- 2. Phase A holds under real training steps ---
    print("\n[2/7] Phase A holds under real optimizer steps (model.train() called every step)...")
    ok = True
    for tag, builder in builders:
        model = _fresh(builder)
        optimizer = _make_optimizer(model)
        set_phase(model, "A")
        backbone = getattr(model, "backbone")
        before_backbone = _snapshot(backbone)
        before_model = _snapshot(model)
        error = _steps_or_error(model, optimizer, 3, image_size, device, "A")
        if error:
            failures.append(f"Phase A holds ({tag}): a training step failed -- {error}"); ok = False
            continue
        after_backbone = _snapshot(backbone)
        after_model = _snapshot(model)
        if not _states_equal(before_backbone, after_backbone):
            failures.append(f"Phase A holds ({tag}): the backbone changed (weights or BatchNorm statistics)"); ok = False
        for name, child in _head_children(model):
            prefix = name + "."
            if not _any_changed(before_model, after_model, lambda k, p=prefix: k.startswith(p)):
                failures.append(f"Phase A holds ({tag}): '{name}' did not learn"); ok = False
        if hasattr(model, "metablock"):
            metablock = getattr(model, "metablock")
            bn1d = [m for m in metablock.modules() if isinstance(m, nn.BatchNorm1d)]
            if len(bn1d) != 2:
                failures.append(f"Phase A holds ({tag}): expected 2 MetaBlock BatchNorm1d layers, found {len(bn1d)}"); ok = False
            if not _bn_running_changed({k: v for k, v in before_model.items() if k.startswith("metablock.")},
                                       {k: v for k, v in after_model.items() if k.startswith("metablock.")}):
                failures.append(f"Phase A holds ({tag}): MetaBlock BatchNorm1d statistics did not update"); ok = False
            if any(int(m.num_batches_tracked) != 3 for m in bn1d):
                failures.append(f"Phase A holds ({tag}): MetaBlock BatchNorm1d did not count exactly 3 batches"); ok = False
    print("  OK -- backbone bit-identical after 3 steps (weights, BatchNorm statistics, counters); "
          "head group learned; MetaBlock BatchNorm1d kept updating" if ok else "  FAILED -- see summary")

    # --- 3. Control: the trap is real ---
    print("\n[3/7] Control: a naive loop really does drift the frozen backbone...")
    ok = True
    for tag, builder in builders:
        model = _fresh(builder)
        optimizer = _make_optimizer(model)
        backbone = getattr(model, "backbone")
        for p in backbone.parameters():
            p.requires_grad = False            # weights frozen, but NO eval mode
        before_backbone = _snapshot(backbone)
        weights_only = lambda k: not ("running_mean" in k or "running_var" in k or "num_batches_tracked" in k)
        error = _steps_or_error(model, optimizer, 3, image_size, device, None)   # bare model.train() each step
        if error:
            failures.append(f"Control ({tag}): a training step failed -- {error}"); ok = False
            continue
        after_backbone = _snapshot(backbone)
        if not _bn_running_changed(before_backbone, after_backbone):
            failures.append(f"Control ({tag}): the naive loop did NOT drift BatchNorm statistics -- check 2 would prove nothing"); ok = False
        if _any_changed(before_backbone, after_backbone, weights_only):
            failures.append(f"Control ({tag}): frozen weights changed in the naive loop (unexpected)"); ok = False
    print("  OK -- naive loop: weights stay put but BatchNorm statistics drift (the trap Phase A removes)"
          if ok else "  FAILED -- see summary")

    # --- 4. Phase B unfreezes ---
    print("\n[4/7] Phase B unfreezes the backbone...")
    ok = True
    for tag, builder in builders:
        model = _fresh(builder)
        optimizer = _make_optimizer(model)
        set_phase(model, "B")
        backbone = getattr(model, "backbone")
        if not all(p.requires_grad for p in model.parameters()):
            failures.append(f"Phase B ({tag}): some parameter is still frozen"); ok = False
        if not all(m.training for m in model.modules()):
            failures.append(f"Phase B ({tag}): some layer is not in train mode"); ok = False
        before_backbone = _snapshot(backbone)
        error = _steps_or_error(model, optimizer, 3, image_size, device, "B")
        if error:
            failures.append(f"Phase B ({tag}): a training step failed -- {error}"); ok = False
            continue
        after_backbone = _snapshot(backbone)
        weights_only = lambda k: not ("running_mean" in k or "running_var" in k or "num_batches_tracked" in k)
        if not _any_changed(before_backbone, after_backbone, weights_only):
            failures.append(f"Phase B ({tag}): backbone weights did not change"); ok = False
        if not _bn_running_changed(before_backbone, after_backbone):
            failures.append(f"Phase B ({tag}): backbone BatchNorm statistics did not update"); ok = False
    print("  OK -- everything trainable and in train mode; backbone weights and BatchNorm statistics both update"
          if ok else "  FAILED -- see summary")

    # --- 5. Stochastic depth ---
    print("\n[5/7] Stochastic depth (off in Phase A, on in Phase B)...")
    ok = True
    for tag, builder in builders:
        model = _fresh(builder)
        backbone = getattr(model, "backbone")
        x = torch.randn(4, 3, image_size, image_size, device=device)
        torch.manual_seed(7)
        set_phase(model, "A")
        with torch.no_grad():
            a1, a2 = backbone(x), backbone(x)
        set_phase(model, "B")
        with torch.no_grad():
            b1, b2 = backbone(x), backbone(x)
        if not torch.equal(a1, a2):
            failures.append(f"Stochastic depth ({tag}): Phase A backbone output is not deterministic"); ok = False
        if torch.equal(b1, b2):
            failures.append(f"Stochastic depth ({tag}): Phase B backbone output is deterministic (stochastic depth not active)"); ok = False
    print("  OK -- Phase A backbone deterministic; Phase B backbone stochastic" if ok else "  FAILED -- see summary")

    # --- 6. Repeatability ---
    print("\n[6/7] Repeatability...")
    ok = True

    def _flags(m):
        return ([p.requires_grad for p in m.parameters()], [mod.training for mod in m.modules()])

    for tag, builder in builders:
        model = _fresh(builder)
        set_phase(model, "A"); once = _flags(model)
        set_phase(model, "A"); twice = _flags(model)
        if once != twice:
            failures.append(f"Repeatability ({tag}): calling set_phase('A') twice changed the state"); ok = False
        set_phase(model, "B"); set_phase(model, "A")
        if _flags(model) != once:
            failures.append(f"Repeatability ({tag}): A -> B -> A did not restore Phase A"); ok = False
        model.eval()                                # what validation leaves behind
        set_phase(model, "A")
        if _flags(model) != once:
            failures.append(f"Repeatability ({tag}): set_phase('A') after model.eval() did not restore Phase A"); ok = False
        model.eval()
        set_phase(model, "B")
        if not all(m.training for m in model.modules()):
            failures.append(f"Repeatability ({tag}): set_phase('B') after model.eval() did not restore train mode"); ok = False
    print("  OK -- idempotent; A -> B -> A re-freezes; restores the phase after model.eval()"
          if ok else "  FAILED -- see summary")

    # --- 7. Epoch mapping and bad inputs ---
    print("\n[7/7] Epoch mapping and bad inputs...")
    ok = True
    expected = {1: "A", 2: "A", PHASE_A_EPOCHS: "A", PHASE_A_EPOCHS + 1: "B", 50: "B", 100: "B"}
    for epoch, want in expected.items():
        if phase_for_epoch(epoch) != want:
            failures.append(f"Epoch mapping: epoch {epoch} gave {phase_for_epoch(epoch)!r}, expected {want!r}"); ok = False
    for bad_epoch in (0, -1, 1.5, "3", True):
        try:
            phase_for_epoch(bad_epoch)
            failures.append(f"Epoch mapping: phase_for_epoch({bad_epoch!r}) did not raise"); ok = False
        except ValueError:
            pass
    try:
        set_phase(_fresh(build_image_only_model), "C")
        failures.append("Epoch mapping: set_phase(model, 'C') did not raise"); ok = False
    except ValueError:
        pass
    try:
        set_phase(nn.Linear(2, 2), "A")
        failures.append("Epoch mapping: set_phase on a non-Phase-3 model did not raise"); ok = False
    except TypeError:
        pass
    print(f"  OK -- epochs 1-{PHASE_A_EPOCHS} -> A, {PHASE_A_EPOCHS + 1}+ -> B; bad epochs, bad phase and wrong model type all raise"
          if ok else "  FAILED -- see summary")

    print("\n" + "=" * 60)
    if failures:
        print(f"VERIFICATION FAILED -- {len(failures)} issue(s):")
        for msg in failures:
            print(f"  - {msg}")
        print("=" * 60)
        raise RuntimeError(f"Phase control verification failed with {len(failures)} issue(s).")
    print("VERIFICATION PASSED -- all checks OK. Module 04 is safe to build on.")
    print("=" * 60)


# --- Usage ---
# run_phase_verification()

In [17]:
run_phase_verification()

PHASE CONTROL VERIFICATION -- Phase 3, Module 04
  image_size=64, device=cpu, backbones randomly initialised (pretrained=False)

[1/7] Phase A state...
  OK -- backbone fully frozen and in eval mode (78 BatchNorm2d); head group trainable and in train mode

[2/7] Phase A holds under real optimizer steps (model.train() called every step)...
  OK -- backbone bit-identical after 3 steps (weights, BatchNorm statistics, counters); head group learned; MetaBlock BatchNorm1d kept updating

[3/7] Control: a naive loop really does drift the frozen backbone...
  OK -- naive loop: weights stay put but BatchNorm statistics drift (the trap Phase A removes)

[4/7] Phase B unfreezes the backbone...
  OK -- everything trainable and in train mode; backbone weights and BatchNorm statistics both update

[5/7] Stochastic depth (off in Phase A, on in Phase B)...
  OK -- Phase A backbone deterministic; Phase B backbone stochastic

[6/7] Repeatability...
  OK -- idempotent; A -> B -> A re-freezes; restores the

# Module 5

In [18]:
"""
05_00_optimizer_builder.txt
Phase 3 -- Module 05: Optimizer -- settings and builder (reusable)

Requires 03_02 (models) and 04_00 (set_phase) already run.

ONE AdamW, TWO PARAMETER GROUPS (Phase-3 plan)
  group 0  "backbone"  -- model.backbone_parameters()   lr 1e-4
  group 1  "head"      -- model.head_parameters()       lr 1e-3
                          (metadata branch, MetaBlock, classification head;
                           for the ablation: the classification head only)
  weight decay 1e-5 on ALL parameters in both groups (biases and BatchNorm
  weights included) -- exactly what Phase 2 did, so the comparison to the locked
  Phase 2 baselines stays clean. Flat learning rates, no scheduler (as Phase 2).

WHY ONE OPTIMIZER FOR BOTH PHASES
  In Phase A the backbone's requires_grad is False, so it receives no gradients.
  AdamW skips any parameter whose gradient is None -- no update, no weight decay,
  no optimizer state. At epoch 6, set_phase(model, "B") unfreezes the backbone and
  the SAME optimizer starts updating it at 1e-4. No rebuild at the phase boundary.

The group order and the "name" labels are fixed, so Module 06 can confirm on
resume that a saved optimizer state lines up with the optimizer built here.

The same function serves the multimodal model and the matched ablation: identical
settings, so the only difference between the two is the metadata branch + MetaBlock.
"""

import torch

BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-5


def build_optimizer(model) -> torch.optim.AdamW:
    """Build the Phase 3 optimizer. Call AFTER the model is built (and moved to its device)."""
    backbone_params = getattr(model, "backbone_parameters")()
    head_params = getattr(model, "head_parameters")()
    return torch.optim.AdamW(
        [{"params": backbone_params, "lr": BACKBONE_LR, "name": "backbone"},
         {"params": head_params, "lr": HEAD_LR, "name": "head"}],
        weight_decay=WEIGHT_DECAY)


print("Optimizer settings loaded.")
print(f"  AdamW, weight decay {WEIGHT_DECAY} on all parameters, flat learning rates (no scheduler)")
print(f"  group 0 'backbone' lr {BACKBONE_LR}   |   group 1 'head' lr {HEAD_LR}")
print("  One optimizer serves both phases: frozen backbone parameters get no gradient, so AdamW skips them.")

Optimizer settings loaded.
  AdamW, weight decay 1e-05 on all parameters, flat learning rates (no scheduler)
  group 0 'backbone' lr 0.0001   |   group 1 'head' lr 0.001
  One optimizer serves both phases: frozen backbone parameters get no gradient, so AdamW skips them.


In [19]:
"""
05_01_optimizer_verification.txt
Phase 3 -- Module 05: Optimizer -- Verification (run once)

Requires 03_00-03_02, 04_00, 05_00 already run. Runs on CPU by default, on small
dummy images (64x64) with randomly initialised backbones (no download, no GPU).
Both models are tested.

All five checks run as a full set (failures collected, one RuntimeError at the
end), same convention as Modules 01-04.

Checks:
  1. Groups          -- exactly two groups, named "backbone" then "head"; disjoint;
                        together they cover every model parameter exactly once;
                        learning rates 1e-4 / 1e-3, weight decay 1e-5 in both.
  2. Phase A         -- 3 real steps: the backbone has NO optimizer state and is
                        bit-identical afterwards (weights, BatchNorm statistics);
                        weight decay did not touch it; the head learned and has state.
  3. Phase B         -- after set_phase(model, "B") on the SAME optimizer (no rebuild),
                        the backbone weights update and now have optimizer state.
  4. Both models     -- the ablation gets identical settings (same rates, decay,
                        group names).
  5. Resume          -- optimizer state_dict -> fresh optimizer -> load_state_dict
                        keeps the group order, names, rates and per-parameter state
                        (the Module 06 checkpoint relies on this).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


def _opt_snapshot(module):
    return {k: v.detach().clone() for k, v in module.state_dict().items()}


def _opt_run_steps(model, optimizer, n_steps, image_size, device, phase):
    for _ in range(n_steps):
        set_phase(model, phase)
        image = torch.randn(4, 3, image_size, image_size, device=device)
        meta = torch.randn(4, METADATA_DIM, device=device)
        labels = torch.randint(0, NUM_CLASSES, (4,), device=device)
        loss = F.cross_entropy(model(image, meta), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


def _opt_error_or_none(model, optimizer, n_steps, image_size, device, phase):
    try:
        _opt_run_steps(model, optimizer, n_steps, image_size, device, phase)
        return None
    except RuntimeError as exc:
        return str(exc).splitlines()[0]


def _has_state(optimizer, params):
    return any(len(optimizer.state.get(p, {})) > 0 for p in params)


def run_optimizer_verification(image_size: int = 64, device: str = "cpu"):
    failures = []
    print("=" * 60)
    print("OPTIMIZER VERIFICATION -- Phase 3, Module 05")
    print("=" * 60)
    print(f"  image_size={image_size}, device={device}, backbones randomly initialised (pretrained=False)")

    builders = (("multimodal", build_multimodal_model), ("ablation  ", build_image_only_model))

    def _fresh(builder):
        torch.manual_seed(42)
        return builder(pretrained=False).to(device)

    # --- 1. Groups ---
    print("\n[1/5] Groups...")
    ok = True
    for tag, builder in builders:
        model = _fresh(builder)
        opt = build_optimizer(model)
        groups = opt.param_groups
        if len(groups) != 2 or [g.get("name") for g in groups] != ["backbone", "head"]:
            failures.append(f"Groups ({tag}): expected two groups named ['backbone', 'head'], got {[g.get('name') for g in groups]}")
            ok = False
            continue
        ids = [[id(p) for p in g["params"]] for g in groups]
        if set(ids[0]) & set(ids[1]):
            failures.append(f"Groups ({tag}): a parameter is in both groups"); ok = False
        if len(ids[0]) + len(ids[1]) != len(list(model.parameters())) or \
                set(ids[0]) | set(ids[1]) != {id(p) for p in model.parameters()}:
            failures.append(f"Groups ({tag}): groups do not cover every model parameter exactly once"); ok = False
        if groups[0]["lr"] != BACKBONE_LR or groups[1]["lr"] != HEAD_LR:
            failures.append(f"Groups ({tag}): learning rates are {groups[0]['lr']} / {groups[1]['lr']}"); ok = False
        if any(g["weight_decay"] != WEIGHT_DECAY for g in groups):
            failures.append(f"Groups ({tag}): weight decay is not {WEIGHT_DECAY} in both groups"); ok = False
        if not isinstance(opt, torch.optim.AdamW):
            failures.append(f"Groups ({tag}): optimizer is not AdamW"); ok = False
        n_back, n_head = len(ids[0]), len(ids[1])
        print(f"    {tag}: backbone {n_back} tensors, head {n_head} tensors")
    print("  OK -- two disjoint groups covering every parameter once; lr 1e-4 / 1e-3; weight decay 1e-5; AdamW"
          if ok else "  FAILED -- see summary")

    # --- 2. Phase A ---
    print("\n[2/5] Phase A: the backbone is untouched by the optimizer...")
    ok = True
    for tag, builder in builders:
        model = _fresh(builder)
        opt = build_optimizer(model)
        set_phase(model, "A")
        backbone = getattr(model, "backbone")
        head_params = getattr(model, "head_parameters")()
        before_backbone = _opt_snapshot(backbone)
        before_head = [p.detach().clone() for p in head_params]
        error = _opt_error_or_none(model, opt, 3, image_size, device, "A")
        if error:
            failures.append(f"Phase A ({tag}): a training step failed -- {error}"); ok = False
            continue
        after_backbone = _opt_snapshot(backbone)
        if _has_state(opt, getattr(model, "backbone_parameters")()):
            failures.append(f"Phase A ({tag}): the backbone has optimizer state (it should have none)"); ok = False
        if before_backbone.keys() != after_backbone.keys() or \
                not all(torch.equal(before_backbone[k], after_backbone[k]) for k in before_backbone):
            failures.append(f"Phase A ({tag}): the backbone changed (update, weight decay or BatchNorm statistics)"); ok = False
        if not _has_state(opt, head_params):
            failures.append(f"Phase A ({tag}): the head group has no optimizer state (it did not train)"); ok = False
        if not any(not torch.equal(a, p.detach()) for a, p in zip(before_head, head_params)):
            failures.append(f"Phase A ({tag}): the head group did not change"); ok = False
    print("  OK -- backbone: no optimizer state, bit-identical after 3 steps (no update, no weight decay); head learned"
          if ok else "  FAILED -- see summary")

    # --- 3. Phase B on the same optimizer ---
    print("\n[3/5] Phase B on the SAME optimizer (no rebuild)...")
    ok = True
    for tag, builder in builders:
        model = _fresh(builder)
        opt = build_optimizer(model)
        error = _opt_error_or_none(model, opt, 2, image_size, device, "A")
        if error:
            failures.append(f"Phase B ({tag}): Phase A warm-up failed -- {error}"); ok = False
            continue
        backbone_params = getattr(model, "backbone_parameters")()
        before = [p.detach().clone() for p in backbone_params]
        error = _opt_error_or_none(model, opt, 2, image_size, device, "B")
        if error:
            failures.append(f"Phase B ({tag}): a training step failed -- {error}"); ok = False
            continue
        if not any(not torch.equal(b, p.detach()) for b, p in zip(before, backbone_params)):
            failures.append(f"Phase B ({tag}): backbone weights did not update after the switch"); ok = False
        if not _has_state(opt, backbone_params):
            failures.append(f"Phase B ({tag}): the backbone has no optimizer state after the switch"); ok = False
        if not all(p.grad is not None for p in backbone_params):
            failures.append(f"Phase B ({tag}): some backbone parameter received no gradient"); ok = False
    print("  OK -- after set_phase(model, 'B') the same optimizer updates the backbone" if ok else "  FAILED -- see summary")

    # --- 4. Both models ---
    print("\n[4/5] Both models get identical settings...")
    ok = True
    settings = []
    for tag, builder in builders:
        opt = build_optimizer(_fresh(builder))
        settings.append([(g.get("name"), g["lr"], g["weight_decay"], g["betas"], g["eps"]) for g in opt.param_groups])
    if settings[0] != settings[1]:
        failures.append("Both models: the multimodal and ablation optimizers have different settings"); ok = False
    print("  OK -- same group names, rates, weight decay, betas and eps" if ok else "  FAILED -- see summary")

    # --- 5. Resume round trip ---
    print("\n[5/5] Optimizer state survives save -> load...")
    ok = True
    for tag, builder in builders:
        model = _fresh(builder)
        opt = build_optimizer(model)
        error = _opt_error_or_none(model, opt, 2, image_size, device, "B")
        if error:
            failures.append(f"Resume ({tag}): a training step failed -- {error}"); ok = False
            continue
        saved = opt.state_dict()
        opt2 = build_optimizer(model)
        opt2.load_state_dict(saved)
        if [g.get("name") for g in opt2.param_groups] != ["backbone", "head"]:
            failures.append(f"Resume ({tag}): group names/order changed after load"); ok = False
        if [(g["lr"], g["weight_decay"]) for g in opt2.param_groups] != [(g["lr"], g["weight_decay"]) for g in opt.param_groups]:
            failures.append(f"Resume ({tag}): rates or weight decay changed after load"); ok = False
        params = [p for g in opt.param_groups for p in g["params"]]
        params2 = [p for g in opt2.param_groups for p in g["params"]]
        same_state = all(
            torch.equal(opt.state[a]["exp_avg"], opt2.state[b]["exp_avg"])
            and torch.equal(opt.state[a]["exp_avg_sq"], opt2.state[b]["exp_avg_sq"])
            and float(opt.state[a]["step"]) == float(opt2.state[b]["step"])
            for a, b in zip(params, params2) if a in opt.state)
        if not same_state or len(opt.state) != len(opt2.state):
            failures.append(f"Resume ({tag}): per-parameter optimizer state differs after load"); ok = False
    print("  OK -- group order, names, rates and per-parameter state identical after load" if ok else "  FAILED -- see summary")

    print("\n" + "=" * 60)
    if failures:
        print(f"VERIFICATION FAILED -- {len(failures)} issue(s):")
        for msg in failures:
            print(f"  - {msg}")
        print("=" * 60)
        raise RuntimeError(f"Optimizer verification failed with {len(failures)} issue(s).")
    print("VERIFICATION PASSED -- all checks OK. Module 05 is safe to build on.")
    print("=" * 60)


# --- Usage ---
# run_optimizer_verification()

In [20]:
run_optimizer_verification()

OPTIMIZER VERIFICATION -- Phase 3, Module 05
  image_size=64, device=cpu, backbones randomly initialised (pretrained=False)

[1/5] Groups...
    multimodal: backbone 338 tensors, head 12 tensors
    ablation  : backbone 338 tensors, head 2 tensors
  OK -- two disjoint groups covering every parameter once; lr 1e-4 / 1e-3; weight decay 1e-5; AdamW

[2/5] Phase A: the backbone is untouched by the optimizer...
  OK -- backbone: no optimizer state, bit-identical after 3 steps (no update, no weight decay); head learned

[3/5] Phase B on the SAME optimizer (no rebuild)...
  OK -- after set_phase(model, 'B') the same optimizer updates the backbone

[4/5] Both models get identical settings...
  OK -- same group names, rates, weight decay, betas and eps

[5/5] Optimizer state survives save -> load...
  OK -- group order, names, rates and per-parameter state identical after load

VERIFICATION PASSED -- all checks OK. Module 05 is safe to build on.


# Module 6

In [21]:
"""
06_00_checkpoint_io.txt
Phase 3 -- Module 06: Checkpoint and Resume -- file I/O (reusable)

Requires 01_00 (N_FOLDS) and 04_01 (phase_for_epoch) already run.

Two files per fold, both written atomically (temporary file + os.replace, so a power
cut mid-write leaves the previous good file intact, never a half-written one):

  <out_dir>/fold_{fold_num}_resume.pt
      Full training state, overwritten EVERY epoch. Used only to resume a fold
      that was interrupted, or to recognise a fold that is already finished.
  <out_dir>/fold_{fold_num}_best.pt
      Best-epoch weights only (plus the epoch and AUC they came from), overwritten
      only when validation AUC-ROC strictly improves. This is the file used for
      evaluation and for fold selection.

WHAT THE RESUME FILE HOLDS
  identity                 which model ("multimodal" / "ablation") and which fold it belongs to
  epoch                    the last COMPLETED epoch, numbered from 1 (next epoch = epoch + 1)
  phase                    "A" or "B" for that epoch -- cross-checked against phase_for_epoch
  model / optimizer state  full state dicts (weights + BatchNorm statistics; Adam moments)
  rng_state                Python, NumPy and torch (CPU and CUDA) random generators
  sampler_generator_state  the balanced sampler's own torch.Generator (None if no sampler)
  best_metric, best_epoch  best validation AUC-ROC so far and the epoch it came from
                           (ALL epochs compete, including Phase A epochs -- locked decision)
  early_stop_counter       non-improving epochs counted so far (counting starts at epoch 6)
  history                  list of per-epoch records (plain dicts), so run_config.json can be
                           built correctly after a resume
  finished                 True once the fold is DONE: early stopping fired OR the last
                           allowed epoch was completed. The training loop sets it (Module 07).
                           Without it a re-run cannot tell "done" from "paused".

IDENTITY CHECK
  Loading refuses a checkpoint that belongs to a different model kind or fold (the
  stale-checkpoint / mis-pasted-cell incident from Phase 2). A corrupt or truncated file
  raises a clear error -- it is never silently treated as "no checkpoint".

BUSY FILES (Windows)
  Antivirus, the search indexer or a backup tool can hold a file open for a moment. The swap
  (os.replace) then fails with PermissionError although nothing is wrong. Every save in this
  module and in Module 07 goes through _replace_with_retry: it tries again after 0.5, 1, 2, 3 and
  5 seconds (6 attempts, about 11.5 s in all) and prints a short "file busy" line each time. Only
  PermissionError is retried; any other error, or a lock that outlasts every wait, is raised
  exactly as before, with the previous good file untouched. Change REPLACE_RETRY_WAITS to tune it.

Usage inside the training loop (Module 07):

    resume_path, best_path = checkpoint_paths(out_dir, fold_num)
    identity = make_identity("multimodal", fold_num)
    state = load_resume_state(resume_path, identity)      # None if there is nothing to resume
    ...
    save_resume_state(resume_path, identity, epoch, phase_for_epoch(epoch), model, optimizer,
                      best_metric, best_epoch, early_stop_counter, history,
                      sampler_generator_state=sampler.generator.get_state(), finished=is_done)
    if val_auc > best_metric:
        save_best_weights(best_path, identity, model, epoch, val_auc)
"""

import os
import random
import tempfile
import time

import numpy as np
import torch

MODEL_KINDS = ("multimodal", "ablation")
CHECKPOINT_FORMAT_VERSION = 1


class CheckpointError(RuntimeError):
    """A checkpoint file is unreadable, incomplete or internally inconsistent."""


class CheckpointMismatchError(CheckpointError):
    """A checkpoint exists but belongs to a different model kind or fold."""


def checkpoint_paths(out_dir, fold_num: int):
    fold_num = int(fold_num)
    return (os.path.join(str(out_dir), f"fold_{fold_num}_resume.pt"),
            os.path.join(str(out_dir), f"fold_{fold_num}_best.pt"))


def make_identity(model_kind: str, fold_num: int) -> dict:
    if model_kind not in MODEL_KINDS:
        raise ValueError(f"model_kind must be one of {MODEL_KINDS}, got {model_kind!r}")
    if not isinstance(fold_num, int) or isinstance(fold_num, bool) or not 0 <= fold_num < N_FOLDS:
        raise ValueError(f"fold_num must be an integer in 0..{N_FOLDS - 1}, got {fold_num!r}")
    return {"model_kind": model_kind, "fold": fold_num}


# Seconds to wait between attempts when the destination file is briefly locked (see BUSY FILES above).
REPLACE_RETRY_WAITS = (0.5, 1.0, 2.0, 3.0, 5.0)


def _replace_with_retry(tmp_path, final_path):
    """os.replace(tmp_path, final_path), retried while the destination is momentarily locked.
    Only PermissionError is retried (what a Windows sharing violation raises). Any other error, and
    a lock that outlasts every wait, is raised as it happened."""
    attempts = len(REPLACE_RETRY_WAITS) + 1
    for attempt in range(attempts):
        try:
            os.replace(tmp_path, final_path)
            return
        except PermissionError:
            if attempt == attempts - 1:
                raise
            wait = REPLACE_RETRY_WAITS[attempt]
            print(f"  (file busy: {os.path.basename(str(final_path))} -- retrying in {wait:g} s, "
                  f"attempt {attempt + 2} of {attempts})")
            time.sleep(wait)


def _remove_quietly(path):
    """Best-effort delete of a leftover temporary file. A failure to delete it (for example a lock)
    must never hide the real error, so it is ignored -- a stray .tmp file is harmless."""
    try:
        if os.path.exists(path):
            os.remove(path)
    except OSError:
        pass


def _atomic_torch_save(obj, final_path: str):
    """Write obj to a temporary file in the same directory, then atomically rename it over
    final_path. os.replace is atomic on POSIX and Windows, so a crash mid-write never leaves a
    corrupted checkpoint -- the old file stays valid until the new one is fully written."""
    final_dir = os.path.dirname(final_path) or "."
    os.makedirs(final_dir, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=final_dir, suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            torch.save(obj, f)
            f.flush()
            os.fsync(f.fileno())
        _replace_with_retry(tmp_path, final_path)
    except BaseException:
        _remove_quietly(tmp_path)
        raise


def capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])
    if torch.cuda.is_available() and "torch_cuda" in state:
        torch.cuda.set_rng_state_all(state["torch_cuda"])


def _load_checked(path, expected_identity, kind_label):
    """torch.load + format-version and identity checks. Returns None if the file is absent."""
    if not os.path.exists(path):
        return None
    try:
        state = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as exc:
        raise CheckpointError(
            f"{kind_label} file {path} could not be read ({type(exc).__name__}: {exc}). "
            f"It is NOT being ignored: delete it deliberately if you want this fold to start over.") from exc
    if not isinstance(state, dict) or state.get("format_version") != CHECKPOINT_FORMAT_VERSION:
        raise CheckpointError(f"{kind_label} file {path} has an unexpected format "
                              f"(expected format_version {CHECKPOINT_FORMAT_VERSION}).")
    if state.get("identity") != expected_identity:
        raise CheckpointMismatchError(
            f"{kind_label} file {path} belongs to {state.get('identity')}, but this run is "
            f"{expected_identity}. Refusing to use it (a stale checkpoint from another script or fold?).")
    return state


def save_resume_state(resume_path, identity, epoch, phase, model, optimizer,
                      best_metric, best_epoch, early_stop_counter, history,
                      sampler_generator_state=None, finished=False):
    if not isinstance(epoch, int) or isinstance(epoch, bool) or epoch < 1:
        raise ValueError(f"epoch must be an integer >= 1, got {epoch!r}")
    if phase != phase_for_epoch(epoch):
        raise ValueError(f"phase {phase!r} does not match epoch {epoch} (phase_for_epoch says "
                         f"{phase_for_epoch(epoch)!r})")
    state = {
        "format_version": CHECKPOINT_FORMAT_VERSION,
        "identity": dict(identity),
        "epoch": epoch,
        "phase": phase,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "rng_state": capture_rng_state(),
        "sampler_generator_state": sampler_generator_state,
        "best_metric": best_metric,
        "best_epoch": best_epoch,
        "early_stop_counter": early_stop_counter,
        "history": list(history),
        "finished": bool(finished),
    }
    _atomic_torch_save(state, resume_path)


def load_resume_state(resume_path, expected_identity):
    """Return the saved state dict, or None if there is no resume file. Raises CheckpointError /
    CheckpointMismatchError for a corrupt, foreign or inconsistent file -- never silently ignores it."""
    state = _load_checked(resume_path, expected_identity, "Resume")
    if state is None:
        return None
    if state.get("phase") != phase_for_epoch(state["epoch"]):
        raise CheckpointError(f"Resume file {resume_path} is inconsistent: epoch {state['epoch']} "
                              f"is saved with phase {state.get('phase')!r}.")
    return state


def save_best_weights(best_path, identity, model, epoch, val_auc):
    _atomic_torch_save({
        "format_version": CHECKPOINT_FORMAT_VERSION,
        "identity": dict(identity),
        "epoch": epoch,
        "val_auc": val_auc,
        "model_state_dict": model.state_dict(),
    }, best_path)


def load_best_weights(best_path, expected_identity, model):
    """Load the best-epoch weights into `model`. Returns {"epoch", "val_auc"}; None if no file."""
    state = _load_checked(best_path, expected_identity, "Best-weights")
    if state is None:
        return None
    model.load_state_dict(state["model_state_dict"])
    return {"epoch": state["epoch"], "val_auc": state["val_auc"]}


print("Checkpoint I/O loaded.")
print("  fold_{n}_resume.pt : full state every epoch (atomic writes; identity, phase and 'finished' flag checked)")
print("  fold_{n}_best.pt   : best-epoch weights + the epoch and AUC they came from")
print(f"  saves retry a briefly-locked file: {len(REPLACE_RETRY_WAITS) + 1} attempts over about {sum(REPLACE_RETRY_WAITS):g} s")

Checkpoint I/O loaded.
  fold_{n}_resume.pt : full state every epoch (atomic writes; identity, phase and 'finished' flag checked)
  fold_{n}_best.pt   : best-epoch weights + the epoch and AUC they came from
  saves retry a briefly-locked file: 6 attempts over about 11.5 s


In [22]:
"""
06_01_checkpoint_verification.txt
Phase 3 -- Module 06: Checkpoint and Resume -- Verification (run once)

Requires 01_00, 04_00, 04_01, 05_00, 06_00 already run. Runs on CPU with a tiny stand-in
model and synthetic data -- no dataset, no pretrained weights, no GPU. The stand-in has the
same structure the real models expose (a backbone with BatchNorm, a head with BatchNorm1d and
Dropout, backbone_parameters() / head_parameters()), so set_phase and build_optimizer run on it
exactly as they will on the real models. Files go to a temporary folder that is deleted afterwards.

All seven checks run as a full set (failures collected, one RuntimeError at the end).

Checks:
  1. Atomic write    -- a simulated crash in the middle of a save leaves the previous
                        checkpoint intact and readable, and no stray temporary file.
  2. Round trip      -- every field comes back identical: model and optimizer state, epoch,
                        phase, best metric/epoch, patience counter, history, finished flag,
                        sampler generator state; restored RNGs produce the same next numbers.
  3. Interrupted run -- training 7 epochs straight equals training with a save / "crash" /
                        rebuild / load / resume in the middle, interrupted mid-Phase-A (after
                        epoch 3) and exactly on the A -> B boundary (after epoch 5): bit-identical
                        weights, BatchNorm statistics and optimizer state at the end. Uses a
                        balanced sampler with its own generator, and RNG-dependent noise.
  4. Refusals        -- wrong model kind, wrong fold, phase not matching the epoch, a truncated
                        file and an unknown format version are all refused with clear errors.
  5. Flags / absence -- the finished flag survives a save and load (default False); no file
                        gives None, never an error.
  6. Best weights    -- the best file round-trips weights, epoch and AUC into a fresh model, holds
                        no optimizer state, and refuses a foreign identity.
  7. Busy file       -- a destination file that is locked for a moment (simulated Windows
                        PermissionError, no real waiting) is waited out: the save succeeds after the
                        retries, for the resume file and the best file. A lock that never clears
                        raises PermissionError after exactly 6 attempts with the previous checkpoint
                        untouched and no temporary file left. Any other error is NOT retried.
"""

import contextlib
import io
import os
import random
import tempfile
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler


class _TinyNet(nn.Module):
    """Stand-in with the same structure as the real models, small enough to train in milliseconds."""

    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(nn.Conv2d(3, 4, 3, padding=1), nn.BatchNorm2d(4), nn.ReLU(),
                                      nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Sequential(nn.Linear(4, 6), nn.BatchNorm1d(6), nn.ReLU(), nn.Dropout(0.5),
                                  nn.Linear(6, 7))

    def forward(self, image, metadata):
        return self.head(self.backbone(image))

    def backbone_parameters(self):
        return list(self.backbone.parameters())

    def head_parameters(self):
        return list(self.head.parameters())


_N_SAMPLES = 24
_BATCH = 8
_LAST_EPOCH = 7            # crosses the A -> B boundary (Phase A = epochs 1-5)


def _make_data():
    g = torch.Generator()
    g.manual_seed(2024)
    images = torch.randn(_N_SAMPLES, 3, 8, 8, generator=g)
    meta = torch.randn(_N_SAMPLES, METADATA_DIM, generator=g)
    labels = torch.arange(_N_SAMPLES) % 7
    return TensorDataset(images, meta, labels)


def _seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def _build_run(seed, sampler_generator_state=None):
    """model, optimizer, sampler, loader -- the same construction a real script would do."""
    _seed_everything(seed)
    model = _TinyNet()
    optimizer = build_optimizer(model)
    generator = torch.Generator()
    if sampler_generator_state is not None:
        generator.set_state(sampler_generator_state)
    else:
        generator.manual_seed(123)
    sampler = WeightedRandomSampler(torch.ones(_N_SAMPLES), num_samples=_N_SAMPLES,
                                    replacement=True, generator=generator)
    loader = DataLoader(_make_data(), batch_size=_BATCH, sampler=sampler, num_workers=0)
    return model, optimizer, sampler, loader


def _train_epochs(model, optimizer, loader, first_epoch, last_epoch, history):
    for epoch in range(first_epoch, last_epoch + 1):
        phase = phase_for_epoch(epoch)
        set_phase(model, phase)
        total = 0.0
        for image, meta, labels in loader:
            # RNG-dependent noise from all three global generators, like real augmentation:
            image = image + 0.05 * torch.randn_like(image) + 0.01 * random.random() + 0.01 * float(np.random.rand())
            loss = nn.functional.cross_entropy(model(image, meta), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += float(loss.detach())
        history.append({"epoch": epoch, "phase": phase, "train_loss": total})


def _save(path, identity, epoch, model, optimizer, sampler, history, **kwargs):
    save_resume_state(path, identity, epoch, phase_for_epoch(epoch), model, optimizer,
                      best_metric=kwargs.get("best_metric", 0.5), best_epoch=kwargs.get("best_epoch", epoch),
                      early_stop_counter=kwargs.get("early_stop_counter", 0), history=history,
                      sampler_generator_state=sampler.generator.get_state(),
                      finished=kwargs.get("finished", False))


def _deep_equal(a, b):
    if isinstance(a, torch.Tensor):
        return isinstance(b, torch.Tensor) and a.shape == b.shape and torch.equal(a.cpu(), b.cpu())
    if isinstance(a, dict):
        return isinstance(b, dict) and a.keys() == b.keys() and all(_deep_equal(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)):
        return type(a) == type(b) and len(a) == len(b) and all(_deep_equal(x, y) for x, y in zip(a, b))
    return a == b


def run_checkpoint_verification():
    failures = []
    print("=" * 60)
    print("CHECKPOINT VERIFICATION -- Phase 3, Module 06")
    print("=" * 60)
    print("  CPU, tiny stand-in model, synthetic data, temporary folder")

    identity = make_identity("multimodal", 0)

    with tempfile.TemporaryDirectory() as tmp:
        resume_path, best_path = checkpoint_paths(tmp, 0)

        # --- 1. Atomic write ---
        print("\n[1/7] Atomic write survives a crash mid-save...")
        ok = True
        model, optimizer, sampler, loader = _build_run(1)
        history = []
        _train_epochs(model, optimizer, loader, 1, 2, history)
        _save(resume_path, identity, 2, model, optimizer, sampler, history)
        with open(resume_path, "rb") as f:
            good_bytes = f.read()
        _train_epochs(model, optimizer, loader, 3, 3, history)

        real_save = torch.save

        def _crashing_save(obj, f, *args, **kwargs):
            f.write(b"partial garbage")          # a half-written file ...
            raise OSError("simulated power cut")  # ... and then the power goes

        torch.save = _crashing_save
        try:
            try:
                _save(resume_path, identity, 3, model, optimizer, sampler, history)
                failures.append("Atomic write: the simulated crash did not raise"); ok = False
            except OSError:
                pass
        finally:
            torch.save = real_save
        try:
            with open(resume_path, "rb") as f:
                if f.read() != good_bytes:
                    failures.append("Atomic write: the previous checkpoint was modified by the failed save"); ok = False
        except OSError:
            failures.append("Atomic write: the previous checkpoint was destroyed by the failed save"); ok = False
        leftovers = [n for n in os.listdir(tmp) if n.endswith(".tmp")]
        if leftovers:
            failures.append(f"Atomic write: stray temporary file(s) left behind: {leftovers}"); ok = False
        try:
            survivor = load_resume_state(resume_path, identity)
            if survivor is None or survivor["epoch"] != 2:
                failures.append("Atomic write: the surviving checkpoint is missing or is not the epoch-2 one"); ok = False
        except CheckpointError as exc:
            failures.append(f"Atomic write: the surviving checkpoint is unreadable -- {exc}"); ok = False
        print("  OK -- previous checkpoint intact and readable, no temporary file left" if ok else "  FAILED -- see summary")

        # --- 2. Round trip ---
        print("\n[2/7] Round trip of every field...")
        ok = True
        model, optimizer, sampler, loader = _build_run(2)
        history = []
        _train_epochs(model, optimizer, loader, 1, 6, history)        # ends in Phase B
        _save(resume_path, identity, 6, model, optimizer, sampler, history,
              best_metric=0.9123, best_epoch=4, early_stop_counter=2, finished=False)
        expected_next = (torch.rand(3), random.random(), float(np.random.rand()))   # what the RNGs produce next
        gen_state_at_save = sampler.generator.get_state().clone()
        # (the draws above advanced the generators; capture_rng_state saved them BEFORE those draws)
        state = load_resume_state(resume_path, identity)
        if state is None:
            failures.append("Round trip: the file could not be loaded"); ok = False
        else:
            if (state["epoch"], state["phase"], state["best_metric"], state["best_epoch"],
                    state["early_stop_counter"], state["finished"]) != (6, "B", 0.9123, 4, 2, False):
                failures.append("Round trip: a scalar field changed"); ok = False
            if state["history"] != history:
                failures.append("Round trip: history changed"); ok = False
            if state["identity"] != identity:
                failures.append("Round trip: identity changed"); ok = False
            if not _deep_equal(state["model_state_dict"], model.state_dict()):
                failures.append("Round trip: model state changed"); ok = False
            if not _deep_equal(state["optimizer_state_dict"], optimizer.state_dict()):
                failures.append("Round trip: optimizer state changed"); ok = False
            if not _deep_equal(state["sampler_generator_state"], gen_state_at_save):
                failures.append("Round trip: sampler generator state changed"); ok = False
            restore_rng_state(state["rng_state"])
            replay = (torch.rand(3), random.random(), float(np.random.rand()))
            if not (torch.equal(replay[0], expected_next[0]) and replay[1] == expected_next[1]
                    and replay[2] == expected_next[2]):
                failures.append("Round trip: restored RNGs do not reproduce the same next numbers"); ok = False
        print("  OK -- model, optimizer, counters, history, sampler generator and RNGs all identical" if ok
              else "  FAILED -- see summary")

        # --- 3. Interrupted run == uninterrupted run ---
        print("\n[3/7] An interrupted and resumed run equals an uninterrupted one...")
        ok = True
        model_u, opt_u, sampler_u, loader_u = _build_run(3)
        hist_u = []
        _train_epochs(model_u, opt_u, loader_u, 1, _LAST_EPOCH, hist_u)
        for interrupt_after in (3, 5):
            model, optimizer, sampler, loader = _build_run(3)
            hist = []
            _train_epochs(model, optimizer, loader, 1, interrupt_after, hist)
            _save(resume_path, identity, interrupt_after, model, optimizer, sampler, hist)
            del model, optimizer, sampler, loader                     # the "power cut": everything is gone
            state = load_resume_state(resume_path, identity)
            # Rebuild from scratch with a DIFFERENT seed, so only the checkpoint can make it match.
            model, optimizer, sampler, loader = _build_run(999, sampler_generator_state=state["sampler_generator_state"])
            model.load_state_dict(state["model_state_dict"])
            optimizer.load_state_dict(state["optimizer_state_dict"])
            restore_rng_state(state["rng_state"])
            hist = list(state["history"])
            _train_epochs(model, optimizer, loader, state["epoch"] + 1, _LAST_EPOCH, hist)
            label = f"interrupted after epoch {interrupt_after}"
            if not _deep_equal(model.state_dict(), model_u.state_dict()):
                failures.append(f"Interrupted run ({label}): final weights/BatchNorm statistics differ"); ok = False
            if not _deep_equal(optimizer.state_dict(), opt_u.state_dict()):
                failures.append(f"Interrupted run ({label}): final optimizer state differs"); ok = False
            if hist != hist_u:
                failures.append(f"Interrupted run ({label}): epoch history differs"); ok = False
        print("  OK -- bit-identical after a cut mid-Phase-A (epoch 3) and on the A -> B boundary (epoch 5)" if ok
              else "  FAILED -- see summary")

        # --- 4. Refusals ---
        print("\n[4/7] Wrong or damaged files are refused...")
        ok = True
        model, optimizer, sampler, loader = _build_run(4)
        hist = []
        _train_epochs(model, optimizer, loader, 1, 1, hist)
        _save(resume_path, identity, 1, model, optimizer, sampler, hist)
        for label, wrong in (("model kind", make_identity("ablation", 0)), ("fold", make_identity("multimodal", 1))):
            try:
                load_resume_state(resume_path, wrong)
                failures.append(f"Refusals: a checkpoint for the wrong {label} was accepted"); ok = False
            except CheckpointMismatchError:
                pass
        try:
            save_resume_state(resume_path, identity, 7, "A", model, optimizer, 0.5, 7, 0, hist)
            failures.append("Refusals: phase 'A' with epoch 7 was accepted on save"); ok = False
        except ValueError:
            pass
        raw = torch.load(resume_path, map_location="cpu", weights_only=False)
        raw["phase"] = "B"                                            # epoch 1 saved as phase B: inconsistent
        torch.save(raw, resume_path)
        try:
            load_resume_state(resume_path, identity)
            failures.append("Refusals: an inconsistent epoch/phase file was accepted on load"); ok = False
        except CheckpointError:
            pass
        raw["phase"] = "A"; raw["format_version"] = 999
        torch.save(raw, resume_path)
        try:
            load_resume_state(resume_path, identity)
            failures.append("Refusals: an unknown format version was accepted"); ok = False
        except CheckpointError:
            pass
        _save(resume_path, identity, 1, model, optimizer, sampler, hist)
        with open(resume_path, "rb") as f:
            data = f.read()
        with open(resume_path, "wb") as f:
            f.write(data[: len(data) // 2])                           # truncated file
        try:
            load_resume_state(resume_path, identity)
            failures.append("Refusals: a truncated file was silently accepted or ignored"); ok = False
        except CheckpointError:
            pass
        print("  OK -- wrong model kind, wrong fold, bad phase, bad format and truncated file all refused"
              if ok else "  FAILED -- see summary")

        # --- 5. Finished flag and absence ---
        print("\n[5/7] Finished flag and missing file...")
        ok = True
        _save(resume_path, identity, 2, model, optimizer, sampler, hist)
        if load_resume_state(resume_path, identity)["finished"] is not False:
            failures.append("Flags: 'finished' is not False by default"); ok = False
        _save(resume_path, identity, 2, model, optimizer, sampler, hist, finished=True)
        if load_resume_state(resume_path, identity)["finished"] is not True:
            failures.append("Flags: 'finished' did not survive save and load"); ok = False
        missing = os.path.join(tmp, "no_such_fold_resume.pt")
        if load_resume_state(missing, identity) is not None:
            failures.append("Flags: a missing resume file did not give None"); ok = False
        if load_best_weights(os.path.join(tmp, "no_such_fold_best.pt"), identity, model) is not None:
            failures.append("Flags: a missing best file did not give None"); ok = False
        print("  OK -- 'finished' defaults to False and survives; missing files give None" if ok else "  FAILED -- see summary")

        # --- 6. Best weights ---
        print("\n[6/7] Best-epoch weights file...")
        ok = True
        model, optimizer, sampler, loader = _build_run(6)
        hist = []
        _train_epochs(model, optimizer, loader, 1, 3, hist)
        save_best_weights(best_path, identity, model, 3, 0.8765)
        _seed_everything(555)
        fresh = _TinyNet()
        info = load_best_weights(best_path, identity, fresh)
        if info != {"epoch": 3, "val_auc": 0.8765}:
            failures.append(f"Best weights: epoch/AUC came back as {info}"); ok = False
        if not _deep_equal(fresh.state_dict(), model.state_dict()):
            failures.append("Best weights: the loaded weights differ from the saved ones"); ok = False
        raw = torch.load(best_path, map_location="cpu", weights_only=False)
        if "optimizer_state_dict" in raw or "rng_state" in raw:
            failures.append("Best weights: the best file holds training state (it should be weights only)"); ok = False
        try:
            load_best_weights(best_path, make_identity("ablation", 0), fresh)
            failures.append("Best weights: a foreign identity was accepted"); ok = False
        except CheckpointMismatchError:
            pass
        print("  OK -- weights, epoch and AUC round-trip; no training state inside; foreign identity refused" if ok
              else "  FAILED -- see summary")

        # --- 7. Busy-file retry ---
        print("\n[7/7] A briefly locked file is retried; a lasting problem is not hidden...")
        ok = True
        busy_dir = os.path.join(tmp, "busy")
        busy_resume, busy_best = checkpoint_paths(busy_dir, 0)
        model, optimizer, sampler, loader = _build_run(7)
        hist = []
        _train_epochs(model, optimizer, loader, 1, 2, hist)
        _save(busy_resume, identity, 2, model, optimizer, sampler, hist)       # an older, good file
        _train_epochs(model, optimizer, loader, 3, 3, hist)

        real_replace, real_sleep = os.replace, time.sleep
        waits, attempts = [], {"n": 0}

        def _lock(fail_times, exc=None):
            """os.replace stand-in: fails `fail_times` times (default: a Windows-style PermissionError), then works."""
            attempts["n"] = 0

            def _replace(src, dst, *args, **kwargs):
                attempts["n"] += 1
                if attempts["n"] <= fail_times:
                    raise (exc or PermissionError(13, "Access is denied (simulated lock)", str(dst)))
                return real_replace(src, dst, *args, **kwargs)
            return _replace

        # (a) a lock that clears after 2 failed swaps: the save must succeed on the 3rd attempt
        waits.clear()
        printed = io.StringIO()
        os.replace, time.sleep = _lock(2), waits.append
        try:
            with contextlib.redirect_stdout(printed):
                _save(busy_resume, identity, 3, model, optimizer, sampler, hist)
        except Exception as exc:
            failures.append(f"Busy file: a lock that clears after 2 tries was not survived -- {type(exc).__name__}: {exc}")
            ok = False
        finally:
            os.replace, time.sleep = real_replace, real_sleep
        if attempts["n"] != 3 or waits != list(REPLACE_RETRY_WAITS[:2]):
            failures.append(f"Busy file: expected 3 attempts with waits {list(REPLACE_RETRY_WAITS[:2])}, "
                            f"got {attempts['n']} attempts with waits {waits}"); ok = False
        if "file busy" not in printed.getvalue():
            failures.append("Busy file: the retry printed no 'file busy' notice"); ok = False
        try:
            saved = load_resume_state(busy_resume, identity)
            if saved is None or saved["epoch"] != 3:
                failures.append("Busy file: after the retried save the resume file is not the epoch-3 one"); ok = False
        except CheckpointError as exc:
            failures.append(f"Busy file: the retried resume file is unreadable -- {exc}"); ok = False

        # (a2) the best-weights file goes through the same retry
        waits.clear()
        os.replace, time.sleep = _lock(1), waits.append
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                save_best_weights(busy_best, identity, model, 3, 0.9)
        except Exception as exc:
            failures.append(f"Busy file: the best-weights save did not survive a 1-try lock -- {type(exc).__name__}: {exc}")
            ok = False
        finally:
            os.replace, time.sleep = real_replace, real_sleep
        if attempts["n"] != 2 or waits != list(REPLACE_RETRY_WAITS[:1]):
            failures.append(f"Busy file: best-weights save made {attempts['n']} attempts with waits {waits}, "
                            f"expected 2 attempts with waits {list(REPLACE_RETRY_WAITS[:1])}"); ok = False
        try:
            if load_best_weights(busy_best, identity, _TinyNet()) != {"epoch": 3, "val_auc": 0.9}:
                failures.append("Busy file: the retried best-weights file holds the wrong epoch/AUC"); ok = False
        except CheckpointError as exc:
            failures.append(f"Busy file: the retried best-weights file is unreadable -- {exc}"); ok = False

        # (b) a lock that never clears: raise PermissionError after ALL attempts, old file untouched
        with open(busy_resume, "rb") as f:
            bytes_before = f.read()
        waits.clear()
        os.replace, time.sleep = _lock(10 ** 9), waits.append
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                _save(busy_resume, identity, 4, model, optimizer, sampler, hist)
            failures.append("Busy file: a lock that never clears did not raise"); ok = False
        except PermissionError:
            pass
        except Exception as exc:
            failures.append(f"Busy file: a lasting lock raised {type(exc).__name__}, expected PermissionError"); ok = False
        finally:
            os.replace, time.sleep = real_replace, real_sleep
        if attempts["n"] != len(REPLACE_RETRY_WAITS) + 1 or waits != list(REPLACE_RETRY_WAITS):
            failures.append(f"Busy file: a lasting lock made {attempts['n']} attempts with waits {waits}, expected "
                            f"{len(REPLACE_RETRY_WAITS) + 1} attempts with waits {list(REPLACE_RETRY_WAITS)}"); ok = False
        with open(busy_resume, "rb") as f:
            if f.read() != bytes_before:
                failures.append("Busy file: the previous checkpoint changed although the save gave up"); ok = False
        if [n for n in os.listdir(busy_dir) if n.endswith(".tmp")]:
            failures.append("Busy file: a temporary file was left behind after the save gave up"); ok = False

        # (c) any other error is raised at once, not retried
        waits.clear()
        os.replace, time.sleep = _lock(10 ** 9, exc=OSError(28, "No space left on device (simulated)")), waits.append
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                _save(busy_resume, identity, 4, model, optimizer, sampler, hist)
            failures.append("Busy file: a disk-full style error did not raise"); ok = False
        except OSError as exc:
            if isinstance(exc, PermissionError):
                failures.append("Busy file: a disk-full style error was turned into a PermissionError"); ok = False
        finally:
            os.replace, time.sleep = real_replace, real_sleep
        if attempts["n"] != 1 or waits:
            failures.append(f"Busy file: a non-lock error was retried ({attempts['n']} attempts, waits {waits})"); ok = False
        if [n for n in os.listdir(busy_dir) if n.endswith(".tmp")]:
            failures.append("Busy file: a temporary file was left behind after a non-lock error"); ok = False
        print(f"  OK -- a lock that clears is waited out (resume and best files); a lasting lock raises after "
              f"{len(REPLACE_RETRY_WAITS) + 1} tries with the old file intact; other errors are not retried"
              if ok else "  FAILED -- see summary")

    print("\n" + "=" * 60)
    if failures:
        print(f"VERIFICATION FAILED -- {len(failures)} issue(s):")
        for msg in failures:
            print(f"  - {msg}")
        print("=" * 60)
        raise RuntimeError(f"Checkpoint verification failed with {len(failures)} issue(s).")
    print("VERIFICATION PASSED -- all checks OK. Module 06 is safe to build on.")
    print("=" * 60)


# --- Usage ---
# run_checkpoint_verification()

In [23]:
run_checkpoint_verification()

CHECKPOINT VERIFICATION -- Phase 3, Module 06
  CPU, tiny stand-in model, synthetic data, temporary folder

[1/7] Atomic write survives a crash mid-save...
  OK -- previous checkpoint intact and readable, no temporary file left

[2/7] Round trip of every field...
  OK -- model, optimizer, counters, history, sampler generator and RNGs all identical

[3/7] An interrupted and resumed run equals an uninterrupted one...
  OK -- bit-identical after a cut mid-Phase-A (epoch 3) and on the A -> B boundary (epoch 5)

[4/7] Wrong or damaged files are refused...
  OK -- wrong model kind, wrong fold, bad phase, bad format and truncated file all refused

[5/7] Finished flag and missing file...
  OK -- 'finished' defaults to False and survives; missing files give None

[6/7] Best-epoch weights file...
  OK -- weights, epoch and AUC round-trip; no training state inside; foreign identity refused

[7/7] A briefly locked file is retried; a lasting problem is not hidden...
  OK -- a lock that clears is wa

# Module 7

In [24]:
"""
07_00_phase2_reused_block.txt
Phase 3 -- Module 07: Training Loop -- functions reused from Phase 2, unchanged

Requires 01_00 (P1_ARTIFACTS_DIR) already run.

The function bodies below are copied VERBATIM from the locked Phase 2 - P3 notebook so the
Phase 3 results are computed exactly the way the Phase 2 baselines were:

  set_all_seeds               (Phase 2, 01_02)  -- same seeding, first line of every fold
                                                   (the definition that was active when Phase 2 ran)
  compute_specificity_macro   (Phase 2, 01_01)
  compute_fold_metrics        (Phase 2, 01_01)  -- AUC-ROC (macro OvR), F1, sensitivity,
                                                   specificity, accuracy; softmax probabilities in
  summarize_across_folds      (Phase 2, 01_01)  -- mean +/- SD (ddof=1) across folds
  build_balanced_sampler      (Phase 2, 01_02)  -- WeightedRandomSampler, inverse class frequency,
                                                   own torch.Generator (state is checkpointed)
  build_weighted_criterion    (Phase 2, 01_02)  -- CrossEntropyLoss with Phase 1's per-fold
                                                   effective-number weights, in CLASS_TO_IDX order

Only the constants right below are Phase 3's own. RANDOM_SEED, DEVICE and WEIGHTS_PATH take the
same values Phase 2 used (its ARTIFACTS_DIR is Phase 1's artifacts folder, i.e. P1_ARTIFACTS_DIR).
CLASSES and CLASS_TO_IDX come from 02_00 (identical to Phase 2's), so they are not redefined here.
"""

import random
from typing import Sequence, cast

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, f1_score, recall_score, roc_auc_score
from torch.utils.data import WeightedRandomSampler

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WEIGHTS_PATH = P1_ARTIFACTS_DIR / "final_class_weights_per_fold.csv"


# ---------------------------------------------------------------------------
# Copied verbatim from Phase 2 - P3 (01_01_metrics.txt)
# ---------------------------------------------------------------------------

def compute_specificity_macro(y_true, y_pred, n_classes: int = 7) -> float:
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    specificities = []
    total = cm.sum()
    for i in range(n_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        specificity_i = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(specificity_i)
    return float(np.mean(specificities))


def compute_fold_metrics(y_true, y_pred_probs, n_classes: int = 7) -> dict:
    """
    y_true: array of true integer labels, one per sample.
    y_pred_probs: array of shape (n_samples, n_classes), softmax probabilities.
    Returns: dict with auc_roc, f1, sensitivity, specificity, accuracy --
             all single floats, this fold's values.
    """
    y_pred_labels = np.argmax(y_pred_probs, axis=1)

    auc_roc = roc_auc_score(
        y_true, y_pred_probs, multi_class="ovr", average="macro",
        labels=list(range(n_classes)),
    )
    f1 = f1_score(y_true, y_pred_labels, average="macro", labels=list(range(n_classes)), zero_division=0)
    sensitivity = recall_score(y_true, y_pred_labels, average="macro", labels=list(range(n_classes)), zero_division=0)
    specificity = compute_specificity_macro(y_true, y_pred_labels, n_classes=n_classes)
    accuracy = float(np.mean(y_true == y_pred_labels))

    return {
        "auc_roc": float(auc_roc),
        "f1": float(f1),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "accuracy": accuracy,
    }


def summarize_across_folds(fold_metrics_list: list[dict]) -> dict:
    """fold_metrics_list: list of 5 dicts (one per fold), each from
    compute_fold_metrics(). Returns mean +/- SD for each metric."""
    summary = {}
    for key in ["auc_roc", "f1", "sensitivity", "specificity", "accuracy"]:
        values = [m[key] for m in fold_metrics_list]
        summary[key] = {"mean": float(np.mean(values)), "sd": float(np.std(values, ddof=1))}
    return summary


# ---------------------------------------------------------------------------
# Copied verbatim from Phase 2 - P3 (01_02_train_fold.txt)
# ---------------------------------------------------------------------------

def build_balanced_sampler(labels, seed=RANDOM_SEED, generator_state=None):
    """Reused from Phase 1's 10_04_balanced_sampler.txt, verbatim logic.
    Built ONCE per fold, reused across all epochs -- the sampler's
    internal generator state advances with each DataLoader iteration, so
    repeated epochs naturally get different draws without rebuilding.

    generator_state: this sampler's torch.Generator is a separate object
    from the global RNGs capture_rng_state()/restore_rng_state() (00_03)
    cover, so a power-cut resume must restore ITS state explicitly too --
    otherwise a resumed fold would silently restart this generator from
    `seed` instead of continuing the draw sequence from where training
    stopped. Pass state["sampler_generator_state"] from the resume
    checkpoint to continue exactly; leave None for a fresh fold."""
    labels_series = pd.Series(labels)
    class_counts = labels_series.value_counts()
    class_weight_map = 1.0 / class_counts
    sample_weights = labels_series.map(class_weight_map).values

    generator = torch.Generator()
    if generator_state is not None:
        generator.set_state(generator_state)
    else:
        generator.manual_seed(seed)

    return WeightedRandomSampler(
        # cast: PyTorch's stub declares weights: Sequence[float], but the real
        # implementation (and PyTorch's own docs) accept a Tensor directly --
        # a stub/runtime mismatch, not a type error.
        weights=cast(Sequence[float], torch.as_tensor(sample_weights, dtype=torch.double)),
        num_samples=len(labels_series),
        replacement=True,
        generator=generator,
    )


def build_weighted_criterion(fold_num: int, class_to_idx: dict) -> nn.CrossEntropyLoss:
    """Script 1 only. Loads Phase 1's saved effective-number weights
    (final_class_weights_per_fold.csv) for this fold, builds the weight
    tensor IN class_to_idx ORDER -- this exact ordering is what
    Mechanism A (00_06) checks against, so it must match."""
    saved_weights = pd.read_csv(WEIGHTS_PATH, index_col="dx")
    fold_col = str(fold_num)
    weight_tensor = torch.zeros(len(class_to_idx), dtype=torch.float32)
    for cls, idx in class_to_idx.items():
        weight_tensor[idx] = float(cast(float, saved_weights.loc[cls, fold_col]))
    return nn.CrossEntropyLoss(weight=weight_tensor.to(DEVICE))


# ---------------------------------------------------------------------------
# Copied verbatim from Phase 2 - P3 (01_02_train_fold.txt, bottom of the cell)
# ---------------------------------------------------------------------------

def set_all_seeds(seed: int = RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    import os
    os.environ["PYTHONHASHSEED"] = str(seed)


print("Phase 2 functions loaded (verbatim): set_all_seeds, compute_fold_metrics, summarize_across_folds,")
print("  build_balanced_sampler, build_weighted_criterion")
print(f"  device: {DEVICE}   |   class weights file: {WEIGHTS_PATH}")

Phase 2 functions loaded (verbatim): set_all_seeds, compute_fold_metrics, summarize_across_folds,
  build_balanced_sampler, build_weighted_criterion
  device: cuda   |   class weights file: T:\Thesis - Development\Resources\Artifacts\Phase 1 - P2\final_class_weights_per_fold.csv


In [25]:
"""
07_01_epoch_helpers.txt
Phase 3 -- Module 07: Training Loop -- settings and per-epoch helpers (reusable)

Requires 07_00 already run (DEVICE).

SETTINGS (all locked; the values here are the only place they live)
  BATCH_SIZE  32    -- as Phase 2 (P3 run)
  MAX_EPOCHS  100   -- as Phase 2
  PATIENCE    10    -- non-improving epochs before stopping; counted from epoch 6 only
                       (Phase A epochs 1-5 never count -- see train_fold in 07_02)
  bf16 autocast on GPU, no GradScaler (bf16 has fp32's exponent range), num_workers=0
  (num_workers=0 is what lets the checkpointed RNG state honestly cover the augmentation
   sequence -- worker processes keep RNG state the main process cannot see or restore).

  train_one_epoch   one pass over the training loader. Returns the mean training loss.
                    STOPS with a clear error on a NaN / infinite loss, BEFORE any optimizer
                    step is taken with it -- the last good checkpoint stays untouched, so
                    fixing the cause and re-running simply resumes.
  predict_split     model.eval() pass over a loader. Returns (softmax probabilities [N, 7],
                    true labels [N]) as NumPy arrays -- the exact inputs compute_fold_metrics
                    takes.

Both models are called model(image, metadata); the loader yields (image, metadata, label).
"""

import math

import numpy as np
import torch

BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 10
AMP_DTYPE = torch.bfloat16
USE_AMP = DEVICE.type == "cuda"


def train_one_epoch(model, loader, criterion, optimizer, device, use_amp=USE_AMP,
                    amp_dtype=AMP_DTYPE, epoch=None):
    total_loss, total_n = 0.0, 0
    for batch_idx, (images, metadata, labels) in enumerate(loader):
        images, metadata, labels = images.to(device), metadata.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            logits = model(images, metadata)
            loss = criterion(logits, labels)
        loss_value = float(loss.detach())
        if not math.isfinite(loss_value):
            raise FloatingPointError(
                f"Non-finite loss ({loss_value}) at epoch {epoch}, batch {batch_idx}. Training stopped "
                f"before the optimizer step. The last saved checkpoint is intact -- fix the cause, "
                f"then re-run to resume.")
        loss.backward()
        optimizer.step()
        total_loss += loss_value * labels.shape[0]
        total_n += labels.shape[0]
    return total_loss / total_n


def predict_split(model, loader, device, use_amp=USE_AMP, amp_dtype=AMP_DTYPE):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, metadata, labels in loader:
            images, metadata = images.to(device), metadata.to(device)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(images, metadata)
            all_probs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


print("Epoch helpers loaded.")
print(f"  batch {BATCH_SIZE}, max {MAX_EPOCHS} epochs, patience {PATIENCE} (counted from epoch {PHASE_A_EPOCHS + 1}), "
      f"bf16 autocast {'ON' if USE_AMP else 'OFF (no GPU)'}, num_workers=0")

Epoch helpers loaded.
  batch 32, max 100 epochs, patience 10 (counted from epoch 6), bf16 autocast ON, num_workers=0


In [26]:
"""
07_02_train_fold.txt
Phase 3 -- Module 07: Training Loop -- one fold, start to finish (reusable)

Requires 01_00, 03_02, 04_00, 04_01, 05_00, 06_00, 07_00, 07_01 already run.

  train_fold(fold_num, model_kind, out_dir, make_datasets, build_model_fn, build_criterion_fn, ...)

One function serves BOTH models ("multimodal" / "ablation"): the model is built by the caller's
build_model_fn, and everything else is identical, so the two are trained by the same code.

WHAT IT DOES, EACH EPOCH (1-based)
  1. set_phase(model, phase_for_epoch(epoch))            -- replaces model.train() (Module 04)
  2. train_one_epoch                                     -- balanced sampler, weighted loss, bf16
  3. predict_split on the validation fold + compute_fold_metrics (Phase 2's)
  4. best-epoch update: val AUC-ROC STRICTLY greater than the best so far. ALL epochs compete,
     Phase A epochs included (locked decision). Improvement saves fold_N_best.pt.
  5. patience counter: reset on improvement; +1 on a non-improving epoch ONLY IN PHASE B
     (epoch >= 6). Earliest possible early stop = epoch 15.
  6. finished = early stop fired OR max epochs reached; fold_N_resume.pt written (Module 06)

RESUME / FINISHED
  A resume file for this model kind and fold continues exactly where it stopped. If it says
  finished, training is skipped and the results are rebuilt from the saved best weights.
  A file for another model kind or fold is refused (CheckpointMismatchError).
  Order per epoch is: best weights, THEN resume file. A power cut between the two just repeats that
  one epoch on resume. The final results always come from the best file's own recorded epoch/AUC.
  Every save retries a briefly-locked destination file (Windows antivirus / indexer) -- see Module 06.

GUARDS
  - refuses a setup where the last training batch would hold exactly one image (BatchNorm1d in the
    MetaBlock cannot train on a single sample);
  - non-finite loss stops the run cleanly (see train_one_epoch).

OUTPUTS in out_dir
  fold_N_resume.pt, fold_N_best.pt                       (Module 06)
  fold_N_val_predictions.npz                             written once, at the end, from the BEST weights:
      image_ids, labels, probs [N, 7] float32, best_epoch, class_names
      (the image ids make every row traceable to the fold table)

RETURNS a dict: fold, model_kind, metrics (from the best weights), best_epoch, best_epoch_phase,
best_val_auc, epochs_run, stop_reason ("early_stop" / "max_epochs"), history, predictions_path.

The caller supplies (so this file has no dataset or model hard-wired, and can be tested on synthetic data):
  make_datasets(fold_num)  -> (train_ds, val_ds); train_ds needs .get_labels(), both need .image_ids
  build_model_fn()         -> a fresh, UNSEEDED model (train_fold seeds first, then builds)
  build_criterion_fn(fold) -> the loss module, already on the right device
  build_sampler_fn         -> defaults to Phase 2's build_balanced_sampler
"""

import os
import tempfile
import time
from pathlib import Path
from typing import cast

import numpy as np
import torch
from torch.utils.data import DataLoader


def _atomic_save_npz(final_path, **arrays):
    """Same idea as the checkpoint writes: temporary file, then an atomic rename -- retried if the
    destination is briefly locked (Module 06's _replace_with_retry)."""
    final_path = str(final_path)
    final_dir = os.path.dirname(final_path) or "."
    os.makedirs(final_dir, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=final_dir, suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            np.savez(f, **arrays)
            f.flush()
            os.fsync(f.fileno())
        _replace_with_retry(tmp_path, final_path)
    except BaseException:
        _remove_quietly(tmp_path)
        raise


def train_fold(fold_num, model_kind, out_dir, make_datasets, build_model_fn, build_criterion_fn,
               build_sampler_fn=None, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, patience=PATIENCE,
               device=DEVICE, use_amp=USE_AMP, log_fn=print):
    if build_sampler_fn is None:
        build_sampler_fn = build_balanced_sampler
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    identity = make_identity(model_kind, fold_num)
    resume_path, best_path = checkpoint_paths(out_dir, fold_num)
    predictions_path = out_dir / f"fold_{fold_num}_val_predictions.npz"
    tag = f"[{model_kind} | fold {fold_num}]"

    set_all_seeds(RANDOM_SEED)                                   # first line, per fold (as Phase 2)

    state = load_resume_state(resume_path, identity)             # BEFORE the sampler: it needs its generator state
    train_ds, val_ds = make_datasets(fold_num)
    if len(train_ds) % batch_size == 1:
        raise ValueError(f"{tag} the last training batch would hold exactly 1 image "
                         f"({len(train_ds)} images, batch {batch_size}); BatchNorm1d cannot train on it")

    sampler = build_sampler_fn(train_ds.get_labels(), seed=RANDOM_SEED,
                               generator_state=state["sampler_generator_state"] if state is not None else None)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    criterion = build_criterion_fn(fold_num)

    model = build_model_fn().to(device)
    optimizer = build_optimizer(model)

    best_metric, best_epoch, early_stop_counter, history, finished = -float("inf"), None, 0, [], False
    start_epoch = 1
    if state is not None:
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        for opt_state in optimizer.state.values():               # optimizer tensors back onto the device
            for k, v in opt_state.items():
                if torch.is_tensor(v):
                    opt_state[k] = v.to(device)
        restore_rng_state(state["rng_state"])
        best_metric, best_epoch = state["best_metric"], state["best_epoch"]
        early_stop_counter, history, finished = state["early_stop_counter"], list(state["history"]), state["finished"]
        start_epoch = state["epoch"] + 1
        if finished:
            log_fn(f"{tag} already finished in a previous run -- skipping training, rebuilding results from the best weights")
        else:
            log_fn(f"{tag} resuming from epoch {start_epoch}")
    else:
        log_fn(f"{tag} starting fresh")

    if not finished:
        for epoch in range(start_epoch, max_epochs + 1):
            started = time.time()
            phase = phase_for_epoch(epoch)
            set_phase(model, phase)                              # INSTEAD of model.train()
            train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device,
                                         use_amp=use_amp, epoch=epoch)

            probs, labels = predict_split(model, val_loader, device, use_amp=use_amp)
            val_auc = compute_fold_metrics(labels, probs, n_classes=len(CLASSES))["auc_roc"]

            improved = val_auc > best_metric                     # strictly greater; all epochs compete
            if improved:
                best_metric, best_epoch, early_stop_counter = val_auc, epoch, 0
                save_best_weights(best_path, identity, model, epoch, val_auc)
            elif phase == "B":
                early_stop_counter += 1                          # patience counts from epoch 6 only

            stopped_early = early_stop_counter >= patience
            finished = stopped_early or epoch >= max_epochs
            history.append({"epoch": epoch, "phase": phase, "train_loss": train_loss,
                            "val_auc_roc": float(val_auc), "best_val_auc_roc": float(best_metric),
                            "improved": bool(improved), "early_stop_counter": early_stop_counter,
                            "epoch_seconds": round(time.time() - started, 2)})

            save_resume_state(resume_path, identity, epoch, phase, model, optimizer,
                              best_metric, best_epoch, early_stop_counter, history,
                              sampler_generator_state=cast(torch.Generator, sampler.generator).get_state(),
                              finished=finished)

            log_fn(f"{tag} epoch {epoch:3d} [{phase}] loss {train_loss:.4f} | val AUC {val_auc:.4f} "
                   f"(best {best_metric:.4f} @ {best_epoch}) | patience {early_stop_counter}/{patience}")
            if finished:
                log_fn(f"{tag} {'early stopping' if stopped_early else 'max epochs reached'} at epoch {epoch}")
                break

    # --- Final results: from the BEST weights, never the last epoch's ---
    info = load_best_weights(best_path, identity, model)
    if info is None:
        raise CheckpointError(f"{tag} no best-weights file at {best_path}; cannot build the final results")
    if best_epoch is not None and info["epoch"] != best_epoch:
        log_fn(f"{tag} NOTE: best file holds epoch {info['epoch']} (resume file recorded {best_epoch}); "
               f"the best file is the truth -- likely a power cut between the two writes")
    probs, labels = predict_split(model, val_loader, device, use_amp=use_amp)
    final_metrics = compute_fold_metrics(labels, probs, n_classes=len(CLASSES))

    _atomic_save_npz(predictions_path,
                     image_ids=np.array(val_ds.image_ids, dtype=str), labels=labels.astype(np.int64),
                     probs=probs.astype(np.float32), best_epoch=np.int64(info["epoch"]),
                     class_names=np.array(CLASSES, dtype=str))

    epochs_run = history[-1]["epoch"] if history else 0
    stop_reason = "early_stop" if early_stop_counter >= patience else "max_epochs"
    log_fn(f"{tag} FINAL (best weights, epoch {info['epoch']}, phase {phase_for_epoch(info['epoch'])}): {final_metrics}")
    return {"fold": fold_num, "model_kind": model_kind, "metrics": final_metrics,
            "best_epoch": info["epoch"], "best_epoch_phase": phase_for_epoch(info["epoch"]),
            "best_val_auc": info["val_auc"], "epochs_run": epochs_run, "stop_reason": stop_reason,
            "history": history, "predictions_path": str(predictions_path)}


print("train_fold loaded.")
print("  one call = one fold of one model ('multimodal' or 'ablation'); resumes, or rebuilds results if already finished")

train_fold loaded.
  one call = one fold of one model ('multimodal' or 'ablation'); resumes, or rebuilds results if already finished


In [27]:
"""
07_03_train_fold_verification.txt
Phase 3 -- Module 07: Training Loop -- Verification (run once)

Requires 01_00, 02_00 (CLASSES), 04_00, 04_01, 05_00, 06_00, 07_00, 07_01, 07_02 already run.
Runs the REAL train_fold on CPU with a tiny stand-in model and synthetic data (24 training and
28 validation "images" of 3x8x8, batch 8): no dataset, no pretrained weights, no GPU. Files go
to a temporary folder that is deleted afterwards.

Where a check needs an exact validation-AUC trajectory (patience, best epoch), the metric function is
temporarily replaced by a scripted sequence; where it needs real numbers (resume equality, the
predictions file) the real compute_fold_metrics is used. Everything is restored afterwards.

All seven checks run as a full set (failures collected, one RuntimeError at the end).

Checks:
  1. Loop rules        -- scripted AUC curves prove: Phase A epochs never count toward patience
                          (a bad epochs-2-5 stretch does NOT stop the run), patience counts from
                          epoch 6 and resets on improvement (earliest stop = epoch 15 with patience
                          10), ties are not improvements, a Phase A epoch CAN be the best epoch,
                          max epochs ends the run with the finished flag set.
  2. Phase discipline  -- inside a real run: the backbone (weights AND BatchNorm statistics) is
                          bit-identical after every Phase A epoch, and changes at epoch 6; the head
                          group changes from epoch 1.
  3. Best weights      -- fold_N_best.pt holds exactly the weights of the best epoch, and the
                          predictions file (ids, labels, probabilities, best epoch, class names)
                          reproduces the reported metrics.
  4. Power cuts        -- runs killed after epoch 3, after epoch 5 (the A -> B boundary) and between
                          the best-weights write and the resume write of epoch 4, then re-run, end
                          identical to an uninterrupted run: metrics, history, best/resume files,
                          predictions.
  5. Finished folds    -- re-running a finished fold trains zero epochs, leaves the resume file
                          byte-identical and returns the same results (early-stopped and
                          max-epoch folds both).
  6. Guards            -- a NaN loss stops the run cleanly, the last checkpoint survives, and a
                          re-run resumes from it; a last batch of one image, a checkpoint belonging
                          to another model kind or fold (a fold-0 file wearing fold 1's name), and an
                          unknown model kind are all refused.
  7. Busy files        -- a full run in which EVERY save (resume, best weights, predictions) is refused
                          twice by a simulated Windows lock (no real waiting) still ends identical to
                          the uninterrupted run. A lock on the predictions file that never clears
                          crashes the run cleanly at the very end (fold already marked finished, no
                          stray files), and a re-run once the lock is gone completes with identical
                          results.
"""

import contextlib
import io
import os
import shutil
import tempfile
import time
import warnings

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


class _P7TinyNet(nn.Module):
    """Stand-in with the real models' structure: BatchNorm backbone, BatchNorm1d + Dropout head."""

    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(nn.Conv2d(3, 4, 3, padding=1), nn.BatchNorm2d(4), nn.ReLU(),
                                      nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Sequential(nn.Linear(4, 6), nn.BatchNorm1d(6), nn.ReLU(), nn.Dropout(0.5),
                                  nn.Linear(6, 7))

    def forward(self, image, metadata):
        return self.head(self.backbone(image))

    def backbone_parameters(self):
        return list(self.backbone.parameters())

    def head_parameters(self):
        return list(self.head.parameters())


class _P7SynthDataset:
    def __init__(self, n, seed, prefix):
        g = torch.Generator()
        g.manual_seed(seed)
        self.images = torch.randn(n, 3, 8, 8, generator=g)
        self.meta = torch.randn(n, METADATA_DIM, generator=g)
        self.labels = np.arange(n) % 7
        self.image_ids = [f"{prefix}_{i:03d}" for i in range(n)]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, i):
        return self.images[i], self.meta[i], int(self.labels[i])

    def get_labels(self):
        return self.labels


class _PowerCut(Exception):
    pass


class _ScriptExhausted(RuntimeError):
    """The scripted AUC curve ran out: the loop asked for more epochs than the scenario allows."""


class _NanCriterion(nn.Module):
    def __init__(self, nan_from_call):
        super().__init__()
        self.calls, self.nan_from_call = 0, nan_from_call

    def forward(self, logits, labels):
        self.calls += 1
        loss = nn.functional.cross_entropy(logits, labels)
        return loss * float("nan") if self.calls >= self.nan_from_call else loss


@contextlib.contextmanager
def _patch_global(name, make_replacement):
    g = globals()
    real = g[name]
    g[name] = make_replacement(real)
    try:
        yield
    finally:
        g[name] = real


@contextlib.contextmanager
def _scripted_auc(sequence):
    """compute_fold_metrics returns the next scripted AUC (one call per epoch, plus one for the final results)."""
    it = iter(sequence)

    def make(_real):
        def fake(y_true, y_pred_probs, n_classes=7):
            try:
                v = next(it)
            except StopIteration:
                raise _ScriptExhausted("scripted AUC curve exhausted")
            return {k: v for k in ("auc_roc", "f1", "sensitivity", "specificity", "accuracy")}
        return fake
    with _patch_global("compute_fold_metrics", make):
        yield


@contextlib.contextmanager
def _snapshots(store):
    """After every training epoch, store a copy of the model's full state."""
    def make(real):
        def wrapped(model, *a, **k):
            out = real(model, *a, **k)
            store.append({kk: v.detach().clone() for kk, v in model.state_dict().items()})
            return out
        return wrapped
    with _patch_global("train_one_epoch", make):
        yield


@contextlib.contextmanager
def _power_cut(when, at_epoch):
    def make(real):
        def wrapped(resume_path, identity, epoch, phase, *a, **k):
            if when == "before" and epoch == at_epoch:
                raise _PowerCut()
            real(resume_path, identity, epoch, phase, *a, **k)
            if when == "after" and epoch == at_epoch:
                raise _PowerCut()
        return wrapped
    with _patch_global("save_resume_state", make):
        yield


def _p7_deep_equal(a, b):
    if isinstance(a, torch.Tensor):
        return isinstance(b, torch.Tensor) and a.shape == b.shape and torch.equal(a.cpu(), b.cpu())
    if isinstance(a, dict):
        return isinstance(b, dict) and a.keys() == b.keys() and all(_p7_deep_equal(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)):
        return type(a) == type(b) and len(a) == len(b) and all(_p7_deep_equal(x, y) for x, y in zip(a, b))
    return a == b


def _load_npz(path):
    """Read every array, then close the file immediately. Windows refuses to overwrite or delete a
    file that is still open, so a lingering np.load() handle would break the next write to it."""
    with np.load(path, allow_pickle=False) as f:
        return {k: f[k] for k in f.files}


def _strip_time(history):
    return [{k: v for k, v in h.items() if k != "epoch_seconds"} for h in history]


def run_train_fold_verification():
    failures = []
    print("=" * 60)
    print("TRAIN-FOLD VERIFICATION -- Phase 3, Module 07")
    print("=" * 60)
    print("  real train_fold, CPU, tiny stand-in model, synthetic data, temporary folder")

    cpu = torch.device("cpu")
    initial_states = []

    def build_model():
        model = _P7TinyNet()
        initial_states.append({k: v.detach().clone() for k, v in model.state_dict().items()})
        return model

    def run(out_dir, model_kind="multimodal", fold_num=0, make_datasets=None, criterion_fn=None, **kwargs):
        return train_fold(
            fold_num, model_kind, out_dir,
            make_datasets=make_datasets or (lambda f: (_P7SynthDataset(24, 1, "tr"), _P7SynthDataset(28, 2, "va"))),
            build_model_fn=build_model,
            build_criterion_fn=criterion_fn or (lambda f: nn.CrossEntropyLoss()),
            batch_size=8, device=cpu, use_amp=False, log_fn=lambda *_: None, **kwargs)

    def run_scripted(sequence, out_dir, snaps=None, **kwargs):
        """run() under a scripted AUC curve; None if the loop needed more epochs than the curve has."""
        try:
            with _scripted_auc(sequence):
                if snaps is None:
                    return run(out_dir, **kwargs)
                with _snapshots(snaps):
                    return run(out_dir, **kwargs)
        except _ScriptExhausted:
            return None

    with tempfile.TemporaryDirectory() as tmp, warnings.catch_warnings():
        # Phase 2's build_balanced_sampler (copied verbatim) triggers a harmless pandas/NumPy
        # "array is not writable" warning; hide it here so the report stays readable.
        warnings.filterwarnings("ignore", message="The given NumPy array is not writable")

        def d(name):
            return os.path.join(tmp, name)

        # --- 1. Loop rules (scripted AUC curves) ---
        print("\n[1/7] Loop rules: best epoch, patience from epoch 6, resets, ties, max epochs...")
        ok = True
        def expect(label, res, want, counters=None, want_counters=None):
            """want = (epochs_run, best_epoch, best_epoch_phase, stop_reason)."""
            if res is None:
                failures.append(f"Loop rules {label}: the loop ran more epochs than the scenario allows"); return False
            got = (res["epochs_run"], res["best_epoch"], res["best_epoch_phase"], res["stop_reason"])
            good = got == want
            if not good:
                failures.append(f"Loop rules {label}: got (epochs_run, best, phase, stop) = {got}; expected {want}")
            if want_counters is not None:
                seen = [h["early_stop_counter"] for h in res["history"]]
                if seen != want_counters:
                    failures.append(f"Loop rules {label}: patience counters were {seen}, expected {want_counters}"); good = False
            return good

        # A: best at epoch 3 (Phase A); epochs 4-5 worse (must not count); then 10 worse Phase B epochs.
        seq_a = [0.60, 0.70, 0.80, 0.75, 0.74] + [0.70] * 10 + [0.5]
        snaps_a = []
        res_a = run_scripted(seq_a, d("A"), snaps=snaps_a)
        ok &= expect("A", res_a, (15, 3, "A", "early_stop"), want_counters=[0] * 5 + list(range(1, 11)))
        # B: a bad Phase A stretch must not stop a patience-3 run; a Phase B improvement resets the counter.
        seq_b = [0.80, 0.70, 0.70, 0.70, 0.70, 0.79, 0.81, 0.80, 0.80, 0.80, 0.5]
        snaps_b = []
        res_b = run_scripted(seq_b, d("B"), snaps=snaps_b, patience=3)
        ok &= expect("B", res_b, (10, 7, "B", "early_stop"), want_counters=[0] * 5 + [1, 0, 1, 2, 3])
        # C: steady improvement runs to max epochs and is marked finished.
        res_c = run_scripted([0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.58, 0.5], d("C"), max_epochs=8)
        if expect("C", res_c, (8, 8, "B", "max_epochs")):
            state_c = load_resume_state(checkpoint_paths(d("C"), 0)[0], make_identity("multimodal", 0))
            if not state_c["finished"]:
                failures.append("Loop rules C: the finished flag is not set after the last allowed epoch"); ok = False
        else:
            ok = False
        # D: a tie is not an improvement -- epoch 1 stays the best epoch.
        res_d = run_scripted([0.70] * 7 + [0.5], d("D"), patience=2)
        ok &= expect("D", res_d, (7, 1, "A", "early_stop"))
        ok = ok and not any(f.startswith("Loop rules") for f in failures)
        print("  OK -- Phase A never counts; patience from epoch 6 (earliest stop 15); resets on improvement; "
              "ties are not improvements; a Phase A epoch can win; max epochs finishes the fold" if ok else "  FAILED -- see summary")

        # --- 2. Phase discipline inside a real run (run B: 10 epochs) ---
        print("\n[2/7] Phase discipline inside the loop...")
        ok = True
        init = initial_states[1]                                      # run B's model, before any training
        back = lambda s: {k: v for k, v in s.items() if k.startswith("backbone.")}
        head = lambda s: {k: v for k, v in s.items() if k.startswith("head.")}
        if len(snaps_b) != 10:
            failures.append(f"Phase discipline: expected 10 epoch snapshots, got {len(snaps_b)}"); ok = False
        else:
            for e in range(5):
                if not _p7_deep_equal(back(snaps_b[e]), back(init)):
                    failures.append(f"Phase discipline: backbone changed during Phase A epoch {e + 1}"); ok = False
            if _p7_deep_equal(back(snaps_b[5]), back(snaps_b[4])):
                failures.append("Phase discipline: backbone did not change at epoch 6 (Phase B)"); ok = False
            if _p7_deep_equal(head(snaps_b[0]), head(init)):
                failures.append("Phase discipline: head did not change in epoch 1"); ok = False
        print("  OK -- backbone (weights + BatchNorm statistics) bit-identical through epoch 5, changes at epoch 6; "
              "head trains from epoch 1" if ok else "  FAILED -- see summary")

        # --- 3 & 4 share the real-metrics baseline run ---
        res_u = run(d("U"), max_epochs=7)                              # uninterrupted, real metrics
        ident = make_identity("multimodal", 0)
        resume_u, best_u = checkpoint_paths(d("U"), 0)

        # --- 3. Best weights and predictions file ---
        print("\n[3/7] Best-epoch weights and predictions file...")
        ok = True
        best_a = torch.load(checkpoint_paths(d("A"), 0)[1], map_location="cpu", weights_only=False)
        if len(snaps_a) < 3 or best_a["epoch"] != 3 or not _p7_deep_equal(best_a["model_state_dict"], snaps_a[2]):
            failures.append("Best weights: fold_0_best.pt (run A) is not the epoch-3 model"); ok = False
        best_b = torch.load(checkpoint_paths(d("B"), 0)[1], map_location="cpu", weights_only=False)
        if len(snaps_b) < 7 or best_b["epoch"] != 7 or not _p7_deep_equal(best_b["model_state_dict"], snaps_b[6]):
            failures.append("Best weights: fold_0_best.pt (run B) is not the epoch-7 model"); ok = False
        # The reported metrics must be those of the BEST weights, not the last epoch's. (Run U's best epoch
        # is not its last epoch, so the two would differ.)
        if res_u["best_epoch"] == res_u["epochs_run"]:
            failures.append("Best weights: test setup is not discriminating (best epoch == last epoch)"); ok = False
        fresh = _P7TinyNet()
        load_best_weights(best_u, ident, fresh)
        fresh_probs, fresh_labels = predict_split(
            fresh, DataLoader(_P7SynthDataset(28, 2, "va"), batch_size=8, shuffle=False), cpu, use_amp=False)
        if compute_fold_metrics(fresh_labels, fresh_probs, n_classes=len(CLASSES)) != res_u["metrics"]:
            failures.append("Best weights: the reported metrics are not those of the best-epoch weights"); ok = False
        preds = _load_npz(res_u["predictions_path"])
        val_ds = _P7SynthDataset(28, 2, "va")
        if sorted(preds) != ["best_epoch", "class_names", "image_ids", "labels", "probs"]:
            failures.append(f"Predictions: unexpected contents {sorted(preds)}"); ok = False
        else:
            if list(preds["image_ids"]) != val_ds.image_ids:
                failures.append("Predictions: image ids do not match the validation set"); ok = False
            if not np.array_equal(preds["labels"], val_ds.labels):
                failures.append("Predictions: labels do not match the validation set"); ok = False
            if preds["probs"].shape != (28, 7) or preds["probs"].dtype != np.float32 \
                    or not np.allclose(preds["probs"].sum(axis=1), 1.0, atol=1e-4):
                failures.append("Predictions: probabilities have the wrong shape/dtype or do not sum to 1"); ok = False
            if int(preds["best_epoch"]) != res_u["best_epoch"] or list(preds["class_names"]) != list(CLASSES):
                failures.append("Predictions: best epoch or class names wrong"); ok = False
            if compute_fold_metrics(preds["labels"], preds["probs"], n_classes=len(CLASSES)) != res_u["metrics"]:
                failures.append("Predictions: the saved probabilities do not reproduce the reported metrics"); ok = False
        print("  OK -- best files hold exactly the best epoch's weights; predictions file traceable and "
              "reproduces the metrics" if ok else "  FAILED -- see summary")

        # --- 4. Power cuts ---
        print("\n[4/7] Runs killed mid-training and re-run equal an uninterrupted run...")
        ok = True
        u_resume = torch.load(resume_u, map_location="cpu", weights_only=False)
        u_best = torch.load(best_u, map_location="cpu", weights_only=False)
        for when, at_epoch in (("after", 3), ("after", 5), ("before", 4)):
            name = f"cut_{when}_{at_epoch}"
            label = f"killed {when} the resume write of epoch {at_epoch}"
            try:
                with _power_cut(when, at_epoch):
                    run(d(name), max_epochs=7)
                failures.append(f"Power cuts ({label}): the simulated cut did not happen"); ok = False
                continue
            except _PowerCut:
                pass
            expect_saved = at_epoch if when == "after" else at_epoch - 1
            saved = load_resume_state(checkpoint_paths(d(name), 0)[0], ident)
            if saved is None or saved["epoch"] != expect_saved:
                failures.append(f"Power cuts ({label}): resume file holds epoch {None if saved is None else saved['epoch']}, "
                                f"expected {expect_saved}"); ok = False
            res = run(d(name), max_epochs=7)
            if res["metrics"] != res_u["metrics"] or res["best_epoch"] != res_u["best_epoch"] \
                    or res["epochs_run"] != res_u["epochs_run"] or res["stop_reason"] != res_u["stop_reason"]:
                failures.append(f"Power cuts ({label}): results differ from the uninterrupted run"); ok = False
            if _strip_time(res["history"]) != _strip_time(res_u["history"]):
                failures.append(f"Power cuts ({label}): epoch history differs"); ok = False
            r_resume = torch.load(checkpoint_paths(d(name), 0)[0], map_location="cpu", weights_only=False)
            r_best = torch.load(checkpoint_paths(d(name), 0)[1], map_location="cpu", weights_only=False)
            if not (_p7_deep_equal(r_resume["model_state_dict"], u_resume["model_state_dict"])
                    and _p7_deep_equal(r_resume["optimizer_state_dict"], u_resume["optimizer_state_dict"])):
                failures.append(f"Power cuts ({label}): final resume state differs"); ok = False
            if not (_p7_deep_equal(r_best["model_state_dict"], u_best["model_state_dict"])
                    and r_best["epoch"] == u_best["epoch"]):
                failures.append(f"Power cuts ({label}): best-weights file differs"); ok = False
            p_a, p_b = _load_npz(res["predictions_path"]), _load_npz(res_u["predictions_path"])
            if p_a.keys() != p_b.keys() or not all(np.array_equal(p_a[k], p_b[k]) for k in p_b):
                failures.append(f"Power cuts ({label}): predictions file differs"); ok = False
        print("  OK -- killed after epoch 3, after epoch 5 (A -> B boundary) and between the best-weights and "
              "resume writes: all end identical to the uninterrupted run" if ok else "  FAILED -- see summary")

        # --- 5. Finished folds ---
        print("\n[5/7] Re-running a finished fold retrains nothing...")
        ok = True
        for name, scripted, kwargs, want_reason in (("U", None, {"max_epochs": 7}, "max_epochs"),
                                                     ("A", [0.5], {}, "early_stop")):
            resume_file = checkpoint_paths(d(name), 0)[0]
            with open(resume_file, "rb") as f:
                before_bytes = f.read()
            calls = []

            def make(real, calls=calls):
                def wrapped(*a, **k):
                    calls.append(1)
                    return real(*a, **k)
                return wrapped
            with _patch_global("train_one_epoch", make):
                again = run_scripted(scripted, d(name), **kwargs) if scripted else run(d(name), **kwargs)
            if again is None:
                failures.append(f"Finished folds (run {name}): the re-run kept evaluating epochs"); ok = False
                continue
            with open(resume_file, "rb") as f:
                after_bytes = f.read()
            if calls:
                failures.append(f"Finished folds (run {name}): a finished fold trained {len(calls)} batch(es) again"); ok = False
            if after_bytes != before_bytes:
                failures.append(f"Finished folds (run {name}): the resume file changed on a re-run"); ok = False
            first = res_u if name == "U" else res_a
            if first is None:
                continue
            if again["stop_reason"] != want_reason or again["epochs_run"] != first["epochs_run"] \
                    or again["best_epoch"] != first["best_epoch"]:
                failures.append(f"Finished folds (run {name}): re-run reported stop={again['stop_reason']}, "
                                f"epochs_run={again['epochs_run']}, best={again['best_epoch']}"); ok = False
            if name == "U" and again["metrics"] != res_u["metrics"]:
                failures.append("Finished folds (run U): metrics changed on a re-run"); ok = False
        print("  OK -- zero epochs retrained, resume file byte-identical, same results (max-epoch and "
              "early-stopped folds)" if ok else "  FAILED -- see summary")

        # --- 6. Guards ---
        print("\n[6/7] Guards...")
        ok = True
        nan_dir = d("nan")
        try:
            run(nan_dir, criterion_fn=lambda f: _NanCriterion(nan_from_call=7), max_epochs=4)   # 3 batches/epoch -> NaN in epoch 3
            failures.append("Guards: a NaN loss did not stop the run"); ok = False
        except Exception as exc:
            if not isinstance(exc, FloatingPointError):
                failures.append(f"Guards: a NaN loss raised {type(exc).__name__}, not the clean FloatingPointError"); ok = False
            saved = load_resume_state(checkpoint_paths(nan_dir, 0)[0], ident)
            if saved is None or saved["epoch"] != 2:
                failures.append("Guards: after the NaN stop the last checkpoint is not the epoch-2 one"); ok = False
            elif isinstance(exc, FloatingPointError):
                try:
                    res = run(nan_dir, max_epochs=4)                   # fixed loss -> resumes from epoch 3
                except Exception as exc2:
                    failures.append(f"Guards: the re-run after a NaN stop crashed ({type(exc2).__name__}: {exc2})"); ok = False
                else:
                    if res["epochs_run"] != 4 or [h["epoch"] for h in res["history"]] != [1, 2, 3, 4]:
                        failures.append("Guards: the re-run after a NaN stop did not resume and complete"); ok = False
        try:
            run(d("odd"), make_datasets=lambda f: (_P7SynthDataset(25, 1, "tr"), _P7SynthDataset(28, 2, "va")))
            failures.append("Guards: a last batch of one image was accepted"); ok = False
        except ValueError as exc:
            if "last training batch" not in str(exc):
                failures.append(f"Guards: the one-image batch was not refused by our guard ({exc})"); ok = False
        shutil.copy(checkpoint_paths(d("U"), 0)[0], checkpoint_paths(d("U"), 1)[0])    # a fold-0 file wearing fold 1's name
        for label, kwargs in (("model kind", {"model_kind": "ablation"}), ("fold", {"fold_num": 1})):
            try:
                run(d("U"), max_epochs=7, **kwargs)
                failures.append(f"Guards: a checkpoint for a different {label} was accepted"); ok = False
            except CheckpointMismatchError:
                pass
        try:
            run(d("bad"), model_kind="banana")
            failures.append("Guards: an unknown model kind was accepted"); ok = False
        except ValueError:
            pass
        print("  OK -- NaN stops cleanly and resumes; single-image batch, foreign model kind/fold and unknown "
              "kind refused" if ok else "  FAILED -- see summary")

        # --- 7. Busy files ---
        print("\n[7/7] Locked output files: waited out, or a clean crash and a clean recovery...")
        ok = True
        real_replace, real_sleep = os.replace, time.sleep
        attempts_seen, refused = {}, []          # tmp path -> refusals so far; names of refused destinations

        def _blocker(fail_first, only=None):
            """os.replace stand-in: refuses the first `fail_first` attempts of every save (or only of saves
            whose destination name ends with `only`) with a Windows-style PermissionError."""
            def _replace(src, dst, *args, **kwargs):
                name = os.path.basename(str(dst))
                if only is None or name.endswith(only):
                    attempts_seen[src] = attempts_seen.get(src, 0) + 1
                    if attempts_seen[src] <= fail_first:
                        refused.append(name)
                        raise PermissionError(13, "Access is denied (simulated lock)", str(dst))
                return real_replace(src, dst, *args, **kwargs)
            return _replace

        def _differences(res, out_dir, label):
            """Everything that must equal the uninterrupted run U (same comparisons as check 4)."""
            found = []
            if res["metrics"] != res_u["metrics"] or res["best_epoch"] != res_u["best_epoch"] \
                    or res["epochs_run"] != res_u["epochs_run"] or res["stop_reason"] != res_u["stop_reason"]:
                found.append(f"Busy files ({label}): results differ from the uninterrupted run")
            if _strip_time(res["history"]) != _strip_time(res_u["history"]):
                found.append(f"Busy files ({label}): epoch history differs")
            b_resume, b_best = checkpoint_paths(out_dir, 0)
            r_resume = torch.load(b_resume, map_location="cpu", weights_only=False)
            r_best = torch.load(b_best, map_location="cpu", weights_only=False)
            if not (_p7_deep_equal(r_resume["model_state_dict"], u_resume["model_state_dict"])
                    and _p7_deep_equal(r_resume["optimizer_state_dict"], u_resume["optimizer_state_dict"])):
                found.append(f"Busy files ({label}): final resume state differs")
            if not (_p7_deep_equal(r_best["model_state_dict"], u_best["model_state_dict"])
                    and r_best["epoch"] == u_best["epoch"]):
                found.append(f"Busy files ({label}): best-weights file differs")
            p_a, p_b = _load_npz(res["predictions_path"]), _load_npz(res_u["predictions_path"])
            if p_a.keys() != p_b.keys() or not all(np.array_equal(p_a[k], p_b[k]) for k in p_b):
                found.append(f"Busy files ({label}): predictions file differs")
            if [n for n in os.listdir(out_dir) if n.endswith(".tmp")]:
                found.append(f"Busy files ({label}): stray temporary file(s) left in the output folder")
            return found

        # (a) every save refused twice, then allowed
        printed = io.StringIO()
        os.replace, time.sleep = _blocker(2), (lambda seconds: None)
        try:
            with contextlib.redirect_stdout(printed):
                res_busy = run(d("busy"), max_epochs=7)
        except Exception as exc:
            res_busy = None
            failures.append(f"Busy files: a run whose every save was refused twice failed -- {type(exc).__name__}: {exc}")
            ok = False
        finally:
            os.replace, time.sleep = real_replace, real_sleep
        want_names = {"fold_0_resume.pt", "fold_0_best.pt", "fold_0_val_predictions.npz"}
        if not want_names <= set(refused):
            failures.append(f"Busy files: the simulated lock did not hit all three output files (hit {sorted(set(refused))})")
            ok = False
        if "file busy" not in printed.getvalue():
            failures.append("Busy files: the retries printed no 'file busy' notice"); ok = False
        if res_busy is not None:
            for message in _differences(res_busy, d("busy"), "every save refused twice"):
                failures.append(message); ok = False

        # (b) a lock on the predictions file that never clears
        attempts_seen.clear(); refused.clear()
        lasting_dir = d("busy_lasting")
        os.replace, time.sleep = _blocker(10 ** 9, only="val_predictions.npz"), (lambda seconds: None)
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                run(lasting_dir, max_epochs=7)
            failures.append("Busy files: a lock that never clears on the predictions file did not stop the run"); ok = False
        except PermissionError:
            pass
        except Exception as exc:
            failures.append(f"Busy files: a lasting lock raised {type(exc).__name__}, expected PermissionError"); ok = False
        finally:
            os.replace, time.sleep = real_replace, real_sleep
        l_resume, l_best = checkpoint_paths(lasting_dir, 0)
        try:
            crashed = load_resume_state(l_resume, ident)
            if crashed is None or not crashed["finished"]:
                failures.append("Busy files: after the crash the fold is not marked finished on disk"); ok = False
        except CheckpointError as exc:
            failures.append(f"Busy files: the resume file is unreadable after the crash -- {exc}"); ok = False
        if os.path.exists(os.path.join(lasting_dir, "fold_0_val_predictions.npz")):
            failures.append("Busy files: a predictions file exists although its save never succeeded"); ok = False
        if [n for n in os.listdir(lasting_dir) if n.endswith(".tmp")]:
            failures.append("Busy files: stray temporary file(s) left after the crash"); ok = False
        # ... and once the lock is gone, a plain re-run finishes the job without retraining
        try:
            res_recovered = run(lasting_dir, max_epochs=7)
            for message in _differences(res_recovered, lasting_dir, "re-run after the lasting lock"):
                failures.append(message); ok = False
        except Exception as exc:
            failures.append(f"Busy files: the re-run after the lasting lock failed -- {type(exc).__name__}: {exc}"); ok = False
        print("  OK -- every save refused twice: identical to the uninterrupted run; a lasting lock crashes "
              "cleanly and the re-run recovers identically" if ok else "  FAILED -- see summary")

    print("\n" + "=" * 60)
    if failures:
        print(f"VERIFICATION FAILED -- {len(failures)} issue(s):")
        for msg in failures:
            print(f"  - {msg}")
        print("=" * 60)
        raise RuntimeError(f"Train-fold verification failed with {len(failures)} issue(s).")
    print("VERIFICATION PASSED -- all checks OK. Module 07 is safe to build on.")
    print("=" * 60)


# --- Usage ---
# run_train_fold_verification()

In [28]:
run_train_fold_verification()

TRAIN-FOLD VERIFICATION -- Phase 3, Module 07
  real train_fold, CPU, tiny stand-in model, synthetic data, temporary folder

[1/7] Loop rules: best epoch, patience from epoch 6, resets, ties, max epochs...
  OK -- Phase A never counts; patience from epoch 6 (earliest stop 15); resets on improvement; ties are not improvements; a Phase A epoch can win; max epochs finishes the fold

[2/7] Phase discipline inside the loop...
  OK -- backbone (weights + BatchNorm statistics) bit-identical through epoch 5, changes at epoch 6; head trains from epoch 1

[3/7] Best-epoch weights and predictions file...
  OK -- best files hold exactly the best epoch's weights; predictions file traceable and reproduces the metrics

[4/7] Runs killed mid-training and re-run equal an uninterrupted run...
  OK -- killed after epoch 3, after epoch 5 (A -> B boundary) and between the best-weights and resume writes: all end identical to the uninterrupted run

[5/7] Re-running a finished fold retrains nothing...
  OK --

# Module 8

In [29]:
"""
08_00_mechanism_checks.txt
Phase 3 -- Module 08: Build Audit -- the two class-balancing checks

Requires 01_00 (FOLD_CSV_PATH not needed here) and 02_00 (CLASSES) already run.

The Methodology (3.4) and the Runbook require the loss weights and the balanced sampler to be
confirmed active INDEPENDENTLY -- a bug can silently disable one while the other still looks fine.

  Mechanism A  verify_mechanism_a            COPIED VERBATIM from Phase 2 - P3 (00_06)
               checks the weight tensor actually wired into the live loss function against a
               by-hand effective-number recomputation from the fold's own class counts, in
               class-index order.

  Mechanism B  verify_mechanism_b_multimodal   ADAPTED for Phase 3 -- not a verbatim copy, on purpose:
               Phase 2's version reads each batch as (images, labels) and takes batch[1] as the
               labels. Phase 3 batches are (image, metadata, label), so batch[1] would be the
               METADATA and the check would tally metadata values as class labels. This version
               takes the label from the LAST element, insists it looks like class labels
               (int64 vector, values 0..6), and counts a class that never appears as a 0% row
               (Phase 2's tally silently skipped such a class).
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

BETA_FINAL = 0.999   # Phase 1's locked value (same constant Phase 2's check used)


# ---------------------------------------------------------------------------
# Copied verbatim from Phase 2 - P3 (00_06_mechanism_ab_verification.txt)
# ---------------------------------------------------------------------------

def compute_effective_number_weights(class_counts, beta):
    classes = list(class_counts.keys()) if isinstance(class_counts, dict) else class_counts.index.tolist()
    counts = np.array([class_counts[c] for c in classes], dtype=np.float64)
    effective_num = (1.0 - np.power(beta, counts)) / (1.0 - beta)
    raw_weights = 1.0 / effective_num
    normalized_weights = raw_weights / raw_weights.sum() * len(classes)
    return dict(zip(classes, normalized_weights))


def verify_mechanism_a(criterion: nn.CrossEntropyLoss, cv_df: pd.DataFrame, fold_num: int,
                        class_to_idx: dict, rtol: float = 1e-4):
    """
    criterion: the actual nn.CrossEntropyLoss instance built for this fold
               (must have been constructed with weight=<tensor>).
    cv_df: the loaded cv_fold_assignments.csv DataFrame.
    fold_num: which fold this criterion belongs to.
    class_to_idx: {class_name: logit_index} -- the SAME mapping used to
                  build the model's output ordering. Mechanism A checks
                  the weight tensor against expected values IN THIS ORDER,
                  since a class-order mismatch here is exactly the kind of
                  bug this mechanism exists to catch.
    """
    print(f"--- Mechanism A: loss-weight wiring check, fold {fold_num} ---")

    if criterion.weight is None:
        raise AssertionError(
            f"Fold {fold_num}: criterion.weight is None -- class-balanced loss is "
            f"NOT active. Script 1 requires weighted CrossEntropyLoss."
        )

    train_subset = cv_df[cv_df["cv_fold"] != fold_num]
    counts = train_subset["dx"].value_counts().reindex(CLASSES, fill_value=0)
    expected = compute_effective_number_weights(counts.to_dict(), BETA_FINAL)

    wired_weights = criterion.weight.detach().cpu().numpy()
    mismatches = []
    for cls, idx in class_to_idx.items():
        wired_val = float(wired_weights[idx])
        expected_val = float(expected[cls])
        if not np.isclose(wired_val, expected_val, rtol=rtol):
            mismatches.append(f"class {cls} (idx {idx}): wired={wired_val:.4f} vs expected={expected_val:.4f}")

    if mismatches:
        raise AssertionError(f"Fold {fold_num}: Mechanism A FAILED -- {mismatches}")
    print(f"  OK -- criterion.weight matches expected effective-number weights, "
          f"correct class order, all {len(CLASSES)} classes")


# ---------------------------------------------------------------------------
# Phase 3 version of Mechanism B (see the note at the top of this file)
# ---------------------------------------------------------------------------

def verify_mechanism_b_multimodal(train_loader: DataLoader, n_batches: int = 300, tolerance_pp: float = 3.0):
    """
    train_loader: the ACTUAL training DataLoader (balanced sampler attached), yielding
                  (image, metadata, label) batches.
    n_batches:    how many batches to tally.
    tolerance_pp: max allowed deviation, in percentage points, of ANY class from the balanced target.
    """
    n_classes = len(CLASSES)
    ideal_target_pct = 100.0 / n_classes
    print(f"--- Mechanism B: balanced-sampler wiring check ({n_batches} batches) ---")

    observed_labels = []
    for batch_num, batch in enumerate(train_loader):
        labels = batch[-1]                                   # (image, metadata, LABEL): label is last
        if (not torch.is_tensor(labels) or labels.dtype != torch.int64 or labels.dim() != 1
                or int(labels.min()) < 0 or int(labels.max()) >= n_classes):
            raise AssertionError("Mechanism B: the last batch element does not look like class labels "
                                 "(int64 vector, values 0..6) -- unexpected batch layout")
        observed_labels.extend(labels.tolist())
        if batch_num + 1 >= n_batches:
            break

    empirical_pct = pd.Series(observed_labels).value_counts(normalize=True).reindex(range(n_classes), fill_value=0.0) * 100
    print(f"  Ideal balanced target: {ideal_target_pct:.2f}% per class")
    print(empirical_pct.rename(index={i: c for i, c in enumerate(CLASSES)}).round(2).to_string())

    max_deviation = float((empirical_pct - ideal_target_pct).abs().max())
    if max_deviation > tolerance_pp:
        raise AssertionError(
            f"Mechanism B FAILED -- max deviation from ideal target is {max_deviation:.2f}pp, exceeds "
            f"tolerance of {tolerance_pp}pp. Balanced sampler may not be genuinely active.")
    print(f"  OK -- max deviation from ideal target: {max_deviation:.2f}pp (tolerance: {tolerance_pp}pp)")


print("Mechanism checks loaded: verify_mechanism_a (verbatim Phase 2), verify_mechanism_b_multimodal (Phase 3 batch layout)")

Mechanism checks loaded: verify_mechanism_a (verbatim Phase 2), verify_mechanism_b_multimodal (Phase 3 batch layout)


In [30]:
"""
08_01_build_audit.txt
Phase 3 -- Module 08: Build Audit (run once, on your real data and GPU)

Requires these cells already run: 01_00, 01_01, 02_00, 02_01, 03_00-03_02, 04_00, 05_00, 07_00, 07_01, 08_00
and an `encoder_record` in memory. After a kernel restart, get it back with:  encoder_record = load_encoder_stats()

WHY THIS EXISTS
  Modules 01-07 were each proven on stand-in data. This is the first time the pieces run TOGETHER on the real
  images, the real metadata, the real pretrained models and the real GPU -- before two models x 5 folds of
  real training are started. It trains for a handful of steps on throwaway models. It writes NO checkpoints
  and NO files; nothing it does is reused by the real run.

WHAT IT CHECKS (all run as a full set; failures are collected, one RuntimeError at the end)
  1. Real data          -- fold sizes add up, no lesion sits in both train and val, every image file exists,
                           the 20-value metadata is well-formed (one-hot blocks, age flag, age z-scored on the
                           training rows), one real batch has the right shapes, dtypes and value ranges.
  2. Mechanism A        -- the loss weights actually wired into the live loss match the by-hand recomputation.
  3. Mechanism B        -- over 300 real augmented batches the balanced sampler delivers ~14.3% per class.
  4. Metadata effect    -- (per model, real pretrained weights) the multimodal logits CHANGE when the metadata
                           changes; the ablation's logits do NOT.
  5. Real training steps-- (per model) Phase A steps then Phase B steps: losses finite, the backbone and its
                           BatchNorm statistics frozen in Phase A and moving in Phase B, the head training
                           throughout. Reports seconds per batch (data loading vs compute), peak GPU memory,
                           and a rough time estimate for the full run.

Informational only (never fails): the software versions, compared with the fingerprint recorded for Phase 2.
"""

import gc
import math
import platform
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Fingerprint of the environment Phase 2 was locked in. A difference is REPORTED, not failed.
LOCKED_ENV = {"python": "3.11.15", "torch": "2.11.0+cu128", "cuda": "12.8"}


def _sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def _metadata_problems(name, meta):
    """Structural checks on a whole split's (n, 20) metadata tensor."""
    if meta.dim() != 2 or meta.shape[1] != METADATA_DIM:
        return [f"{name}: metadata shape {tuple(meta.shape)}, expected (n, {METADATA_DIM})"]
    problems = []
    if meta.dtype != torch.float32:
        problems.append(f"{name}: metadata dtype {meta.dtype}, expected float32")
    if not bool(torch.isfinite(meta).all()):
        return problems + [f"{name}: metadata contains NaN or infinite values"]
    for block, lo, hi in (("sex", SEX_OFFSET, SITE_OFFSET), ("site", SITE_OFFSET, METADATA_DIM)):
        part = meta[:, lo:hi]
        if not bool(((part == 0) | (part == 1)).all()):
            problems.append(f"{name}: the {block} block holds values other than 0 and 1")
        elif not bool((part.sum(dim=1) == 1).all()):
            problems.append(f"{name}: a row's {block} block is not exactly one-hot")
    recorded = meta[:, 1]
    if not bool(((recorded == 0) | (recorded == 1)).all()):
        problems.append(f"{name}: age_recorded holds values other than 0 and 1")
    else:
        missing = recorded == 0
        if bool(missing.any()) and not bool((meta[missing, 0] == 0).all()):
            problems.append(f"{name}: age_z is not 0 where the age is missing")
    return problems


def _audit_data(encoder_record, fold_num):
    """Check 1. Returns (context dict, list of problems). The context is None if it could not be built."""
    problems = []
    fold_table = load_fold_table_with_metadata()
    train_ds = HAM10000MultimodalFoldDataset(fold_num, "train", encoder_record, fold_table=fold_table)
    val_ds = HAM10000MultimodalFoldDataset(fold_num, "val", encoder_record, fold_table=fold_table)
    n_tr, n_va = len(train_ds), len(val_ds)
    print(f"  fold {fold_num}: {n_tr} training images, {n_va} validation images "
          f"(pool should total {EXPECTED_POOL_ROWS})")

    if n_tr + n_va != EXPECTED_POOL_ROWS:
        problems.append(f"Real data: train + val = {n_tr + n_va}, expected {EXPECTED_POOL_ROWS}")
    if n_tr % BATCH_SIZE == 1:
        problems.append(f"Real data: {n_tr} training images leave a final batch of ONE image "
                        f"(BatchNorm1d cannot train on it; train_fold refuses this)")
    if not bool((train_ds.df["cv_fold"] != fold_num).all()) or not bool((val_ds.df["cv_fold"] == fold_num).all()):
        problems.append("Real data: the train/val split does not follow the cv_fold column")
    if set(train_ds.image_ids) & set(val_ds.image_ids):
        problems.append("Real data: some image is in BOTH the training and the validation set")
    shared_lesions = set(train_ds.df["lesion_id"]) & set(val_ds.df["lesion_id"])
    if shared_lesions:
        problems.append(f"Real data: {len(shared_lesions)} lesion(s) have images in BOTH train and val (leakage)")

    for name, ds in (("train", train_ds), ("val", val_ds)):
        present = sorted(int(x) for x in np.unique(ds.labels))
        if present != list(range(len(CLASSES))):
            problems.append(f"Real data: {name} labels cover {present}, expected 0..{len(CLASSES) - 1}")
    counts_tr = {c: int((train_ds.labels == i).sum()) for c, i in CLASS_TO_IDX.items()}
    counts_va = {c: int((val_ds.labels == i).sum()) for c, i in CLASS_TO_IDX.items()}
    print(f"  train class counts: {counts_tr}")
    print(f"  val   class counts: {counts_va}")

    missing_files = [i for i in train_ds.image_ids + val_ds.image_ids
                     if not (FINAL_IMAGES_DIR / f"{i}.jpg").is_file()]
    if missing_files:
        problems.append(f"Real data: {len(missing_files)} image file(s) missing, e.g. {missing_files[:3]}")
    else:
        print(f"  all {n_tr + n_va} image files exist")

    problems += _metadata_problems("train", train_ds.metadata)
    problems += _metadata_problems("val", val_ds.metadata)

    tr_meta = train_ds.metadata
    rec_mask = tr_meta[:, 1] == 1
    n_missing_age = int((~rec_mask).sum())
    if int(rec_mask.sum()) > 0:
        z = tr_meta[rec_mask, 0].numpy().astype(np.float64)
        print(f"  train age_z over recorded ages: mean {z.mean():+.5f}, std {z.std():.5f} "
              f"({n_missing_age} training images have no recorded age)")
        if abs(z.mean()) > 1e-3 or abs(z.std() - 1.0) > 1e-3:
            problems.append(f"Real data: train age_z is not standardised (mean {z.mean():+.5f}, std {z.std():.5f})")
    else:
        problems.append("Real data: no training image has a recorded age")

    with warnings.catch_warnings():
        # Phase 2's build_balanced_sampler (copied verbatim) triggers a harmless pandas/NumPy
        # "array is not writable" warning; hidden so the report stays readable.
        warnings.filterwarnings("ignore", message="The given NumPy array is not writable")
        sampler = build_balanced_sampler(train_ds.get_labels(), seed=RANDOM_SEED)
    # Built exactly as train_fold builds them (07_02).
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    images, metadata, labels = next(iter(train_loader))
    print(f"  one real training batch: image {tuple(images.shape)} {images.dtype}, "
          f"metadata {tuple(metadata.shape)} {metadata.dtype}, labels {tuple(labels.shape)} {labels.dtype}")
    if images.shape != (BATCH_SIZE, 3, 300, 300) or images.dtype != torch.float32:
        problems.append(f"Real data: image batch is {tuple(images.shape)} {images.dtype}, "
                        f"expected ({BATCH_SIZE}, 3, 300, 300) float32")
    elif not bool(torch.isfinite(images).all()):
        problems.append("Real data: the image batch contains NaN or infinite values")
    else:
        lo, hi, sd = float(images.min()), float(images.max()), float(images.std())
        print(f"  image value range after normalisation: min {lo:.2f}, max {hi:.2f}, std {sd:.2f}")
        if lo < -4.0 or hi > 4.0 or sd < 0.1:
            problems.append(f"Real data: image values look wrong (min {lo:.2f}, max {hi:.2f}, std {sd:.2f}); "
                            f"expected roughly -2.2..2.7 after ImageNet normalisation")
    if metadata.shape != (BATCH_SIZE, METADATA_DIM) or metadata.dtype != torch.float32:
        problems.append(f"Real data: metadata batch is {tuple(metadata.shape)} {metadata.dtype}")
    if (labels.dtype != torch.int64 or labels.dim() != 1 or labels.shape[0] != BATCH_SIZE
            or int(labels.min()) < 0 or int(labels.max()) >= len(CLASSES)):
        problems.append("Real data: the label batch is not an int64 vector of class ids 0..6")

    ctx = {"fold_num": fold_num, "train_ds": train_ds, "val_ds": val_ds,
           "train_loader": train_loader, "val_loader": val_loader, "batch": (images, metadata, labels)}
    return ctx, problems


def _audit_one_model(kind, build_fn, ctx, pretrained, n_timing_steps, failures, notes):
    """Checks 4 and 5 for one model. Returns its timing numbers (or None if it could not be measured)."""
    train_loader, val_loader, fold_num = ctx["train_loader"], ctx["val_loader"], ctx["fold_num"]
    images0, metadata0, _ = ctx["batch"]
    print(f"\n--- {kind} model (pretrained={pretrained}) ---")
    model = optimizer = criterion = None
    try:
        model = build_fn(pretrained=pretrained).to(DEVICE)
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        # ---- Check 4: does the metadata reach the logits? (eval mode, no grad, full precision) ----
        model.eval()
        with torch.no_grad():
            imgs, meta = images0.to(DEVICE), metadata0.to(DEVICE)
            base = model(imgs, meta).float()
            rolled = model(imgs, meta.roll(1, dims=0)).float()
            zeroed = model(imgs, torch.zeros_like(meta)).float()
        if base.shape != (BATCH_SIZE, len(CLASSES)) or not bool(torch.isfinite(base).all()):
            failures.append(f"{kind}: logits are {tuple(base.shape)} or non-finite on a real batch")
        d_roll = float((base - rolled).abs().max())
        d_zero = float((base - zeroed).abs().max())
        print(f"  [4] metadata effect: max |logit change| with shuffled metadata {d_roll:.3e}, "
              f"with zeroed metadata {d_zero:.3e}")
        if kind == "multimodal":
            if not (d_roll > 1e-6 and d_zero > 1e-6):
                failures.append("multimodal: the logits do NOT change when the metadata changes "
                                "(the metadata is not reaching the output)")
        else:
            if max(d_roll, d_zero) > 1e-6:
                failures.append("ablation: the logits CHANGE when the metadata changes "
                                "(the ablation must ignore the metadata)")

        # ---- Check 5: real training steps, both phases ----
        criterion = build_weighted_criterion(fold_num, CLASS_TO_IDX)
        optimizer = build_optimizer(model)
        backbone = getattr(model, "backbone")
        bb_param = next(backbone.parameters())
        bn_layer = next(m for m in backbone.modules() if isinstance(m, nn.BatchNorm2d))
        head_param = getattr(model, "head_parameters")()[0]
        train_iter = iter(train_loader)

        def one_step():
            t0 = time.perf_counter()
            images, metadata, labels = next(train_iter)
            t_data = time.perf_counter() - t0
            t1 = time.perf_counter()
            images, metadata, labels = images.to(DEVICE), metadata.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(images, metadata)
                loss = criterion(logits, labels)
            loss_value = float(loss.detach())
            loss.backward()
            optimizer.step()
            _sync()
            return t_data, time.perf_counter() - t1, loss_value

        result = {}
        for phase in ("A", "B"):
            set_phase(model, phase)
            before = (bb_param.detach().clone(), bn_layer.running_mean.detach().clone(),
                      head_param.detach().clone())
            steps = [one_step() for _ in range(n_timing_steps + 1)]      # the first step is a warm-up
            after = (bb_param.detach().clone(), bn_layer.running_mean.detach().clone(),
                     head_param.detach().clone())

            losses = [s[2] for s in steps]
            if not all(math.isfinite(v) for v in losses):
                failures.append(f"{kind}: non-finite loss in Phase {phase} on real data: {losses}")
            bb_moved = not torch.equal(before[0], after[0])
            bn_moved = not torch.equal(before[1], after[1])
            head_moved = not torch.equal(before[2], after[2])
            want_bb = phase == "B"
            if bb_moved != want_bb:
                failures.append(f"{kind}: backbone weights {'changed' if bb_moved else 'did not change'} "
                                f"in Phase {phase} (expected {'to change' if want_bb else 'frozen'})")
            if bn_moved != want_bb:
                failures.append(f"{kind}: backbone BatchNorm statistics {'changed' if bn_moved else 'did not change'} "
                                f"in Phase {phase} (expected {'to change' if want_bb else 'frozen'})")
            if not head_moved:
                failures.append(f"{kind}: the head did not train in Phase {phase}")

            timed = steps[1:]
            data_s = sum(t[0] for t in timed) / len(timed)
            comp_s = sum(t[1] for t in timed) / len(timed)
            result[phase] = (data_s, comp_s)
            print(f"  [5] Phase {phase}: {data_s:.2f} s data loading + {comp_s:.2f} s compute = "
                  f"{data_s + comp_s:.2f} s/batch | losses {[round(v, 3) for v in losses]} | "
                  f"backbone {'moves' if bb_moved else 'frozen'}, BatchNorm stats "
                  f"{'move' if bn_moved else 'frozen'}, head {'moves' if head_moved else 'FROZEN'}")

        if DEVICE.type == "cuda":
            peak_gb = torch.cuda.max_memory_allocated() / 1e9
            total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
            result["peak_gb"] = peak_gb
            print(f"  [5] peak GPU memory during training: {peak_gb:.2f} GB of {total_gb:.2f} GB")
            if peak_gb > 0.9 * total_gb:
                notes.append(f"{kind}: peak GPU memory {peak_gb:.2f} GB is over 90% of the {total_gb:.2f} GB "
                             f"available -- close to running out; a long run could hit an out-of-memory error")

        # validation-style pass (eval mode, no grad, bf16 autocast) -- the same path predict_split uses
        model.eval()
        val_iter = iter(val_loader)
        eval_times = []
        with torch.no_grad():
            for _ in range(n_timing_steps + 1):
                t0 = time.perf_counter()
                images, metadata, _ = next(val_iter)
                images, metadata = images.to(DEVICE), metadata.to(DEVICE)
                with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                    logits = model(images, metadata)
                probs = torch.softmax(logits.float(), dim=1).cpu()
                eval_times.append(time.perf_counter() - t0)
        if not bool(torch.isfinite(probs).all()) or not bool(torch.allclose(probs.sum(dim=1), torch.ones(probs.shape[0]),
                                                                            atol=1e-4)):
            failures.append(f"{kind}: validation probabilities are non-finite or do not sum to 1")
        result["eval"] = sum(eval_times[1:]) / len(eval_times[1:])
        print(f"  [5] validation pass: {result['eval']:.2f} s/batch (including data loading)")
        return result
    except Exception as exc:                                    # noqa: BLE001 -- reported, not hidden
        failures.append(f"{kind}: stopped with {type(exc).__name__}: {exc}")
        print(f"  FAILED -- {type(exc).__name__}: {exc}")
        return None
    finally:
        model = optimizer = criterion = None


def _print_estimates(results, ctx):
    n_train_b = math.ceil(len(ctx["train_ds"]) / BATCH_SIZE)
    n_val_b = math.ceil(len(ctx["val_ds"]) / BATCH_SIZE)
    print(f"\n--- Rough time estimate ({n_train_b} training + {n_val_b} validation batches per epoch) ---")
    print("  Measured on only a few steps, so treat it as a ballpark. The real epoch count is decided by early "
          "stopping (earliest stop: epoch 15; ceiling: 100).")
    grand = {15: 0.0, 30: 0.0, 50: 0.0}
    for kind, r in results.items():
        epoch_a = n_train_b * sum(r["A"]) + n_val_b * r["eval"]
        epoch_b = n_train_b * sum(r["B"]) + n_val_b * r["eval"]
        print(f"  {kind}: one epoch ~{epoch_a / 60:.1f} min in Phase A, ~{epoch_b / 60:.1f} min in Phase B")
        parts = []
        for n_ep in grand:
            fold_s = PHASE_A_EPOCHS * epoch_a + (n_ep - PHASE_A_EPOCHS) * epoch_b
            grand[n_ep] += N_FOLDS * fold_s
            parts.append(f"{n_ep} epochs ~{fold_s / 3600:.1f} h")
        print(f"     one fold: " + ", ".join(parts))
    if len(results) == 2:
        print("  Whole run, both models x 5 folds, if every fold ran that many epochs: "
              + ", ".join(f"{n_ep} -> ~{s / 3600:.0f} h" for n_ep, s in grand.items()))


def run_build_audit(encoder_record, fold_num=0, pretrained=True, n_mech_b_batches=300, n_timing_steps=3):
    """
    encoder_record:   the dict from load_encoder_stats() (Module 01).
    fold_num:         which fold to audit (0 is enough; every fold is built the same way).
    pretrained:       True = the real ImageNet weights, as in the real run (first use may download them).
    n_mech_b_batches: batches tallied by the sampler check (300 = the Phase 2 setting; the slowest step here).
    n_timing_steps:   timed steps per phase and per model (one extra warm-up step is run and not counted).
    """
    if n_timing_steps < 1:
        raise ValueError("n_timing_steps must be at least 1")
    failures, notes = [], []
    set_all_seeds(RANDOM_SEED)

    print("=" * 60)
    print(f"BUILD AUDIT -- fold {fold_num}, pretrained={pretrained}")
    print("=" * 60)

    print("\nEnvironment (informational)")
    found = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda}
    for key, want in LOCKED_ENV.items():
        same = found[key] == want
        print(f"  {key}: {found[key]}  (locked for Phase 2: {want})  {'OK' if same else 'DIFFERS'}")
        if not same:
            notes.append(f"{key} version {found[key]} differs from the Phase 2 fingerprint {want}")
    if DEVICE.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}, bf16 autocast ON")
    else:
        print("  NO GPU FOUND -- the audit will run on CPU, slowly, and its timings will not represent the real run")
        notes.append("no GPU was available, so the timings and memory numbers are not meaningful")

    # ---- Check 1: real data ----
    print("\n[1] Real data structure")
    ctx = None
    try:
        ctx, problems = _audit_data(encoder_record, fold_num)
        failures += problems
        print("  OK" if not problems else "  FAILED -- see summary")
    except Exception as exc:                                    # noqa: BLE001
        failures.append(f"Real data: could not be built -- {type(exc).__name__}: {exc}")
        print(f"  FAILED -- {type(exc).__name__}: {exc}")

    results = {}
    if ctx is None:
        print("\nChecks 2-5 skipped: they need the real datasets from check 1.")
        failures.append("Checks 2-5 were skipped because the real datasets could not be built")
    else:
        # ---- Check 2: Mechanism A ----
        print("\n[2] Mechanism A")
        try:
            cv_df = pd.read_csv(FOLD_CSV_PATH)
            criterion = build_weighted_criterion(fold_num, CLASS_TO_IDX)
            verify_mechanism_a(criterion, cv_df, fold_num, CLASS_TO_IDX)
        except Exception as exc:                                # noqa: BLE001
            failures.append(f"Mechanism A: {type(exc).__name__}: {exc}")
            print(f"  FAILED -- {exc}")

        # ---- Check 3: Mechanism B ----
        print(f"\n[3] Mechanism B (tallies {n_mech_b_batches} real augmented batches -- a minute or two)")
        try:
            verify_mechanism_b_multimodal(ctx["train_loader"], n_batches=n_mech_b_batches)
        except Exception as exc:                                # noqa: BLE001
            failures.append(f"Mechanism B: {type(exc).__name__}: {exc}")
            print(f"  FAILED -- {exc}")

        # ---- Checks 4 and 5, one model at a time ----
        print("\n[4]+[5] Metadata effect and real training steps, per model")
        for kind, build_fn in (("multimodal", build_multimodal_model), ("ablation", build_image_only_model)):
            r = _audit_one_model(kind, build_fn, ctx, pretrained, n_timing_steps, failures, notes)
            if r is not None and all(k in r for k in ("A", "B", "eval")):
                results[kind] = r
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        if results:
            _print_estimates(results, ctx)

    print("\n" + "=" * 60)
    for note in notes:
        print(f"NOTE: {note}")
    if failures:
        print(f"BUILD AUDIT FAILED -- {len(failures)} issue(s):")
        for msg in failures:
            print(f"  - {msg}")
        print("=" * 60)
        raise RuntimeError(f"Build audit failed with {len(failures)} issue(s).")
    print("BUILD AUDIT PASSED -- the pieces work together on the real data. Ready for Module 09.")
    print("=" * 60)


# --- Usage ---
# encoder_record = load_encoder_stats()      # only needed after a kernel restart
# run_build_audit(encoder_record)

In [31]:
encoder_record = load_encoder_stats()

In [32]:
run_build_audit(encoder_record)

BUILD AUDIT -- fold 0, pretrained=True

Environment (informational)
  python: 3.11.15  (locked for Phase 2: 3.11.15)  OK
  torch: 2.11.0+cu128  (locked for Phase 2: 2.11.0+cu128)  OK
  cuda: 12.8  (locked for Phase 2: 12.8)  OK
  GPU: NVIDIA GeForce RTX 5060 Ti, bf16 autocast ON

[1] Real data structure
  fold 0: 6866 training images, 1717 validation images (pool should total 8583)
  train class counts: {'akiec': 227, 'bcc': 349, 'bkl': 751, 'df': 79, 'mel': 763, 'nv': 4597, 'vasc': 100}
  val   class counts: {'akiec': 54, 'bcc': 89, 'bkl': 190, 'df': 21, 'mel': 191, 'nv': 1149, 'vasc': 23}
  all 8583 image files exist
  train age_z over recorded ages: mean -0.00000, std 1.00000 (41 training images have no recorded age)
  one real training batch: image (32, 3, 300, 300) torch.float32, metadata (32, 20) torch.float32, labels (32,) torch.int64
  image value range after normalisation: min -2.04, max 2.64, std 0.76
  OK

[2] Mechanism A
--- Mechanism A: loss-weight wiring check, fold 0 ---

# Module 9

In [33]:
"""
09_00_run_config.txt
Phase 3 -- Module 09: Training Drivers -- run folder and run_config.json (reusable)

Requires these cells already run: 01_00, 02_00, 03_00, 04_00, 05_00, 06_00, 07_00, 07_01, 08_00.

THE RUN FOLDER  (TRAINING_DIR -- placeholder name, rename freely; everything below follows the variable)
  <TRAINING_DIR>/run_config.json        this file: how the run is configured, and its results so far
  <TRAINING_DIR>/multimodal/            fold_N_resume.pt, fold_N_best.pt, fold_N_val_predictions.npz
  <TRAINING_DIR>/ablation/              the same files for the matched image-only model
  The two models get separate folders because their files carry the same names (fold_0_resume.pt ...).

run_config.json HAS FOUR PARTS
  settings       every choice that defines HOW the models are trained (batch size, learning rates,
                 patience, phase split, loss, sampler, model description, augmentation ...). Read from
                 the live constants, so it cannot drift from what actually runs. GUARDED: a re-run whose
                 settings differ from the file is REFUSED (RunConfigMismatchError) -- the point being
                 that half a run can never silently continue under different rules. To change a setting
                 deliberately, start a new TRAINING_DIR.
  paths          where the fold table, images and class weights came from (recorded, not guarded, so
                 moving a drive does not block a resume).
  environment    Python / PyTorch / CUDA / cuDNN / library versions and the GPU, captured on the first
                 start. If a later start finds a different environment it is NOT refused (drivers get
                 updated) -- it prints a warning and appends the differences to environment_changes.
  results        per_fold metrics for each model as folds finish, mean +/- SD across folds, the paired
                 (multimodal - ablation) AUC-ROC difference per fold, and a "complete" flag. Updated after
                 every finished fold, so it is always current after a power cut.

Every write is atomic (temporary file + retrying os.replace, like Module 06): a power cut can never leave a
half-written run_config.json.
"""

import json
import os
import platform
import tempfile
from datetime import datetime
from pathlib import Path

import numpy as np
import torch

TRAINING_DIR = PHASE3_ARTIFACTS_DIR / "cv_training"     # placeholder name -- rename freely
RUN_CONFIG_FILENAME = "run_config.json"
RUN_CONFIG_FORMAT_VERSION = 1


class RunConfigError(RuntimeError):
    """run_config.json is unreadable or not in the expected format."""


class RunConfigMismatchError(RunConfigError):
    """The settings of this run differ from the ones recorded in run_config.json."""


def model_dir(training_dir, model_kind):
    return Path(training_dir) / model_kind


def run_config_path(training_dir):
    return Path(training_dir) / RUN_CONFIG_FILENAME


def _now():
    return datetime.now().astimezone().isoformat(timespec="seconds")


def build_settings():
    """The settings section, read from the live constants of Modules 01-08."""
    return {
        "seed": RANDOM_SEED,
        "run_order": "fold by fold; within each fold: multimodal first, then the image-only ablation",
        "data": {
            "n_folds": N_FOLDS,
            "cv_pool_rows": EXPECTED_POOL_ROWS,
            "classes": list(CLASSES),
            "image_source": "Phase 1 final preprocessed images, 300x300",
            "metadata_dim": METADATA_DIM,
            "metadata_features": list(FEATURE_NAMES),
            "augmentation": dict(AUGMENTATION_CONFIG),
        },
        "model": {
            "multimodal": "EfficientNet-B3 backbone (ImageNet-1K V1) -> MetaBlock fusion on the unpooled 10x10 map "
                          "-> global average pool -> Dropout -> Linear",
            "ablation": "EfficientNet-B3 backbone (ImageNet-1K V1) -> global average pool -> Dropout -> Linear "
                        "(metadata ignored)",
            "feature_channels": FEATURE_CHANNELS,
            "metadata_branch": f"Linear({METADATA_DIM} -> {META_HIDDEN}) -> ReLU -> Dropout({META_DROPOUT})",
            "head_dropout": HEAD_DROPOUT,
            "num_classes": NUM_CLASSES,
        },
        "training": {
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "phase_a_epochs": PHASE_A_EPOCHS,
            "phase_a": "backbone frozen (weights and BatchNorm statistics); head trains",
            "phase_b": "everything trains",
            "patience_counted_from_epoch": PHASE_A_EPOCHS + 1,
            "earliest_stop_epoch": PHASE_A_EPOCHS + PATIENCE,
            "best_epoch_rule": "highest validation AUC-ROC (strictly greater; ties do not count); "
                               "all epochs compete, Phase A included",
            "optimizer": "AdamW",
            "backbone_lr": BACKBONE_LR,
            "head_lr": HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
            "weight_decay_applies_to": "all parameters",
            "lr_schedule": "none (flat)",
            "loss": "CrossEntropyLoss with per-fold effective-number class weights",
            "effective_number_beta": BETA_FINAL,
            "sampler": "WeightedRandomSampler, balanced by class, with replacement",
            "precision": "bf16 autocast on GPU" if USE_AMP else "fp32 (no GPU)",
            "num_workers": 0,
        },
    }


def build_paths():
    return {
        "fold_table": str(FOLD_CSV_PATH),
        "class_weights": str(WEIGHTS_PATH),
        "images_dir": str(FINAL_IMAGES_DIR),
    }


def capture_environment():
    env = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    for module_name in ("torchvision", "numpy", "pandas", "sklearn", "cv2"):
        try:
            env[module_name] = getattr(__import__(module_name), "__version__", "unknown")
        except Exception:                                       # noqa: BLE001 -- a missing library is data, not an error
            env[module_name] = "not installed"
    return env


def _atomic_write_json(path, obj):
    path = str(path)
    folder = os.path.dirname(path) or "."
    os.makedirs(folder, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=folder, suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=2, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        _replace_with_retry(tmp_path, path)
    except BaseException:
        _remove_quietly(tmp_path)
        raise


def _read_config(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            config = json.load(f)
    except (OSError, ValueError) as exc:
        raise RunConfigError(f"{path} could not be read ({type(exc).__name__}: {exc}). Do not delete it "
                             f"while checkpoints exist next to it -- restore it or ask for help.") from exc
    if not isinstance(config, dict) or config.get("format_version") != RUN_CONFIG_FORMAT_VERSION \
            or not all(k in config for k in ("settings", "environment", "results")):
        raise RunConfigError(f"{path} is not a run_config.json this module wrote (missing parts or wrong format version).")
    return config


def _flatten(d, prefix=""):
    out = {}
    for key, value in d.items():
        if isinstance(value, dict):
            out.update(_flatten(value, f"{prefix}{key}."))
        else:
            out[f"{prefix}{key}"] = value
    return out


def prepare_run_config(training_dir):
    """First start: write run_config.json. Later starts: check it. Returns the config dict.
    Raises RunConfigMismatchError if the settings differ from the recorded ones."""
    path = run_config_path(training_dir)
    settings = json.loads(json.dumps(build_settings()))       # exactly the form it takes after a save and load
    environment = capture_environment()

    if not path.exists():
        config = {
            "format_version": RUN_CONFIG_FORMAT_VERSION,
            "created_at": _now(),
            "settings": settings,
            "paths": build_paths(),
            "environment": environment,
            "environment_changes": [],
            "results": {"per_fold": {kind: {} for kind in MODEL_KINDS}, "mean_sd": {},
                        "paired_auc_roc_difference": {}, "complete": False},
        }
        _atomic_write_json(path, config)
        print(f"run_config.json created: {path}")
        return config

    config = _read_config(path)
    old, new = _flatten(config["settings"]), _flatten(settings)
    differences = [f"  {key}: recorded {old.get(key, '<absent>')!r}, now {new.get(key, '<absent>')!r}"
                   for key in sorted(set(old) | set(new)) if old.get(key, "<absent>") != new.get(key, "<absent>")]
    if differences:
        raise RunConfigMismatchError(
            f"The settings of this run differ from those recorded in {path}:\n" + "\n".join(differences) +
            "\nA run must keep one set of rules from its first epoch to its last. If the change is intended, "
            "start a NEW training folder (change TRAINING_DIR); to keep the old rules, restore the old values.")

    changed = {key: {"was": config["environment"].get(key), "now": environment.get(key)}
               for key in sorted(set(config["environment"]) | set(environment))
               if config["environment"].get(key) != environment.get(key)}
    if changed:
        print("WARNING: the software environment differs from the one this run started in:")
        for key, pair in changed.items():
            print(f"  {key}: was {pair['was']!r}, now {pair['now']!r}")
        log = config.setdefault("environment_changes", [])
        if not log or log[-1]["differences"] != changed:        # a repeat of the same difference is not logged again
            log.append({"at": _now(), "differences": changed})
            _atomic_write_json(path, config)
    print(f"run_config.json found and matches this run's settings: {path}")
    return config


def _refresh_summaries(results):
    """Recompute mean +/- SD, the paired AUC difference and the complete flag from per_fold."""
    results["mean_sd"] = {}
    for kind, folds in results["per_fold"].items():
        if len(folds) >= 2:
            ordered = [folds[k]["metrics"] for k in sorted(folds, key=int)]
            results["mean_sd"][kind] = {"n_folds": len(folds), **summarize_across_folds(ordered)}

    per_fold = results["per_fold"]
    shared = sorted(set(per_fold.get("multimodal", {})) & set(per_fold.get("ablation", {})), key=int)
    results["paired_auc_roc_difference"] = {}
    if shared:
        diffs = {k: per_fold["multimodal"][k]["metrics"]["auc_roc"] - per_fold["ablation"][k]["metrics"]["auc_roc"]
                 for k in shared}
        results["paired_auc_roc_difference"] = {"per_fold": diffs, "n_folds": len(diffs)}
        if len(diffs) >= 2:
            values = list(diffs.values())
            results["paired_auc_roc_difference"]["mean"] = float(np.mean(values))
            results["paired_auc_roc_difference"]["sd"] = float(np.std(values, ddof=1))
    results["complete"] = all(len(per_fold.get(kind, {})) == N_FOLDS for kind in MODEL_KINDS)


def record_fold_result(training_dir, model_kind, fold_result):
    """Merge one finished fold (the dict train_fold returns) into run_config.json. Read, update, atomic write."""
    path = run_config_path(training_dir)
    config = _read_config(path)
    results = config["results"]
    results["per_fold"].setdefault(model_kind, {})[str(fold_result["fold"])] = {
        "metrics": {k: float(v) for k, v in fold_result["metrics"].items()},
        "best_epoch": int(fold_result["best_epoch"]),
        "best_epoch_phase": fold_result["best_epoch_phase"],
        "best_val_auc_roc": float(fold_result["best_val_auc"]),
        "epochs_run": int(fold_result["epochs_run"]),
        "stop_reason": fold_result["stop_reason"],
        "predictions_file": Path(fold_result["predictions_path"]).name,
    }
    _refresh_summaries(results)
    _atomic_write_json(path, config)
    return config


print("Run configuration loaded.")
print(f"  run folder (TRAINING_DIR): {TRAINING_DIR}")
print(f"  run_config.json: settings (guarded) | paths | environment (warned) | results (updated per finished fold)")

Run configuration loaded.
  run folder (TRAINING_DIR): T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\cv_training
  run_config.json: settings (guarded) | paths | environment (warned) | results (updated per finished fold)


In [34]:
"""
09_01_fold_drivers.txt
Phase 3 -- Module 09: Training Drivers -- the run itself (reusable)

Requires these cells already run: 01_00, 01_01, 02_00, 02_01, 03_02, 04_00, 05_00, 06_00, 07_00, 07_01, 07_02,
08_00, 09_00, and an `encoder_record` in memory (after a kernel restart:  encoder_record = load_encoder_stats()).

  run_cv_training(encoder_record)   trains the multimodal model AND the matched image-only ablation on all 5
                                    folds, FOLD BY FOLD:
                                        fold 0 multimodal, fold 0 ablation, fold 1 multimodal, fold 1 ablation, ...
                                    so after each fold there is a complete matched pair, and an interruption
                                    never leaves one model finished and the other barely started.
  training_status()                 one line per fold and model: not started / in progress (last completed
                                    epoch) / finished. Safe to run any time, also while nothing is running.

POWER CUTS
  Just run the same cell again. train_fold (Module 07) skips finished folds (rebuilding their results from the
  saved best weights, a few seconds each), resumes the interrupted one from its last completed epoch, and starts
  the rest fresh. run_config.json (Module 09_00) is checked, so the run cannot silently continue under
  different settings. Ctrl+C / "Interrupt" in the notebook is also safe: it stops cleanly, nothing is lost.

WHAT GETS SAVED  (in TRAINING_DIR -- see 09_00)
  after every EPOCH:   <model>/fold_N_resume.pt (and fold_N_best.pt when validation AUC-ROC improved)
  after every FOLD:    <model>/fold_N_val_predictions.npz, and the fold's metrics merged into run_config.json
  at the END:          the printed results table (mean +/- SD per model, paired AUC-ROC difference)

The settings passed to train_fold are read from the live constants of Modules 07_01 / 09_00 at CALL time, so what
run_config.json records is exactly what runs.
"""

import time
from pathlib import Path

MODEL_BUILDERS = {
    "multimodal": lambda: build_multimodal_model(pretrained=True),
    "ablation": lambda: build_image_only_model(pretrained=True),
}

_METRIC_LABELS = (("auc_roc", "AUC-ROC"), ("f1", "F1"), ("sensitivity", "Sensitivity"),
                  ("specificity", "Specificity"), ("accuracy", "Accuracy"))


def _real_dataset_factory(encoder_record, fold_table):
    def make_datasets(fold_num):
        return (HAM10000MultimodalFoldDataset(fold_num, "train", encoder_record, fold_table=fold_table),
                HAM10000MultimodalFoldDataset(fold_num, "val", encoder_record, fold_table=fold_table))
    return make_datasets


def _check_folds(folds):
    folds = [f for f in (range(N_FOLDS) if folds is None else folds)]
    for f in folds:
        if not isinstance(f, int) or isinstance(f, bool) or not 0 <= f < N_FOLDS:
            raise ValueError(f"folds must be integers in 0..{N_FOLDS - 1}, got {f!r}")
    if len(set(folds)) != len(folds):
        raise ValueError(f"folds contains a repeated fold: {folds}")
    return folds


def training_status(training_dir=None, folds=None, print_table=True):
    """{(fold, model_kind): status text}. A damaged resume file is reported, never raised."""
    training_dir = Path(training_dir) if training_dir else TRAINING_DIR
    status = {}
    for fold in _check_folds(folds):
        for kind in MODEL_KINDS:
            resume_path, _ = checkpoint_paths(model_dir(training_dir, kind), fold)
            try:
                state = load_resume_state(resume_path, make_identity(kind, fold))
                if state is None:
                    text = "not started"
                elif state["finished"]:
                    text = f"finished (epoch {state['epoch']}, best val AUC {state['best_metric']:.4f} @ epoch {state['best_epoch']})"
                else:
                    text = f"in progress (last completed epoch {state['epoch']})"
            except CheckpointError as exc:
                text = f"UNREADABLE -- {exc}"
            status[(fold, kind)] = text
    if print_table:
        print(f"Training status -- {training_dir}")
        for (fold, kind), text in status.items():
            print(f"  fold {fold} | {kind:10s} | {text}")
    return status


def print_results_table(results):
    """Print mean +/- SD per model and the paired AUC-ROC difference from run_config.json's results section."""
    print("\n" + "=" * 60)
    print("RESULTS SO FAR (validation folds, best-epoch weights)")
    print("=" * 60)
    for kind in MODEL_KINDS:
        folds_done = sorted(results["per_fold"].get(kind, {}), key=int)
        if kind in results["mean_sd"]:
            summary = results["mean_sd"][kind]
            parts = [f"{label} {summary[key]['mean']:.4f} +/- {summary[key]['sd']:.4f}" for key, label in _METRIC_LABELS]
            print(f"  {kind:10s} ({summary['n_folds']} folds): " + " | ".join(parts))
        else:
            print(f"  {kind:10s}: {len(folds_done)} fold(s) finished -- mean +/- SD needs at least 2")
    paired = results.get("paired_auc_roc_difference", {})
    if paired.get("per_fold"):
        per_fold = ", ".join(f"fold {k}: {v:+.4f}" for k, v in paired["per_fold"].items())
        line = f"  AUC-ROC difference (multimodal - ablation), {per_fold}"
        if "mean" in paired:
            line += f" | mean {paired['mean']:+.4f} +/- {paired['sd']:.4f}"
        print(line)
    print("  " + ("COMPLETE: all folds of both models are finished." if results["complete"]
                  else "NOT complete yet: run the cell again to finish the remaining folds."))
    print("=" * 60)


def run_cv_training(encoder_record, folds=None, training_dir=None, make_datasets=None, model_builders=None,
                    criterion_fn=None, log_fn=print):
    """
    encoder_record: the dict from load_encoder_stats() (Module 01).
    folds:          None = all 5, or a list such as [0, 1] to do only those (the rest can follow later).
    training_dir:   None = TRAINING_DIR.
    make_datasets, model_builders, criterion_fn: leave None. They exist so the driver can be tested on
                    stand-in data; the real datasets, models and weighted loss are the defaults.
    Returns the results section of run_config.json.
    """
    training_dir = Path(training_dir) if training_dir else TRAINING_DIR
    folds = _check_folds(folds)
    builders = model_builders or MODEL_BUILDERS
    if make_datasets is None:
        make_datasets = _real_dataset_factory(encoder_record, load_fold_table_with_metadata())
    if criterion_fn is None:
        criterion_fn = lambda fold: build_weighted_criterion(fold, CLASS_TO_IDX)

    print("=" * 60)
    print(f"CV TRAINING -- folds {folds}, both models, fold by fold")
    print("=" * 60)
    prepare_run_config(training_dir)
    training_status(training_dir, folds)

    started = time.time()
    try:
        for fold in folds:
            pair = {}
            for kind in MODEL_KINDS:
                result = train_fold(
                    fold, kind, model_dir(training_dir, kind),
                    make_datasets=make_datasets, build_model_fn=builders[kind], build_criterion_fn=criterion_fn,
                    batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                    device=DEVICE, use_amp=USE_AMP, log_fn=log_fn)
                record_fold_result(training_dir, kind, result)
                pair[kind] = result
            mm, ab = pair["multimodal"], pair["ablation"]
            print(f"\n>>> Fold {fold} finished | multimodal AUC-ROC {mm['metrics']['auc_roc']:.4f} "
                  f"(best epoch {mm['best_epoch']}, {mm['epochs_run']} run) | ablation AUC-ROC "
                  f"{ab['metrics']['auc_roc']:.4f} (best epoch {ab['best_epoch']}, {ab['epochs_run']} run) | "
                  f"difference {mm['metrics']['auc_roc'] - ab['metrics']['auc_roc']:+.4f} | "
                  f"elapsed {(time.time() - started) / 3600:.2f} h this session\n")
    except KeyboardInterrupt:
        print("\nStopped by you. Nothing is lost -- run this cell again to resume where it stopped.")
        raise

    results = _read_config(run_config_path(training_dir))["results"]
    print_results_table(results)
    return results


print("Fold drivers loaded.")
print("  run_cv_training(encoder_record)   -- both models, all folds, fold by fold; safe to re-run after any interruption")
print("  training_status()                 -- what is finished / in progress / not started")

Fold drivers loaded.
  run_cv_training(encoder_record)   -- both models, all folds, fold by fold; safe to re-run after any interruption
  training_status()                 -- what is finished / in progress / not started


In [35]:
"""
09_02_driver_verification.txt
Phase 3 -- Module 09: Training Drivers -- Verification (run once)

Requires 01_00, 02_00, 03_00, 04_00, 04_01, 05_00, 06_00, 07_00, 07_01, 07_02, 08_00, 09_00, 09_01 already run.
Runs the REAL run_cv_training / run_config code on CPU with a tiny stand-in model and synthetic data (24 training
and 28 validation "images" of 3x8x8, batch 8, 6 epochs per fold): no dataset, no pretrained weights, no GPU. To do
that it temporarily swaps DEVICE, USE_AMP, BATCH_SIZE, MAX_EPOCHS and PATIENCE for small values and restores them afterwards.
Files go to a temporary folder that is deleted afterwards. Your real TRAINING_DIR is never touched.

All six checks run as a full set (failures collected, one RuntimeError at the end).

Checks:
  1. Run config      -- run_config.json is created with all four parts; its settings equal the live constants;
                        the environment matches this session; a second start finds nothing to rewrite.
  2. Settings guard  -- a changed setting (patience, backbone learning rate) is refused with the setting named and
                        the file left untouched; a changed environment only warns and is logged once; a corrupt
                        file or a wrong format version is refused.
  3. Fold by fold    -- the training order is fold 0 multimodal, fold 0 ablation, fold 1 multimodal, ...; train_fold
                        receives exactly the batch size / max epochs / patience run_config.json records; results
                        merge into run_config.json across separate calls; mean +/- SD and the paired AUC-ROC
                        difference equal a by-hand recomputation; the recorded metrics reproduce from the
                        predictions files on disk; earlier folds are untouched by later calls.
  4. Interruption    -- a run cut in the middle of fold 1's ablation (run_config.json already holds every finished
                        fold at that moment) and re-run: finished folds are not retrained,
                        the cut fold resumes from its last completed epoch (exactly 39 epochs trained in the
                        re-run), and the final results equal an uninterrupted run's.
  5. Status report   -- training_status names finished / in progress (with the epoch) / not started correctly, and
                        reports a damaged resume file without crashing.
  6. Guards          -- fold numbers out of range, repeated or non-integer are refused; a settings change is
                        refused by the driver BEFORE any training starts.
"""

import contextlib
import io
import json
import os
import platform
import tempfile
import warnings

import numpy as np
import torch
import torch.nn as nn


class _D9TinyNet(nn.Module):
    """Stand-in with the real models' structure. `hidden` differs between the two 'models' so their results differ."""

    def __init__(self, hidden=6):
        super().__init__()
        self.backbone = nn.Sequential(nn.Conv2d(3, 4, 3, padding=1), nn.BatchNorm2d(4), nn.ReLU(),
                                      nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Sequential(nn.Linear(4, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.5),
                                  nn.Linear(hidden, 7))

    def forward(self, image, metadata):
        return self.head(self.backbone(image))

    def backbone_parameters(self):
        return list(self.backbone.parameters())

    def head_parameters(self):
        return list(self.head.parameters())


class _D9SynthDataset:
    def __init__(self, n, seed, prefix):
        g = torch.Generator()
        g.manual_seed(seed)
        self.images = torch.randn(n, 3, 8, 8, generator=g)
        self.meta = torch.randn(n, METADATA_DIM, generator=g)
        self.labels = np.arange(n) % 7
        self.image_ids = [f"{prefix}_{i:03d}" for i in range(n)]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, i):
        return self.images[i], self.meta[i], int(self.labels[i])

    def get_labels(self):
        return self.labels


class _D9PowerCut(Exception):
    pass


@contextlib.contextmanager
def _d9_patch_globals(**overrides):
    g = globals()
    saved = {name: g[name] for name in overrides}
    g.update(overrides)
    try:
        yield
    finally:
        g.update(saved)


@contextlib.contextmanager
def _d9_patch_function(name, make_replacement):
    g = globals()
    real = g[name]
    g[name] = make_replacement(real)
    try:
        yield
    finally:
        g[name] = real


def _d9_read(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _d9_bytes(path):
    with open(path, "rb") as f:
        return f.read()


def run_driver_verification():
    failures = []
    print("=" * 60)
    print("DRIVER VERIFICATION -- Phase 3, Module 09")
    print("=" * 60)
    print("  real run_cv_training + run_config, CPU, tiny stand-in models, synthetic data, temporary folder")

    cpu = torch.device("cpu")
    make_datasets = lambda fold: (_D9SynthDataset(24, 100 + fold, "tr"), _D9SynthDataset(28, 200 + fold, "va"))
    builders = {"multimodal": lambda: _D9TinyNet(6), "ablation": lambda: _D9TinyNet(5)}
    criterion_fn = lambda fold: nn.CrossEntropyLoss()

    def drive(training_dir, folds=None):
        """run_cv_training with the stand-ins; returns (results, printed text)."""
        printed = io.StringIO()
        with contextlib.redirect_stdout(printed):
            results = run_cv_training(None, folds=folds, training_dir=training_dir, make_datasets=make_datasets,
                                      model_builders=builders, criterion_fn=criterion_fn, log_fn=lambda *_: None)
        return results, printed.getvalue()

    small = dict(DEVICE=cpu, USE_AMP=False, BATCH_SIZE=8, MAX_EPOCHS=6, PATIENCE=8)   # PATIENCE != its real 10, so a hard-coded 10 shows

    with tempfile.TemporaryDirectory() as tmp, warnings.catch_warnings(), _d9_patch_globals(**small):
        # Phase 2's build_balanced_sampler (copied verbatim) triggers a harmless pandas/NumPy
        # "array is not writable" warning; hide it here so the report stays readable.
        warnings.filterwarnings("ignore", message="The given NumPy array is not writable")

        def d(name):
            return os.path.join(tmp, name)

        # --- 1. Run config ---
        print("\n[1/6] run_config.json: created, complete, matches the live constants...")
        ok = True
        with contextlib.redirect_stdout(io.StringIO()):
            prepare_run_config(d("cfg"))
        cfg_path = run_config_path(d("cfg"))
        try:
            cfg = _d9_read(cfg_path)
        except Exception as exc:
            cfg = {}
            failures.append(f"Run config: the file cannot be read back as JSON -- {exc}"); ok = False
        for part in ("format_version", "created_at", "settings", "paths", "environment", "environment_changes", "results"):
            if part not in cfg:
                failures.append(f"Run config: part '{part}' is missing"); ok = False
        if cfg:
            training, data = cfg["settings"]["training"], cfg["settings"]["data"]
            live = {"batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
                    "phase_a_epochs": PHASE_A_EPOCHS, "backbone_lr": BACKBONE_LR, "head_lr": HEAD_LR,
                    "weight_decay": WEIGHT_DECAY, "effective_number_beta": BETA_FINAL,
                    "earliest_stop_epoch": PHASE_A_EPOCHS + PATIENCE, "patience_counted_from_epoch": PHASE_A_EPOCHS + 1}
            for key, value in live.items():
                if training.get(key) != value:
                    failures.append(f"Run config: settings.training.{key} is {training.get(key)!r}, live value {value!r}"); ok = False
            if cfg["settings"]["seed"] != RANDOM_SEED or data["classes"] != list(CLASSES) or data["n_folds"] != N_FOLDS:
                failures.append("Run config: seed, classes or fold count differ from the live constants"); ok = False
            env = cfg["environment"]
            if env.get("python") != platform.python_version() or env.get("torch") != torch.__version__ \
                    or env.get("cuda") != torch.version.cuda:
                failures.append("Run config: the recorded Python / PyTorch / CUDA versions differ from this session's"); ok = False
            res = cfg["results"]
            if set(res["per_fold"]) != set(MODEL_KINDS) or res["complete"] is not False or cfg["environment_changes"] != []:
                failures.append("Run config: the fresh results section or environment log is not empty as expected"); ok = False
        before = _d9_bytes(cfg_path)
        with contextlib.redirect_stdout(io.StringIO()):
            prepare_run_config(d("cfg"))
        if _d9_bytes(cfg_path) != before:
            failures.append("Run config: a second start with nothing changed rewrote the file"); ok = False
        print("  OK -- all four parts present; settings equal the live constants; environment recorded; "
              "a second start leaves the file alone" if ok else "  FAILED -- see summary")

        # --- 2. Settings guard ---
        print("\n[2/6] Settings guard...")
        ok = True
        before = _d9_bytes(cfg_path)
        for name, value, must_name in (("PATIENCE", 7, "training.patience"), ("BACKBONE_LR", 5e-4, "training.backbone_lr")):
            try:
                with _d9_patch_globals(**{name: value}), contextlib.redirect_stdout(io.StringIO()):
                    prepare_run_config(d("cfg"))
                failures.append(f"Settings guard: a changed {name} was accepted"); ok = False
            except RunConfigMismatchError as exc:
                if must_name not in str(exc):
                    failures.append(f"Settings guard: the refusal for {name} does not name '{must_name}'"); ok = False
        if _d9_bytes(cfg_path) != before:
            failures.append("Settings guard: run_config.json changed although the settings were refused"); ok = False

        real_env = capture_environment()
        fake = lambda tag: (lambda: {**real_env, "torch": tag})
        printed = io.StringIO()
        with _d9_patch_globals(capture_environment=fake("0.0-test-a")), contextlib.redirect_stdout(printed):
            prepare_run_config(d("cfg"))
            prepare_run_config(d("cfg"))                        # the same difference again: logged only once
        if "WARNING" not in printed.getvalue():
            failures.append("Settings guard: a changed environment printed no warning"); ok = False
        log = _d9_read(cfg_path)["environment_changes"]
        if len(log) != 1 or log[0]["differences"].get("torch", {}).get("now") != "0.0-test-a":
            failures.append(f"Settings guard: the same environment difference was logged {len(log)} time(s), expected 1"); ok = False
        with _d9_patch_globals(capture_environment=fake("0.0-test-b")), contextlib.redirect_stdout(io.StringIO()):
            prepare_run_config(d("cfg"))
        after = _d9_read(cfg_path)
        if len(after["environment_changes"]) != 2:
            failures.append("Settings guard: a NEW environment difference was not logged"); ok = False
        if after["environment"].get("torch") != torch.__version__:
            failures.append("Settings guard: the recorded original environment was overwritten"); ok = False

        os.makedirs(d("corrupt"), exist_ok=True)
        with open(run_config_path(d("corrupt")), "w") as f:
            f.write("{not json")
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                prepare_run_config(d("corrupt"))
            failures.append("Settings guard: a corrupt run_config.json was accepted"); ok = False
        except RunConfigError:
            pass
        os.makedirs(d("oldfmt"), exist_ok=True)
        with open(run_config_path(d("oldfmt")), "w") as f:
            json.dump({**cfg, "format_version": 99}, f)
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                prepare_run_config(d("oldfmt"))
            failures.append("Settings guard: a wrong format version was accepted"); ok = False
        except RunConfigError:
            pass
        print("  OK -- changed settings refused (named, file untouched); environment change warns and is logged once; "
              "corrupt / foreign file refused" if ok else "  FAILED -- see summary")

        # --- 3. Fold by fold ---
        print("\n[3/6] Fold-by-fold order, merged results, mean +/- SD, paired difference...")
        ok = True
        calls, passed = [], []

        def make_recorder(real):
            def wrapped(fold_num, model_kind, *args, **kwargs):
                calls.append((fold_num, model_kind))
                passed.append({k: kwargs.get(k) for k in ("batch_size", "max_epochs", "patience", "use_amp")})
                return real(fold_num, model_kind, *args, **kwargs)
            return wrapped

        with _d9_patch_function("train_fold", make_recorder):
            _, text_a = drive(d("run"), folds=[0, 1])
        want_order = [(0, "multimodal"), (0, "ablation"), (1, "multimodal"), (1, "ablation")]
        if calls != want_order:
            failures.append(f"Fold by fold: training order was {calls}, expected {want_order}"); ok = False
        if "NOT complete" not in text_a:
            failures.append("Fold by fold: the partial run did not say it is not complete"); ok = False
        recorded = _d9_read(run_config_path(d("run")))["settings"]["training"]
        for used in passed:
            if used != {"batch_size": recorded["batch_size"], "max_epochs": recorded["max_epochs"],
                        "patience": recorded["patience"], "use_amp": False}:
                failures.append(f"Fold by fold: train_fold was given {used}, but run_config.json records "
                                f"batch {recorded['batch_size']}, max epochs {recorded['max_epochs']}, "
                                f"patience {recorded['patience']}"); ok = False
                break
        res2 = _d9_read(run_config_path(d("run")))["results"]
        for kind in MODEL_KINDS:
            if sorted(res2["per_fold"].get(kind, {})) != ["0", "1"]:
                failures.append(f"Fold by fold: {kind} has folds {sorted(res2['per_fold'].get(kind, {}))} recorded, expected ['0', '1']"); ok = False
        if res2["complete"] is not False:
            failures.append("Fold by fold: 'complete' is set after only 2 of 5 folds"); ok = False

        def by_hand(per_fold_kind, key):
            values = [per_fold_kind[k]["metrics"][key] for k in sorted(per_fold_kind, key=int)]
            return float(np.mean(values)), float(np.std(values, ddof=1))

        def check_summaries(res, label):
            good = True
            for kind in MODEL_KINDS:
                entry = res["mean_sd"].get(kind)
                if entry is None or entry["n_folds"] != len(res["per_fold"][kind]):
                    failures.append(f"Fold by fold ({label}): mean +/- SD for {kind} is missing or has the wrong fold count"); good = False
                    continue
                for key in ("auc_roc", "f1", "sensitivity", "specificity", "accuracy"):
                    mean, sd = by_hand(res["per_fold"][kind], key)
                    if not (np.isclose(entry[key]["mean"], mean, atol=1e-12) and np.isclose(entry[key]["sd"], sd, atol=1e-12)):
                        failures.append(f"Fold by fold ({label}): {kind} {key} mean +/- SD differs from a by-hand recomputation"); good = False
            paired = res["paired_auc_roc_difference"]
            diffs = {k: res["per_fold"]["multimodal"][k]["metrics"]["auc_roc"] - res["per_fold"]["ablation"][k]["metrics"]["auc_roc"]
                     for k in res["per_fold"]["multimodal"] if k in res["per_fold"]["ablation"]}
            if set(paired.get("per_fold", {})) != set(diffs) or not all(np.isclose(paired["per_fold"][k], diffs[k], atol=1e-12) for k in diffs):
                failures.append(f"Fold by fold ({label}): the paired AUC-ROC differences are wrong"); good = False
            elif not (np.isclose(paired.get("mean", np.nan), np.mean(list(diffs.values())), atol=1e-12)
                      and np.isclose(paired.get("sd", np.nan), np.std(list(diffs.values()), ddof=1), atol=1e-12)):
                failures.append(f"Fold by fold ({label}): the paired difference mean / SD is wrong"); good = False
            if all(abs(v) < 1e-12 for v in diffs.values()):
                failures.append(f"Fold by fold ({label}): the two stand-in models gave identical AUCs -- test setup is not discriminating"); good = False
            return good
        ok = check_summaries(res2, "2 folds") and ok

        for kind in MODEL_KINDS:                                 # recorded metrics reproduce from the files on disk
            for fold_key in ("0", "1"):
                if fold_key not in res2["per_fold"].get(kind, {}):
                    continue                                     # already reported as missing above
                preds_file = os.path.join(model_dir(d("run"), kind), res2["per_fold"][kind][fold_key]["predictions_file"])
                with np.load(preds_file, allow_pickle=False) as f:
                    recomputed = compute_fold_metrics(f["labels"], f["probs"], n_classes=len(CLASSES))
                if recomputed != res2["per_fold"][kind][fold_key]["metrics"]:
                    failures.append(f"Fold by fold: recorded metrics of {kind} fold {fold_key} do not reproduce from its predictions file"); ok = False

        snapshot = json.loads(json.dumps(res2["per_fold"]))
        calls.clear()
        with _d9_patch_function("train_fold", make_recorder):
            _, text_b = drive(d("run"), folds=[2, 3, 4])
        if calls != [(f, k) for f in (2, 3, 4) for k in MODEL_KINDS]:
            failures.append(f"Fold by fold: the second call trained {calls}"); ok = False
        res5 = _d9_read(run_config_path(d("run")))["results"]
        for kind in MODEL_KINDS:
            if sorted(res5["per_fold"][kind], key=int) != ["0", "1", "2", "3", "4"]:
                failures.append(f"Fold by fold: {kind} has folds {sorted(res5['per_fold'][kind])} after both calls"); ok = False
            for fold_key in ("0", "1"):
                if res5["per_fold"][kind].get(fold_key) != snapshot.get(kind, {}).get(fold_key):
                    failures.append(f"Fold by fold: {kind} fold {fold_key} changed when later folds were added"); ok = False
        if res5["complete"] is not True or "COMPLETE" not in text_b:
            failures.append("Fold by fold: after all 5 folds the run is not reported complete"); ok = False
        ok = check_summaries(res5, "5 folds") and ok
        print("  OK -- order fold 0 mm, fold 0 ab, fold 1 mm ...; results merge across calls; mean +/- SD and paired "
              "difference match by-hand values; metrics reproduce from the predictions files" if ok else "  FAILED -- see summary")

        # --- 4. Interruption ---
        print("\n[4/6] A run cut in the middle of fold 1's ablation, then re-run...")
        ok = True

        def cut_fold1_ablation(real):
            def wrapped(resume_path, identity, epoch, *args, **kwargs):
                real(resume_path, identity, epoch, *args, **kwargs)
                if identity["fold"] == 1 and identity["model_kind"] == "ablation" and epoch == 3:
                    raise _D9PowerCut()
            return wrapped

        try:
            with _d9_patch_function("save_resume_state", cut_fold1_ablation):
                drive(d("cut"))
            failures.append("Interruption: the simulated power cut did not happen"); ok = False
        except _D9PowerCut:
            pass
        after_cut = _d9_read(run_config_path(d("cut")))["results"]["per_fold"]
        if sorted(after_cut["multimodal"]) != ["0", "1"] or sorted(after_cut["ablation"]) != ["0"]:
            failures.append(f"Interruption: right after the cut run_config.json holds multimodal folds "
                            f"{sorted(after_cut['multimodal'])} and ablation folds {sorted(after_cut['ablation'])}; "
                            f"expected ['0', '1'] and ['0'] (every finished fold is recorded at once)"); ok = False
        status = training_status(d("cut"), print_table=False)
        expect = {(0, "multimodal"): "finished", (0, "ablation"): "finished", (1, "multimodal"): "finished",
                  (1, "ablation"): "in progress (last completed epoch 3)", (2, "multimodal"): "not started",
                  (4, "ablation"): "not started"}
        for key, want in expect.items():
            if not status[key].startswith(want):
                failures.append(f"Status report {key}: '{status[key]}', expected it to start with '{want}'"); ok = False

        epoch_calls = []

        def count_epochs(real):
            def wrapped(*args, **kwargs):
                epoch_calls.append(1)
                return real(*args, **kwargs)
            return wrapped

        with _d9_patch_function("train_one_epoch", count_epochs):
            resumed, _ = drive(d("cut"))
        if len(epoch_calls) != 39:
            failures.append(f"Interruption: the re-run trained {len(epoch_calls)} epochs, expected 39 "
                            f"(3 left in fold 1's ablation + 3 folds x 2 models x 6)"); ok = False
        if resumed["complete"] is not True:
            failures.append("Interruption: the re-run did not complete the run"); ok = False
        if resumed["per_fold"] != res5["per_fold"] or resumed["mean_sd"] != res5["mean_sd"]:
            failures.append("Interruption: the final results differ from the uninterrupted run's"); ok = False
        print("  OK -- finished folds not retrained; the cut fold resumed from epoch 3; 39 epochs trained; final "
              "results equal the uninterrupted run's" if ok else "  FAILED -- see summary")

        # --- 5. Status report ---
        print("\n[5/6] Status report on a finished run with one damaged file...")
        ok = True
        status = training_status(d("cut"), print_table=False)
        if not all(text.startswith("finished") for text in status.values()):
            failures.append("Status report: not every fold of the completed run is reported finished"); ok = False
        damaged = checkpoint_paths(model_dir(d("cut"), "multimodal"), 4)[0]
        with open(damaged, "wb") as f:
            f.write(b"garbage")
        try:
            status = training_status(d("cut"), print_table=False)
            if not status[(4, "multimodal")].startswith("UNREADABLE"):
                failures.append("Status report: a damaged resume file was not reported as UNREADABLE"); ok = False
            if not status[(4, "ablation")].startswith("finished"):
                failures.append("Status report: a damaged file spoiled the report of an undamaged one"); ok = False
        except Exception as exc:
            failures.append(f"Status report: a damaged file crashed it -- {type(exc).__name__}: {exc}"); ok = False
        print("  OK -- finished / in progress (with epoch) / not started named correctly; a damaged file is "
              "reported, not raised" if ok else "  FAILED -- see summary")

        # --- 6. Guards ---
        print("\n[6/6] Guards...")
        ok = True
        for bad in ([5], [0, 0], [True], [-1], ["0"]):
            try:
                drive(d("guards"), folds=bad)
                failures.append(f"Guards: folds={bad} was accepted"); ok = False
            except ValueError:
                pass
        calls.clear()
        with _d9_patch_function("train_fold", make_recorder):
            drive(d("guards"), folds=[0])
            try:
                with _d9_patch_globals(PATIENCE=7):
                    drive(d("guards"), folds=[1])
                failures.append("Guards: the driver ran with settings that differ from run_config.json"); ok = False
            except RunConfigMismatchError:
                pass
        if calls != [(0, "multimodal"), (0, "ablation")]:
            failures.append(f"Guards: training happened before the settings check refused the run ({calls})"); ok = False
        print("  OK -- bad fold lists refused; a settings change is refused before any training starts" if ok
              else "  FAILED -- see summary")

    print("\n" + "=" * 60)
    if failures:
        print(f"VERIFICATION FAILED -- {len(failures)} issue(s):")
        for msg in failures:
            print(f"  - {msg}")
        print("=" * 60)
        raise RuntimeError(f"Driver verification failed with {len(failures)} issue(s).")
    print("VERIFICATION PASSED -- all checks OK. Module 09's drivers are safe to build on.")
    print("=" * 60)


# --- Usage ---
# run_driver_verification()

In [36]:
run_driver_verification()

DRIVER VERIFICATION -- Phase 3, Module 09
  real run_cv_training + run_config, CPU, tiny stand-in models, synthetic data, temporary folder

[1/6] run_config.json: created, complete, matches the live constants...
  OK -- all four parts present; settings equal the live constants; environment recorded; a second start leaves the file alone

[2/6] Settings guard...
  OK -- changed settings refused (named, file untouched); environment change warns and is logged once; corrupt / foreign file refused

[3/6] Fold-by-fold order, merged results, mean +/- SD, paired difference...
  OK -- order fold 0 mm, fold 0 ab, fold 1 mm ...; results merge across calls; mean +/- SD and paired difference match by-hand values; metrics reproduce from the predictions files

[4/6] A run cut in the middle of fold 1's ablation, then re-run...
  OK -- finished folds not retrained; the cut fold resumed from epoch 3; 39 epochs trained; final results equal the uninterrupted run's

[5/6] Status report on a finished run wit

In [37]:
"""
09_03_real_data_trial.txt
Phase 3 -- Module 09: Training Drivers -- real-data trial (run once, BEFORE the full run)

Requires these cells already run: 01_00, 01_01, 02_00, 02_01, 03_00-03_02, 04_00, 04_01, 05_00, 06_00, 07_00,
07_01, 07_02, 08_00, 09_00, 09_01, and an `encoder_record` in memory (after a kernel restart:
encoder_record = load_encoder_stats()).

WHY
  Everything so far was proven on stand-in data (Modules 04-07, 09_02) or on a handful of real steps (Module 08).
  The one thing never run is train_fold ITSELF on the real images, the real models, the real GPU and your real
  drive: real checkpoint sizes written to T:, real 7-class validation AUC-ROC, the Phase A -> Phase B switch, and
  a power cut followed by a resume. This trial does exactly that on a small stratified slice of fold 0 (about
  46 training and 21 validation images per class), for BOTH models, in about 5-8 minutes.

WHAT IT DOES, per model
  1. Runs train_fold for 7 epochs (5 Phase A + 2 Phase B), simulating a power cut right after epoch 3's save.
  2. Checks the resume file left behind (epoch 3, Phase A, not finished), then re-runs: it must continue with
     epoch 4 (exactly 4 more epochs, not a restart) and finish.
  3. Checks the finished fold: 7 epochs, Phase A/B labels, finite metrics, the best-weights file, the predictions
     file (ids, labels, probabilities that sum to 1), the 'finished' flag, no stray temporary files.
  4. Runs the same 7 epochs uninterrupted (its epoch lines are printed, so you can watch loss and AUC move) and
     compares it with the resumed run. On a GPU an exact match is NOT guaranteed, so this comparison is printed
     as information and never fails the trial.
  5. Times one save of the real resume file and prints the file sizes.
  Everything is written to TRIAL_DIR (a temporary folder next to your real results), which is deleted at the end.
  Your real TRAINING_DIR is never touched, and nothing here is reused by the real run.
"""

import contextlib
import shutil
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset

TRIAL_DIR = PHASE3_ARTIFACTS_DIR / "_trial_run"      # temporary; created and deleted by the trial
TRIAL_EPOCHS = 7                                     # 5 Phase A + 2 Phase B
TRIAL_CUT_AFTER_EPOCH = 3                            # the simulated power cut lands right after this epoch's save
TRIAL_TRAIN_PER_CLASS = 46
TRIAL_VAL_PER_CLASS = 21


class _T9PowerCut(Exception):
    pass


class _T9Subset(Dataset):
    """A fixed subset of a fold dataset, with the attributes train_fold needs (get_labels, image_ids)."""

    def __init__(self, base, indices):
        self.base = base
        self.indices = [int(i) for i in indices]
        self.image_ids = [base.image_ids[i] for i in self.indices]
        self.labels = np.asarray(base.labels)[self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.base[self.indices[i]]

    def get_labels(self):
        return self.labels


def _t9_stratified_indices(labels, per_class, batch_size=None):
    """The first `per_class` images of every class, in dataset order. If batch_size is given, one image is
    dropped when the count would leave a final batch of exactly ONE image (BatchNorm1d cannot train on it)."""
    labels = np.asarray(labels)
    picked = []
    for class_index in range(len(CLASSES)):
        picked.extend(np.flatnonzero(labels == class_index)[:per_class].tolist())
    picked.sort()
    if batch_size and len(picked) % batch_size == 1:
        picked = picked[:-1]
    return picked


@contextlib.contextmanager
def _t9_power_cut(after_epoch):
    g = globals()
    real = g["save_resume_state"]

    def wrapped(resume_path, identity, epoch, *args, **kwargs):
        real(resume_path, identity, epoch, *args, **kwargs)
        if epoch == after_epoch:
            raise _T9PowerCut()
    g["save_resume_state"] = wrapped
    try:
        yield
    finally:
        g["save_resume_state"] = real


@contextlib.contextmanager
def _t9_count_epochs():
    """Counts the training epochs train_one_epoch is asked to run while the block is active."""
    g = globals()
    real = g["train_one_epoch"]
    counter = []

    def wrapped(*args, **kwargs):
        counter.append(1)
        return real(*args, **kwargs)
    g["train_one_epoch"] = wrapped
    try:
        yield counter
    finally:
        g["train_one_epoch"] = real


def _t9_mb(path):
    return Path(path).stat().st_size / 1e6


def _t9_check_finished_fold(kind, res, out_dir, val_sub, failures):
    """Structural checks on a finished fold. Returns True if all held."""
    ok = True
    resume_path, best_path = checkpoint_paths(out_dir, res["fold"])
    identity = make_identity(kind, res["fold"])

    if res["epochs_run"] != TRIAL_EPOCHS or res["stop_reason"] != "max_epochs":
        failures.append(f"{kind}: ran {res['epochs_run']} epochs, stop reason '{res['stop_reason']}'; "
                        f"expected {TRIAL_EPOCHS} epochs and 'max_epochs'"); ok = False
    phases = [h["phase"] for h in res["history"]]
    if phases != ["A"] * PHASE_A_EPOCHS + ["B"] * (TRIAL_EPOCHS - PHASE_A_EPOCHS):
        failures.append(f"{kind}: epoch phases were {phases}"); ok = False
    metrics = res["metrics"]
    if not all(np.isfinite(v) and 0.0 <= v <= 1.0 for v in metrics.values()):
        failures.append(f"{kind}: metrics are not all finite values in [0, 1]: {metrics}"); ok = False
    if not 1 <= res["best_epoch"] <= TRIAL_EPOCHS:
        failures.append(f"{kind}: best epoch {res['best_epoch']} is outside 1..{TRIAL_EPOCHS}"); ok = False

    try:
        state = load_resume_state(resume_path, identity)
        if state is None or not state["finished"] or state["epoch"] != TRIAL_EPOCHS:
            failures.append(f"{kind}: the resume file does not show a finished fold at epoch {TRIAL_EPOCHS}"); ok = False
    except CheckpointError as exc:
        failures.append(f"{kind}: the resume file is unreadable -- {exc}"); ok = False

    try:
        best = torch.load(best_path, map_location="cpu", weights_only=False)
        if best["epoch"] != res["best_epoch"] or abs(best["val_auc"] - res["best_val_auc"]) > 1e-12 \
                or "model_state_dict" not in best:
            failures.append(f"{kind}: the best-weights file disagrees with the reported best epoch / AUC"); ok = False
    except Exception as exc:                                   # noqa: BLE001
        failures.append(f"{kind}: the best-weights file cannot be read -- {type(exc).__name__}: {exc}"); ok = False

    try:
        with np.load(res["predictions_path"], allow_pickle=False) as f:
            preds = {k: f[k] for k in f.files}
        probs = preds["probs"]
        if list(preds["image_ids"]) != list(val_sub.image_ids) or not np.array_equal(preds["labels"], val_sub.labels):
            failures.append(f"{kind}: the predictions file's image ids / labels do not match the validation slice"); ok = False
        if probs.shape != (len(val_sub), len(CLASSES)) or probs.dtype != np.float32 or not np.isfinite(probs).all() \
                or not np.allclose(probs.sum(axis=1), 1.0, atol=1e-4):
            failures.append(f"{kind}: predicted probabilities have the wrong shape / dtype or do not sum to 1"); ok = False
        if int(preds["best_epoch"]) != res["best_epoch"] or list(preds["class_names"]) != list(CLASSES):
            failures.append(f"{kind}: the predictions file's best epoch or class names are wrong"); ok = False
    except Exception as exc:                                   # noqa: BLE001
        failures.append(f"{kind}: the predictions file cannot be read -- {type(exc).__name__}: {exc}"); ok = False

    if [p for p in Path(out_dir).iterdir() if p.name.endswith(".tmp")]:
        failures.append(f"{kind}: stray temporary file(s) left in the output folder"); ok = False
    return ok


def _t9_run_one_model(kind, fold_num, trial_dir, make_datasets, builders, criterion_fn, val_sub, failures):
    cut_dir, full_dir = trial_dir / f"{kind}_cut", trial_dir / f"{kind}_full"
    quiet = lambda *_: None

    def run(out_dir, log_fn):
        return train_fold(fold_num, kind, out_dir, make_datasets=make_datasets, build_model_fn=builders[kind],
                          build_criterion_fn=criterion_fn, batch_size=BATCH_SIZE, max_epochs=TRIAL_EPOCHS,
                          patience=PATIENCE, device=DEVICE, use_amp=USE_AMP, log_fn=log_fn)

    print(f"\n--- {kind} model ---")
    started = time.time()

    # 1-2. simulated power cut, then the state it leaves behind
    print(f"  [a] training, with a simulated power cut right after epoch {TRIAL_CUT_AFTER_EPOCH}...")
    try:
        with _t9_power_cut(TRIAL_CUT_AFTER_EPOCH):
            run(cut_dir, quiet)
        failures.append(f"{kind}: the simulated power cut did not happen"); return
    except _T9PowerCut:
        pass
    resume_path, _ = checkpoint_paths(cut_dir, fold_num)
    try:
        left = load_resume_state(resume_path, make_identity(kind, fold_num))
    except CheckpointError as exc:
        failures.append(f"{kind}: the resume file left by the power cut is unreadable -- {exc}"); return
    if left is None or left["epoch"] != TRIAL_CUT_AFTER_EPOCH or left["phase"] != "A" or left["finished"]:
        failures.append(f"{kind}: after the power cut the resume file should show epoch {TRIAL_CUT_AFTER_EPOCH}, "
                        f"Phase A, not finished; got {None if left is None else (left['epoch'], left['phase'], left['finished'])}")
        return
    print(f"      resume file left behind: epoch {left['epoch']}, Phase {left['phase']}, not finished  OK")

    # 2-3. resume and finish
    print(f"  [b] re-running: must resume after epoch {TRIAL_CUT_AFTER_EPOCH} and finish...")
    with _t9_count_epochs() as trained:
        res_cut = run(cut_dir, quiet)
    ok = _t9_check_finished_fold(kind, res_cut, cut_dir, val_sub, failures)
    if len(trained) != TRIAL_EPOCHS - TRIAL_CUT_AFTER_EPOCH:
        failures.append(f"{kind}: the re-run trained {len(trained)} epochs; expected {TRIAL_EPOCHS - TRIAL_CUT_AFTER_EPOCH} "
                        f"(it should continue from epoch {TRIAL_CUT_AFTER_EPOCH + 1}, not start over)"); ok = False
    print(f"      finished {res_cut['epochs_run']} epochs, best epoch {res_cut['best_epoch']} "
          f"(Phase {res_cut['best_epoch_phase']}), val AUC-ROC {res_cut['best_val_auc']:.4f}  "
          f"{'OK' if ok else 'FAILED -- see summary'}")

    # 4. the same run, uninterrupted (its epoch lines are shown)
    print("  [c] the same 7 epochs uninterrupted (epoch lines shown):")
    res_full = run(full_dir, lambda message: print("      " + message))
    ok_full = _t9_check_finished_fold(kind, res_full, full_dir, val_sub, failures)
    if not ok_full:
        print("      the uninterrupted run failed its checks -- see summary")

    # 5. sizes and save time
    resume_full, best_full = checkpoint_paths(full_dir, fold_num)
    state = torch.load(resume_full, map_location="cpu", weights_only=False)
    timing_file = full_dir / "_timing_probe.pt"
    t0 = time.time()
    _atomic_torch_save(state, str(timing_file))
    save_seconds = time.time() - t0
    timing_file.unlink()
    print(f"  [d] files: resume {_t9_mb(resume_full):.1f} MB (rewritten every epoch; one save took {save_seconds:.2f} s), "
          f"best weights {_t9_mb(best_full):.1f} MB, predictions {Path(res_full['predictions_path']).stat().st_size / 1e3:.0f} KB")

    # comparison, information only
    auc_cut = [h["val_auc_roc"] for h in res_cut["history"]]
    auc_full = [h["val_auc_roc"] for h in res_full["history"]]
    gap = max(abs(a - b) for a, b in zip(auc_cut, auc_full)) if len(auc_cut) == len(auc_full) else float("nan")
    if gap == 0:
        verdict = "IDENTICAL -- the resumed run reproduced the uninterrupted one exactly"
    else:
        verdict = (f"val AUC differs by up to {gap:.2e} per epoch -- normal on a GPU (arithmetic is not bit-exact "
                   f"across a restart); not an error")
    print(f"  [e] resumed vs uninterrupted (information only): {verdict}")
    print(f"      val AUC by epoch (uninterrupted): " + ", ".join(f"{v:.3f}" for v in auc_full))
    print(f"      time for this model: {(time.time() - started) / 60:.1f} min")


def run_real_data_trial(encoder_record, fold_num=0, trial_dir=None, make_base_datasets=None, model_builders=None,
                        criterion_fn=None):
    """
    encoder_record:    the dict from load_encoder_stats() (Module 01).
    fold_num:          the fold to slice (0 is fine).
    trial_dir:         None = TRIAL_DIR. An existing folder is only deleted if it is the default TRIAL_DIR.
    make_base_datasets, model_builders, criterion_fn: leave None (they exist so the trial can be smoke-tested on
                       stand-in data); the real datasets, models and weighted loss are the defaults.
    """
    trial_dir = Path(trial_dir) if trial_dir else TRIAL_DIR
    builders = model_builders or MODEL_BUILDERS
    if criterion_fn is None:
        criterion_fn = lambda fold: build_weighted_criterion(fold, CLASS_TO_IDX)
    failures, notes = [], []

    print("=" * 60)
    print(f"REAL-DATA TRIAL -- fold {fold_num}, both models, {TRIAL_EPOCHS} epochs on a small slice")
    print("=" * 60)
    if DEVICE.type != "cuda":
        print("  NO GPU FOUND -- the trial will run on CPU, slowly.")
        notes.append("no GPU was available, so this trial did not exercise the GPU path")

    if trial_dir.exists():
        if trial_dir != TRIAL_DIR:
            raise ValueError(f"{trial_dir} already exists and is not the default trial folder; refusing to delete it. "
                             f"Choose a folder that does not exist yet.")
        shutil.rmtree(trial_dir)                               # left over from an earlier, interrupted trial
    trial_dir.mkdir(parents=True)

    started = time.time()
    try:
        if make_base_datasets is None:
            fold_table = load_fold_table_with_metadata()
            base_train = HAM10000MultimodalFoldDataset(fold_num, "train", encoder_record, fold_table=fold_table)
            base_val = HAM10000MultimodalFoldDataset(fold_num, "val", encoder_record, fold_table=fold_table)
        else:
            base_train, base_val = make_base_datasets(fold_num)
        train_sub = _T9Subset(base_train, _t9_stratified_indices(base_train.labels, TRIAL_TRAIN_PER_CLASS, BATCH_SIZE))
        val_sub = _T9Subset(base_val, _t9_stratified_indices(base_val.labels, TRIAL_VAL_PER_CLASS))
        print(f"  slice: {len(train_sub)} training and {len(val_sub)} validation images "
              f"(per class: {TRIAL_TRAIN_PER_CLASS} / {TRIAL_VAL_PER_CLASS} at most)")
        if len(np.unique(train_sub.labels)) != len(CLASSES) or len(np.unique(val_sub.labels)) != len(CLASSES):
            failures.append("The slice does not contain all 7 classes in both splits")
        else:
            make_datasets = lambda fold: (train_sub, val_sub)
            for kind in MODEL_KINDS:
                try:
                    _t9_run_one_model(kind, fold_num, trial_dir, make_datasets, builders, criterion_fn, val_sub, failures)
                except Exception as exc:                        # noqa: BLE001 -- reported, not hidden
                    failures.append(f"{kind}: stopped with {type(exc).__name__}: {exc}")
                    print(f"  FAILED -- {type(exc).__name__}: {exc}")
    finally:
        try:
            shutil.rmtree(trial_dir)
            print(f"\n  trial folder deleted: {trial_dir}")
        except OSError as exc:
            notes.append(f"the trial folder could not be deleted ({exc}); delete it by hand: {trial_dir}")

    print("\n" + "=" * 60)
    print(f"total time: {(time.time() - started) / 60:.1f} min")
    for note in notes:
        print(f"NOTE: {note}")
    if failures:
        print(f"TRIAL FAILED -- {len(failures)} issue(s):")
        for msg in failures:
            print(f"  - {msg}")
        print("=" * 60)
        raise RuntimeError(f"Real-data trial failed with {len(failures)} issue(s).")
    print("TRIAL PASSED -- train_fold works end to end on the real data, and a power cut resumes correctly.")
    print("Next: run_cv_training(encoder_record) for the full run.")
    print("=" * 60)


# --- Usage ---
# encoder_record = load_encoder_stats()      # only needed after a kernel restart
# run_real_data_trial(encoder_record)

In [38]:
run_real_data_trial(encoder_record)

REAL-DATA TRIAL -- fold 0, both models, 7 epochs on a small slice
  slice: 322 training and 147 validation images (per class: 46 / 21 at most)

--- multimodal model ---
  [a] training, with a simulated power cut right after epoch 3...
      resume file left behind: epoch 3, Phase A, not finished  OK
  [b] re-running: must resume after epoch 3 and finish...
      finished 7 epochs, best epoch 6 (Phase B), val AUC-ROC 0.6879  OK
  [c] the same 7 epochs uninterrupted (epoch lines shown):
      [multimodal | fold 0] starting fresh
      [multimodal | fold 0] epoch   1 [A] loss 1.8712 | val AUC 0.6484 (best 0.6484 @ 1) | patience 0/10
      [multimodal | fold 0] epoch   2 [A] loss 1.7335 | val AUC 0.6731 (best 0.6731 @ 2) | patience 0/10
      [multimodal | fold 0] epoch   3 [A] loss 1.4794 | val AUC 0.6693 (best 0.6731 @ 2) | patience 0/10
      [multimodal | fold 0] epoch   4 [A] loss 1.2742 | val AUC 0.6867 (best 0.6867 @ 4) | patience 0/10
      [multimodal | fold 0] epoch   5 [A] loss 

In [39]:
run_cv_training(encoder_record)

CV TRAINING -- folds [0, 1, 2, 3, 4], both models, fold by fold
run_config.json created: T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\cv_training\run_config.json
Training status -- T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\cv_training
  fold 0 | multimodal | not started
  fold 0 | ablation   | not started
  fold 1 | multimodal | not started
  fold 1 | ablation   | not started
  fold 2 | multimodal | not started
  fold 2 | ablation   | not started
  fold 3 | multimodal | not started
  fold 3 | ablation   | not started
  fold 4 | multimodal | not started
  fold 4 | ablation   | not started
[multimodal | fold 0] starting fresh
[multimodal | fold 0] epoch   1 [A] loss 1.4799 | val AUC 0.7310 (best 0.7310 @ 1) | patience 0/10
[multimodal | fold 0] epoch   2 [A] loss 1.3333 | val AUC 0.7585 (best 0.7585 @ 2) | patience 0/10
[multimodal | fold 0] epoch   3 [A] loss 1.2538 | val AUC 0.7422 (best 0.7585 @ 2) | patience 0/10
[multimodal | fold 0] epoch   4 [A] loss 1.2

{'per_fold': {'multimodal': {'0': {'metrics': {'auc_roc': 0.9266232633190009,
     'f1': 0.6247901946376393,
     'sensitivity': 0.6936111216767481,
     'specificity': 0.9472573486235666,
     'accuracy': 0.6895748398369249},
    'best_epoch': 26,
    'best_epoch_phase': 'B',
    'best_val_auc_roc': 0.9266232633190009,
    'epochs_run': 36,
    'stop_reason': 'early_stop',
    'predictions_file': 'fold_0_val_predictions.npz'},
   '1': {'metrics': {'auc_roc': 0.9443881461545383,
     'f1': 0.7036121879642879,
     'sensitivity': 0.7277515071723115,
     'specificity': 0.9621084044305146,
     'accuracy': 0.798951048951049},
    'best_epoch': 61,
    'best_epoch_phase': 'B',
    'best_val_auc_roc': 0.9443881461545383,
    'epochs_run': 71,
    'stop_reason': 'early_stop',
    'predictions_file': 'fold_1_val_predictions.npz'},
   '2': {'metrics': {'auc_roc': 0.9264344588313859,
     'f1': 0.599210469089969,
     'sensitivity': 0.660882307809974,
     'specificity': 0.9494074553737492,
  

# Module 10

In [47]:
"""
10_00_collect_results.txt
Phase 3 -- Module 10: Results, Selection and Exit Gate -- collect the results (reusable)

Requires these cells already run: 01_00, 02_00, 06_00, 07_00, 09_00
(after a kernel restart, run the earlier modules' reusable cells again first).

  collect_results()   reads run_config.json and the fold_N_val_predictions.npz files of every finished fold in
                      TRAINING_DIR, checks each predictions file (shape, probabilities, labels, class order, best
                      epoch), and builds five tables:

      per_fold            one row per model and fold: the five metrics, best epoch (and its phase), epochs run,
                          stop reason, and whether the recorded metrics reproduce from the predictions file
      mean_sd             mean +/- SD (ddof=1) across folds per model and metric, plus the paired
                          multimodal - ablation AUC-ROC difference
      paired              per fold: multimodal AUC-ROC, ablation AUC-ROC, difference
      per_class           one row per model, fold and class: support, one-vs-rest AUC-ROC, sensitivity, specificity
      per_class_mean_sd   the per-class numbers summarised across folds

      The tables are printed compactly and saved as CSV files in RESULTS_SUMMARY_DIR (atomic writes).
      Works on a PARTIAL run too (whatever folds are finished); `complete` says whether everything is there.

  It reads only what the run left on disk. It trains nothing, loads no model and needs no GPU.

NAMES  (placeholders -- rename freely; everything below follows these variables)
  RESULTS_SUMMARY_DIR / FINAL_MODEL_DIR   the two output folders (FINAL_MODEL_DIR is used by 10_01 and 10_02)
  *_CSV                                   the CSV file names
  COLUMN_NAMES                            internal column name -> the name written to the CSV files and returned
                                          in the tables. Rename any value here; the code never uses the output names.

HOW THE PER-CLASS NUMBERS RELATE TO THE FOLD METRICS
  The fold metrics are macro averages over the 7 classes, so the per-class values average back to them:
  mean of per-class AUC-ROC = the fold's AUC-ROC (macro one-vs-rest), mean of per-class sensitivity = its
  sensitivity, mean of per-class specificity = its specificity. The verification cell checks exactly that.
"""

import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# --- Names (placeholders -- rename freely) ---------------------------------
RESULTS_SUMMARY_DIR = PHASE3_ARTIFACTS_DIR / "results_summary"
FINAL_MODEL_DIR = PHASE3_ARTIFACTS_DIR / "final_model"

PER_FOLD_CSV = "per_fold_metrics.csv"
MEAN_SD_CSV = "mean_sd_summary.csv"
PAIRED_CSV = "paired_auc_roc_difference.csv"
PER_CLASS_CSV = "per_class_metrics.csv"
PER_CLASS_MEAN_SD_CSV = "per_class_mean_sd.csv"

PAIRED_LABEL = "multimodal_minus_ablation"     # the `model` value of the paired-difference row in mean_sd

# internal name -> name in the saved / returned tables. Change the VALUES freely (keep them unique).
COLUMN_NAMES = {
    "model": "model", "fold": "fold", "class": "class",
    "auc_roc": "auc_roc", "f1": "f1", "sensitivity": "sensitivity", "specificity": "specificity",
    "accuracy": "accuracy",
    "best_epoch": "best_epoch", "best_epoch_phase": "best_epoch_phase", "epochs_run": "epochs_run",
    "stop_reason": "stop_reason", "metrics_reproduce": "metrics_reproduce",
    "support": "support", "support_total": "support_total",
    "metric": "metric", "mean": "mean", "sd": "sd", "n_folds": "n_folds",
    "multimodal_auc_roc": "multimodal_auc_roc", "ablation_auc_roc": "ablation_auc_roc", "difference": "difference",
}

METRIC_KEYS = ("auc_roc", "f1", "sensitivity", "specificity", "accuracy")
PER_CLASS_KEYS = ("auc_roc", "sensitivity", "specificity")
REPRODUCE_TOLERANCE = 1e-9          # recorded metrics vs the same metrics recomputed from the predictions file
PREDICTION_KEYS = ("image_ids", "labels", "probs", "best_epoch", "class_names")


class ResultsError(RuntimeError):
    """A results file is missing, damaged or inconsistent with run_config.json."""


# ---------------------------------------------------------------------------
# Reading and checking the files the run left behind
# ---------------------------------------------------------------------------

def load_predictions(path, expected_best_epoch=None):
    """Load and check one fold_N_val_predictions.npz. Returns {"image_ids", "labels", "probs", "best_epoch"}.
    Raises ResultsError (naming the file) for anything that is not as train_fold (Module 07) wrote it."""
    path = Path(path)
    if not path.exists():
        raise ResultsError(f"Predictions file not found: {path}")
    try:
        with np.load(path, allow_pickle=False) as data:
            missing = [k for k in PREDICTION_KEYS if k not in data.files]
            if missing:
                raise ResultsError(f"{path} is missing {missing}")
            image_ids = [str(x) for x in data["image_ids"]]
            labels = np.asarray(data["labels"])
            probs = np.asarray(data["probs"])
            best_epoch = int(data["best_epoch"])
            class_names = [str(x) for x in data["class_names"]]
    except ResultsError:
        raise
    except Exception as exc:                                     # noqa: BLE001 -- any read failure is reported with the path
        raise ResultsError(f"{path} could not be read ({type(exc).__name__}: {exc})") from exc

    if class_names != list(CLASSES):
        raise ResultsError(f"{path}: class order {class_names} differs from CLASSES {list(CLASSES)}")
    if probs.ndim != 2 or probs.shape[1] != len(CLASSES) or probs.shape[0] == 0:
        raise ResultsError(f"{path}: probs has shape {probs.shape}, expected (N, {len(CLASSES)}) with N > 0")
    n = probs.shape[0]
    if labels.shape != (n,) or labels.dtype.kind not in "iu":
        raise ResultsError(f"{path}: labels must be {n} integers, got shape {labels.shape}, dtype {labels.dtype}")
    if labels.min() < 0 or labels.max() >= len(CLASSES):
        raise ResultsError(f"{path}: labels outside 0..{len(CLASSES) - 1}")
    if len(image_ids) != n or len(set(image_ids)) != n:
        raise ResultsError(f"{path}: image_ids must be {n} distinct ids (got {len(image_ids)}, {len(set(image_ids))} distinct)")
    if not np.isfinite(probs).all() or probs.min() < -1e-6 or probs.max() > 1 + 1e-6:
        raise ResultsError(f"{path}: probs contains non-finite values or values outside [0, 1]")
    if not np.allclose(probs.sum(axis=1), 1.0, atol=1e-3):
        raise ResultsError(f"{path}: probability rows do not sum to 1 (they should be softmax outputs)")
    if expected_best_epoch is not None and best_epoch != expected_best_epoch:
        raise ResultsError(f"{path}: best_epoch {best_epoch} in the file, {expected_best_epoch} recorded in run_config.json")
    return {"image_ids": image_ids, "labels": labels.astype(np.int64), "probs": probs, "best_epoch": best_epoch}


def _check_record(kind, fold_str, record):
    """One entry of run_config.json results.per_fold[kind][fold] must hold everything the tables need."""
    where = f"run_config.json, {kind} fold {fold_str}"
    if not str(fold_str).isdigit() or not 0 <= int(fold_str) < N_FOLDS:
        raise ResultsError(f"{where}: fold key must be 0..{N_FOLDS - 1}")
    needed = ("metrics", "best_epoch", "best_epoch_phase", "best_val_auc_roc", "epochs_run", "stop_reason", "predictions_file")
    missing = [k for k in needed if k not in record]
    if missing or any(k not in record["metrics"] for k in METRIC_KEYS):
        raise ResultsError(f"{where}: entry is incomplete (missing {missing or 'metrics'})")
    expected_name = f"fold_{int(fold_str)}_val_predictions.npz"
    if record["predictions_file"] != expected_name:
        raise ResultsError(f"{where}: predictions_file is {record['predictions_file']!r}, expected {expected_name!r}")


def load_run_results(training_dir=None):
    """(run_config dict, {model_kind: {fold: predictions}}) for every finished fold recorded in run_config.json."""
    training_dir = Path(training_dir) if training_dir else TRAINING_DIR
    config = _read_config(run_config_path(training_dir))
    recorded = config["results"]["per_fold"]
    predictions = {kind: {} for kind in MODEL_KINDS}
    for kind in MODEL_KINDS:
        for fold_str in sorted(recorded.get(kind, {}), key=lambda k: int(k) if str(k).isdigit() else -1):
            record = recorded[kind][fold_str]
            _check_record(kind, fold_str, record)
            predictions[kind][int(fold_str)] = load_predictions(
                model_dir(training_dir, kind) / record["predictions_file"], expected_best_epoch=record["best_epoch"])
    return config, predictions


# ---------------------------------------------------------------------------
# Tables
# ---------------------------------------------------------------------------

def per_class_metrics(labels, probs):
    """One-vs-rest AUC-ROC, sensitivity and specificity (argmax decision) for each class, in CLASSES order.
    A class with no positives (or no negatives) gives NaN -- it never crashes."""
    labels, probs = np.asarray(labels), np.asarray(probs)
    predicted = np.argmax(probs, axis=1)
    rows = []
    for c, name in enumerate(CLASSES):
        is_c = labels == c
        n_pos, n_neg = int(is_c.sum()), int((~is_c).sum())
        rows.append({
            "class": name,
            "support": n_pos,
            "auc_roc": float(roc_auc_score(is_c.astype(int), probs[:, c])) if n_pos and n_neg else float("nan"),
            "sensitivity": float((predicted[is_c] == c).mean()) if n_pos else float("nan"),
            "specificity": float((predicted[~is_c] != c).mean()) if n_neg else float("nan"),
        })
    return rows


def _mean_sd(values):
    vals = np.array([v for v in values if np.isfinite(v)], dtype=np.float64)
    if len(vals) == 0:
        return float("nan"), float("nan"), 0
    return float(vals.mean()), (float(np.std(vals, ddof=1)) if len(vals) >= 2 else float("nan")), int(len(vals))


def build_tables(config, predictions):
    """The five tables, with INTERNAL column names (collect_results renames them on the way out)."""
    recorded = config["results"]["per_fold"]
    fold_rows, class_rows = [], []
    for kind in MODEL_KINDS:
        for fold_str in sorted(recorded.get(kind, {}), key=int):
            rec, fold = recorded[kind][fold_str], int(fold_str)
            pred = predictions[kind][fold]
            again = compute_fold_metrics(pred["labels"], pred["probs"], n_classes=len(CLASSES))
            reproduce = all(abs(again[k] - rec["metrics"][k]) <= REPRODUCE_TOLERANCE for k in METRIC_KEYS)
            fold_rows.append({"model": kind, "fold": fold, **{k: float(rec["metrics"][k]) for k in METRIC_KEYS},
                              "best_epoch": int(rec["best_epoch"]), "best_epoch_phase": rec["best_epoch_phase"],
                              "epochs_run": int(rec["epochs_run"]), "stop_reason": rec["stop_reason"],
                              "metrics_reproduce": bool(reproduce)})
            for row in per_class_metrics(pred["labels"], pred["probs"]):
                class_rows.append({"model": kind, "fold": fold, **row})

    mean_sd_rows = []
    for kind in MODEL_KINDS:
        folds = sorted(recorded.get(kind, {}), key=int)
        if len(folds) >= 2:
            summary = summarize_across_folds([recorded[kind][f]["metrics"] for f in folds])
            for key in METRIC_KEYS:
                mean_sd_rows.append({"model": kind, "metric": key, "mean": summary[key]["mean"],
                                     "sd": summary[key]["sd"], "n_folds": len(folds)})

    paired_rows = []
    for f in sorted(set(recorded.get("multimodal", {})) & set(recorded.get("ablation", {})), key=int):
        mm = float(recorded["multimodal"][f]["metrics"]["auc_roc"])
        ab = float(recorded["ablation"][f]["metrics"]["auc_roc"])
        paired_rows.append({"fold": int(f), "multimodal_auc_roc": mm, "ablation_auc_roc": ab, "difference": mm - ab})
    if len(paired_rows) >= 2:
        diffs = [r["difference"] for r in paired_rows]
        mean_sd_rows.append({"model": PAIRED_LABEL, "metric": "auc_roc", "mean": float(np.mean(diffs)),
                             "sd": float(np.std(diffs, ddof=1)), "n_folds": len(diffs)})

    per_class = pd.DataFrame(class_rows, columns=["model", "fold", "class", "support", *PER_CLASS_KEYS])
    class_summary_rows = []
    for kind in MODEL_KINDS:
        for name in CLASSES:
            part = per_class[(per_class["model"] == kind) & (per_class["class"] == name)]
            if len(part) < 2:
                continue
            for key in PER_CLASS_KEYS:
                mean, sd, n = _mean_sd(part[key].tolist())
                class_summary_rows.append({"model": kind, "class": name, "support_total": int(part["support"].sum()),
                                           "metric": key, "mean": mean, "sd": sd, "n_folds": n})

    return {
        "per_fold": pd.DataFrame(fold_rows, columns=["model", "fold", *METRIC_KEYS, "best_epoch", "best_epoch_phase",
                                                     "epochs_run", "stop_reason", "metrics_reproduce"]),
        "mean_sd": pd.DataFrame(mean_sd_rows, columns=["model", "metric", "mean", "sd", "n_folds"]),
        "paired": pd.DataFrame(paired_rows, columns=["fold", "multimodal_auc_roc", "ablation_auc_roc", "difference"]),
        "per_class": per_class,
        "per_class_mean_sd": pd.DataFrame(class_summary_rows, columns=["model", "class", "support_total", "metric",
                                                                       "mean", "sd", "n_folds"]),
    }


# ---------------------------------------------------------------------------
# Saving and printing
# ---------------------------------------------------------------------------

def _atomic_write_csv(df, path):
    """Temporary file + fsync + retrying os.replace -- the same pattern as every other write in Phase 3."""
    path = str(path)
    folder = os.path.dirname(path) or "."
    os.makedirs(folder, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=folder, suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="") as f:
            df.to_csv(f, index=False)
            f.flush()
            os.fsync(f.fileno())
        _replace_with_retry(tmp_path, path)
    except BaseException:
        _remove_quietly(tmp_path)
        raise


def _ms(mean, sd):
    return f"{mean:.4f} +/- {sd:.4f}" if np.isfinite(sd) else f"{mean:.4f}"


def print_result_tables(tables, complete):
    pf, ms, paired, pcs = tables["per_fold"], tables["mean_sd"], tables["paired"], tables["per_class_mean_sd"]
    print("\n" + "=" * 92)
    print("PER-FOLD RESULTS (validation folds, best-epoch weights)")
    print("=" * 92)
    print(f"  {'model':<11}{'fold':>4}  {'AUC-ROC':>8}{'F1':>8}{'Sens':>8}{'Spec':>8}{'Acc':>8}  {'best ep':>8}  {'epochs':>6}  stop")
    for _, r in pf.iterrows():
        flag = "" if r["metrics_reproduce"] else "   <-- metrics do NOT reproduce from the predictions file"
        print(f"  {r['model']:<11}{int(r['fold']):>4}  {r['auc_roc']:>8.4f}{r['f1']:>8.4f}{r['sensitivity']:>8.4f}"
              f"{r['specificity']:>8.4f}{r['accuracy']:>8.4f}  {str(int(r['best_epoch'])) + '(' + r['best_epoch_phase'] + ')':>8}"
              f"  {int(r['epochs_run']):>6}  {r['stop_reason']}{flag}")

    print("\nMEAN +/- SD ACROSS FOLDS")
    labels = {"auc_roc": "AUC-ROC", "f1": "F1", "sensitivity": "Sensitivity", "specificity": "Specificity", "accuracy": "Accuracy"}
    for kind in MODEL_KINDS:
        part = ms[ms["model"] == kind]
        if part.empty:
            print(f"  {kind:<11}: fewer than 2 finished folds -- no mean +/- SD yet")
            continue
        cells = " | ".join(f"{labels[r['metric']]} {_ms(r['mean'], r['sd'])}" for _, r in part.iterrows())
        print(f"  {kind:<11}({int(part['n_folds'].iloc[0])} folds): {cells}")
    if not paired.empty:
        per_fold = ", ".join(f"fold {int(r['fold'])}: {r['difference']:+.4f}" for _, r in paired.iterrows())
        line = f"  AUC-ROC difference (multimodal - ablation): {per_fold}"
        row = ms[ms["model"] == PAIRED_LABEL]
        if not row.empty:
            line += f" | mean {row['mean'].iloc[0]:+.4f} +/- {row['sd'].iloc[0]:.4f}"
        line += f" | multimodal higher in {int((paired['difference'] > 0).sum())} of {len(paired)} folds"
        print(line)

    if not pcs.empty:
        print("\nPER-CLASS (one-vs-rest), mean +/- SD across folds; n = images of that class over all finished folds")
        for key, title in (("auc_roc", "AUC-ROC"), ("sensitivity", "Sensitivity"), ("specificity", "Specificity")):
            print(f"  {title}")
            print(f"    {'class':<7}{'n':>6}  {'multimodal':>17}  {'ablation':>17}  {'difference':>10}")
            for name in CLASSES:
                cells, means, n_total = {}, {}, 0
                for kind in MODEL_KINDS:
                    row = pcs[(pcs["model"] == kind) & (pcs["class"] == name) & (pcs["metric"] == key)]
                    if not row.empty:
                        cells[kind], means[kind] = _ms(row["mean"].iloc[0], row["sd"].iloc[0]), row["mean"].iloc[0]
                        n_total = int(row["support_total"].iloc[0])
                diff = f"{means['multimodal'] - means['ablation']:+.4f}" if len(means) == 2 else "-"
                print(f"    {name:<7}{n_total:>6}  {cells.get('multimodal', '-'):>17}  {cells.get('ablation', '-'):>17}  {diff:>10}")
    print("\n  " + ("COMPLETE: all folds of both models are finished." if complete
                    else "NOT complete: some folds are not finished yet (tables show what is finished)."))
    print("=" * 92)


def collect_results(training_dir=None, save=True, out_dir=None, print_tables=True):
    """Read the run, check it, build the five tables. Returns a dict:
        {"per_fold", "mean_sd", "paired", "per_class", "per_class_mean_sd"}  (DataFrames, COLUMN_NAMES applied),
        "complete": bool, "folds_finished": {model: [fold, ...]}, "saved_to": folder or None.
    training_dir: None = TRAINING_DIR.   out_dir: None = RESULTS_SUMMARY_DIR.   save=False writes nothing."""
    training_dir = Path(training_dir) if training_dir else TRAINING_DIR
    out_dir = Path(out_dir) if out_dir else RESULTS_SUMMARY_DIR
    config, predictions = load_run_results(training_dir)
    tables = build_tables(config, predictions)
    folds_finished = {kind: sorted(predictions[kind]) for kind in MODEL_KINDS}
    complete = all(len(folds_finished[kind]) == N_FOLDS for kind in MODEL_KINDS)

    if print_tables:
        print_result_tables(tables, complete)
    if not tables["per_fold"]["metrics_reproduce"].all():
        print("WARNING: at least one fold's recorded metrics do not reproduce from its predictions file -- see the table above.")

    out = {name: df.rename(columns=COLUMN_NAMES) for name, df in tables.items()}
    saved_to = None
    if save:
        for name, filename in (("per_fold", PER_FOLD_CSV), ("mean_sd", MEAN_SD_CSV), ("paired", PAIRED_CSV),
                               ("per_class", PER_CLASS_CSV), ("per_class_mean_sd", PER_CLASS_MEAN_SD_CSV)):
            _atomic_write_csv(out[name], out_dir / filename)
        saved_to = str(out_dir)
        print(f"Tables saved (CSV) in: {out_dir}")
    out.update({"complete": complete, "folds_finished": folds_finished, "saved_to": saved_to})
    return out


print("Results collection loaded.")
print(f"  collect_results()   -- tables from the finished folds in {TRAINING_DIR}; CSVs go to {RESULTS_SUMMARY_DIR}")
print(f"  final model folder  -- {FINAL_MODEL_DIR}   (used by 10_01 and 10_02)")

Results collection loaded.
  collect_results()   -- tables from the finished folds in T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\cv_training; CSVs go to T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\results_summary
  final model folder  -- T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\final_model   (used by 10_01 and 10_02)


In [48]:
"""
10_01_select_and_freeze.txt
Phase 3 -- Module 10: Results, Selection and Exit Gate -- choose the final model and freeze it (reusable)

Requires these cells already run: 01_00, 01_01, 02_00, 02_01, 03_00, 03_01, 03_02, 06_00, 07_00, 07_01, 09_00, 10_00
(after a kernel restart, run the earlier modules' reusable cells again first).

  select_and_freeze()   runs only after the WHOLE run is finished (all 5 folds of both models), then:
      1. SELECT   the fold whose multimodal model has the highest validation AUC-ROC (best-epoch weights).
                  One fold's model -- not an ensemble. An exact tie goes to the lower fold number.
      2. FREEZE   copies that fold's fold_N_best.pt, byte for byte, to
                      <FINAL_MODEL_DIR>/phase3_final_multimodal_model.pt
                  and proves the copy with SHA-256. Before copying it checks that the source file really is
                  that fold's multimodal model (identity, best epoch and AUC all match run_config.json).
                  An identical file already there is fine (nothing is copied). A DIFFERENT file already
                  there is never overwritten: the call stops and says so.
      3. RELOAD   builds the model fresh, loads the FROZEN file, predicts the selected fold's validation images
                  again, and compares with what training recorded (AUC-ROC within 1e-4; every probability within
                  1e-2 of the saved predictions; same images, same labels, same order).
      4. CARD     writes <FINAL_MODEL_DIR>/phase3_model_card.json -- everything a later phase needs to use this
                  model without guessing (see below).

MODEL CARD  (phase3_model_card.json) -- rename any key in build_model_card() if you want
  format_version, frozen_at, last_checked_at, description,
  model          class, architecture, call convention, output, how to build and load it, Grad-CAM layer
  selection      rule, tie rule, selected fold, its validation AUC-ROC, best epoch and phase, all five folds'
                 scores, runner-up fold and margin
  frozen_file    name, SHA-256, size in bytes, the file it was copied from
  input_contract image preprocessing (exact steps and ImageNet constants) and metadata encoding: the 20 feature
                 names, sex / site categories, and THIS fold's age mean / std (the ones Phase 5 must reuse for
                 BCN20000 -- never the new dataset's own statistics), class order
  cross_fold_summary   mean +/- SD per model and the paired AUC-ROC difference, as recorded by the run
  training_settings, environment (at training start, changes during training, at freeze time)
  reload_check   the numbers from step 3
  downstream_rule      the model is frozen; later phases only load and use it

NAMES  (placeholders -- rename freely) are the variables right below; FINAL_MODEL_DIR comes from 10_00.
The reload check takes its dataset and model builder as arguments (defaults: the real ones) so it can be tested
on stand-in data.
"""

import hashlib
import json
import os
import shutil
import tempfile
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

FROZEN_MODEL_FILENAME = "phase3_final_multimodal_model.pt"
MODEL_CARD_FILENAME = "phase3_model_card.json"
MODEL_CARD_FORMAT_VERSION = 1

SELECTION_RULE = ("The single fold whose multimodal model has the highest validation AUC-ROC "
                  "(best-epoch weights). One fold's model, not an ensemble of folds.")
TIE_RULE = "exact tie in validation AUC-ROC -> the lower fold number"
DOWNSTREAM_RULE = ("FROZEN. This file and the preprocessing in input_contract are fixed. Nothing in Phase 4 (XAI) or "
                   "Phase 5 (external validation) may feed back into this model: no different fold, no retraining, "
                   "no fine-tuning, no edited weights. Later phases only load it (check the SHA-256 first) and use it.")

RELOAD_AUC_TOLERANCE = 1e-4         # recomputed vs recorded validation AUC-ROC
RELOAD_PROB_TOLERANCE = 1e-2        # largest allowed difference to any saved probability (bf16 on GPU is not bit-exact across runs)


class FrozenModelError(ResultsError):
    """The frozen model file is missing, different from the selected weights, or fails its reload check."""


# ---------------------------------------------------------------------------
# 1. Select
# ---------------------------------------------------------------------------

def select_best_fold(results):
    """results: the `results` section of run_config.json. Returns the selection record (see the card's
    'selection' block). Refuses unless the whole run is complete."""
    if not results.get("complete"):
        raise ResultsError("The run is not complete (run_config.json results.complete is not true): the final model is "
                           "chosen only after all 5 folds of both models are finished. Finish the run first.")
    per_fold = results["per_fold"].get("multimodal", {})
    if sorted(per_fold, key=int) != [str(f) for f in range(N_FOLDS)]:
        raise ResultsError(f"Multimodal folds recorded: {sorted(per_fold)}; expected all of 0..{N_FOLDS - 1}.")
    scores = {int(f): float(per_fold[f]["metrics"]["auc_roc"]) for f in per_fold}
    best = max(scores.values())
    tied = sorted(f for f, s in scores.items() if s == best)
    selected = tied[0]
    runner_up = max((f for f in scores if f != selected), key=lambda f: (scores[f], -f))
    record = per_fold[str(selected)]
    return {
        "rule": SELECTION_RULE,
        "tie_rule": TIE_RULE,
        "selected_fold": selected,
        "val_auc_roc": best,
        "best_epoch": int(record["best_epoch"]),
        "best_epoch_phase": record["best_epoch_phase"],
        "all_fold_val_auc_roc": {str(f): scores[f] for f in sorted(scores)},
        "tied_folds": tied,
        "runner_up_fold": runner_up,
        "margin_to_runner_up": best - scores[runner_up],
    }


# ---------------------------------------------------------------------------
# 2. Freeze
# ---------------------------------------------------------------------------

def sha256_of_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _copy_atomic(source, destination):
    """Copy to a temporary file next to the destination, fsync, then an atomic (retrying) rename."""
    destination = str(destination)
    folder = os.path.dirname(destination) or "."
    os.makedirs(folder, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=folder, suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as out, open(source, "rb") as src:
            shutil.copyfileobj(src, out, 1024 * 1024)
            out.flush()
            os.fsync(out.fileno())
        _replace_with_retry(tmp_path, destination)
    except BaseException:
        _remove_quietly(tmp_path)
        raise


def freeze_model(selection, results, training_dir=None, final_dir=None):
    """Copy the selected fold's best weights to FINAL_MODEL_DIR and prove the copy. Returns the 'frozen_file' record."""
    training_dir = Path(training_dir) if training_dir else TRAINING_DIR
    final_dir = Path(final_dir) if final_dir else FINAL_MODEL_DIR
    fold = selection["selected_fold"]
    _, source = checkpoint_paths(model_dir(training_dir, "multimodal"), fold)

    state = _load_checked(source, make_identity("multimodal", fold), "Best-weights")
    if state is None:
        raise FrozenModelError(f"The selected model's file does not exist: {source}")
    record = results["per_fold"]["multimodal"][str(fold)]
    if state["epoch"] != record["best_epoch"] or abs(float(state["val_auc"]) - float(record["best_val_auc_roc"])) > 1e-12:
        raise FrozenModelError(f"{source} holds epoch {state['epoch']} / AUC {state['val_auc']}, but run_config.json records "
                               f"epoch {record['best_epoch']} / AUC {record['best_val_auc_roc']} for multimodal fold {fold}.")

    frozen_path = final_dir / FROZEN_MODEL_FILENAME
    source_hash = sha256_of_file(source)
    if frozen_path.exists():
        existing_hash = sha256_of_file(frozen_path)
        if existing_hash != source_hash:
            raise FrozenModelError(
                f"A DIFFERENT file is already frozen at {frozen_path}\n"
                f"  its SHA-256:                       {existing_hash}\n"
                f"  selected fold {fold}'s best weights: {source_hash}\n"
                f"A frozen model is never overwritten. If the old file is stale and you are sure, delete it yourself "
                f"(or change FINAL_MODEL_DIR) and run this again.")
        status = "already frozen -- an identical file was found, nothing copied"
    else:
        _copy_atomic(source, frozen_path)
        if sha256_of_file(frozen_path) != source_hash:
            _remove_quietly(str(frozen_path))
            raise FrozenModelError(f"The copy at {frozen_path} does not match its source (SHA-256 differs); it was removed.")
        status = "copied"
    return {"name": FROZEN_MODEL_FILENAME, "path": str(frozen_path), "sha256": source_hash,
            "bytes": os.path.getsize(frozen_path), "copied_from": str(source), "status": status}


# ---------------------------------------------------------------------------
# 3. Reload check
# ---------------------------------------------------------------------------

def reload_check(frozen_path, fold, recorded_metrics, predictions_path, encoder_record=None,
                 make_val_dataset=None, build_model_fn=None, device=None, use_amp=None, batch_size=None):
    """Build the model fresh, load the frozen file, predict fold `fold`'s validation images, compare with what
    training recorded. Returns a dict with "passed", "problems" (list of text) and the numbers.
    Defaults are read at CALL time: device = DEVICE, use_amp = USE_AMP, batch_size = BATCH_SIZE."""
    device = device if device is not None else DEVICE
    use_amp = USE_AMP if use_amp is None else use_amp
    batch_size = batch_size or BATCH_SIZE
    if build_model_fn is None:
        build_model_fn = lambda: build_multimodal_model(pretrained=False)
    if make_val_dataset is None:
        encoder_record = encoder_record if encoder_record is not None else load_encoder_stats()
        make_val_dataset = lambda f: HAM10000MultimodalFoldDataset(f, "val", encoder_record)

    problems = []
    saved = load_predictions(predictions_path)
    state = _load_checked(frozen_path, make_identity("multimodal", fold), "Frozen-model")
    if state is None:
        raise FrozenModelError(f"The frozen model file does not exist: {frozen_path}")

    set_all_seeds(RANDOM_SEED)
    model = build_model_fn()
    try:
        model.load_state_dict(state["model_state_dict"])
    except RuntimeError as exc:
        return {"passed": False, "problems": [f"the frozen weights do not fit the model built here: {exc}"]}
    model.to(device)

    val_ds = make_val_dataset(fold)
    loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    probs, labels = predict_split(model, loader, device, use_amp=use_amp)

    ids_match = list(val_ds.image_ids) == saved["image_ids"]
    labels_match = bool(np.array_equal(labels.astype(np.int64), saved["labels"]))
    if not ids_match:
        problems.append("the validation images (or their order) differ from the ones training recorded")
    if not labels_match:
        problems.append("the validation labels differ from the ones training recorded")

    metrics = compute_fold_metrics(labels, probs, n_classes=len(CLASSES))
    metric_differences = {k: float(metrics[k] - recorded_metrics[k]) for k in METRIC_KEYS}
    auc_difference = abs(metric_differences["auc_roc"])
    if auc_difference > RELOAD_AUC_TOLERANCE:
        problems.append(f"AUC-ROC after reloading is {metrics['auc_roc']:.6f}, recorded {recorded_metrics['auc_roc']:.6f} "
                        f"(difference {auc_difference:.2e} > {RELOAD_AUC_TOLERANCE:g})")
    max_prob_difference = None
    if ids_match and probs.shape == saved["probs"].shape:
        max_prob_difference = float(np.abs(probs.astype(np.float64) - saved["probs"].astype(np.float64)).max())
        if max_prob_difference > RELOAD_PROB_TOLERANCE:
            problems.append(f"largest probability difference to the saved predictions is {max_prob_difference:.2e} "
                            f"(> {RELOAD_PROB_TOLERANCE:g})")
    elif ids_match:
        problems.append(f"probability array shape {probs.shape} differs from the saved {saved['probs'].shape}")

    return {"passed": not problems, "problems": problems, "n_rows": int(len(labels)),
            "auc_roc_recomputed": float(metrics["auc_roc"]), "auc_roc_recorded": float(recorded_metrics["auc_roc"]),
            "auc_roc_abs_difference": float(auc_difference), "metric_differences": metric_differences,
            "max_abs_probability_difference": max_prob_difference,
            "image_ids_match": ids_match, "labels_match": labels_match,
            "device": str(device), "use_amp": bool(use_amp), "batch_size": int(batch_size),
            "tolerances": {"auc_roc": RELOAD_AUC_TOLERANCE, "probability": RELOAD_PROB_TOLERANCE}}


# ---------------------------------------------------------------------------
# 4. Model card
# ---------------------------------------------------------------------------

def build_model_card(config, selection, frozen, reload_result, encoder_record, frozen_at=None):
    """The model card dict (see the header for the layout). Rename keys here if you want different ones."""
    fold = selection["selected_fold"]
    age_mean, age_std = get_fold_age_stats(encoder_record, fold)
    fold_stats = encoder_record["folds"][str(fold)]
    results = config["results"]
    now = _now()
    return {
        "format_version": MODEL_CARD_FORMAT_VERSION,
        "frozen_at": frozen_at or now,
        "last_checked_at": now,
        "description": ("Phase 3 final model: EfficientNet-B3 + MetaBlock multimodal classifier for the seven HAM10000 "
                        "classes, the best cross-validation fold's model (chosen by validation AUC-ROC)."),
        "model": {
            "class": "MultimodalNet",
            "architecture": config["settings"]["model"]["multimodal"],
            "call": "model(image, metadata) -> logits of shape (batch, 7); softmax gives the class probabilities",
            "inference_mode": "call model.eval() first (Dropout off, BatchNorm uses its running statistics)",
            "how_to_load": "model = build_multimodal_model(pretrained=False); "
                           "model.load_state_dict(torch.load(file)['model_state_dict'])",
            "checkpoint_contents": "format_version, identity {model_kind, fold}, epoch, val_auc, model_state_dict",
            "grad_cam_layer": "model.backbone[-1]",
        },
        "selection": {**selection, "source_file": frozen["copied_from"]},
        "frozen_file": {"name": frozen["name"], "sha256": frozen["sha256"], "bytes": frozen["bytes"],
                        "copied_from": frozen["copied_from"]},
        "input_contract": {
            "image": {
                "source": "Phase 1 final preprocessed images",
                "size": [300, 300],
                "steps": "cv2.imread (BGR) -> RGB -> divide by 255 -> (x - mean) / std per channel -> channels first "
                         "(3, 300, 300), float32. No augmentation at inference.",
                "imagenet_mean": [round(float(x), 4) for x in IMAGENET_MEAN],
                "imagenet_std": [round(float(x), 4) for x in IMAGENET_STD],
            },
            "metadata": {
                "dim": METADATA_DIM,
                "feature_names": list(FEATURE_NAMES),
                "sex_categories": list(SEX_CATEGORIES),
                "site_categories": list(SITE_CATEGORIES),
                "age": {"mean": age_mean, "std": age_std, "std_ddof": encoder_record.get("age_std_ddof"),
                        "n_recorded": fold_stats.get("n_recorded"), "n_train_images": fold_stats.get("n_train_images"),
                        "rule": f"age_z = (age - mean) / std, with the mean and std of fold {fold}'s TRAINING rows; "
                                f"reuse these values for any other dataset (never that dataset's own statistics)",
                        "stats_file": str(ENCODER_STATS_PATH)},
                "age_missing_rule": encoder_record.get("age_missing_rule"),
                "encoder": "Module 01 encode_metadata: float32 vector in feature_names order",
            },
            "classes_output_order": list(CLASSES),
        },
        "cross_fold_summary": {"multimodal": results["mean_sd"].get("multimodal"),
                               "ablation": results["mean_sd"].get("ablation"),
                               "paired_auc_roc_difference": results["paired_auc_roc_difference"]},
        "training_settings": config["settings"],
        "environment": {"at_training_start": config["environment"],
                        "changes_during_training": config.get("environment_changes", []),
                        "at_freeze_time": capture_environment()},
        "reload_check": reload_result,
        "downstream_rule": DOWNSTREAM_RULE,
    }


# ---------------------------------------------------------------------------
# All four steps
# ---------------------------------------------------------------------------

def select_and_freeze(encoder_record=None, training_dir=None, final_dir=None,
                      make_val_dataset=None, build_model_fn=None):
    """Select, freeze, reload-check and write the model card. Returns
    {"selection", "frozen", "reload_check", "model_card_path"}. Safe to run again (see the header)."""
    training_dir = Path(training_dir) if training_dir else TRAINING_DIR
    final_dir = Path(final_dir) if final_dir else FINAL_MODEL_DIR
    config = _read_config(run_config_path(training_dir))
    results = config["results"]

    print("=" * 60)
    print("SELECT AND FREEZE THE FINAL MODEL")
    print("=" * 60)
    selection = select_best_fold(results)
    fold = selection["selected_fold"]
    scores = "  ".join(f"fold {f}: {s:.4f}" for f, s in selection["all_fold_val_auc_roc"].items())
    print(f"  validation AUC-ROC per fold (multimodal): {scores}")
    print(f"  SELECTED: fold {fold}  (AUC-ROC {selection['val_auc_roc']:.4f}, best epoch {selection['best_epoch']} "
          f"[phase {selection['best_epoch_phase']}]; runner-up fold {selection['runner_up_fold']}, "
          f"margin {selection['margin_to_runner_up']:.4f})")
    if len(selection["tied_folds"]) > 1:
        print(f"  NOTE: folds {selection['tied_folds']} tie exactly; the lower number wins.")

    frozen = freeze_model(selection, results, training_dir, final_dir)
    print(f"  FROZEN: {frozen['status']}")
    print(f"    file    {frozen['path']}  ({frozen['bytes'] / 1e6:.1f} MB)")
    print(f"    SHA-256 {frozen['sha256']}")

    if encoder_record is None:
        encoder_record = load_encoder_stats()
    record = results["per_fold"]["multimodal"][str(fold)]
    print("  Reload check (fresh model + the frozen file, predicting the selected fold's validation images)...")
    reload_result = reload_check(frozen["path"], fold, record["metrics"],
                                 model_dir(training_dir, "multimodal") / record["predictions_file"],
                                 encoder_record=encoder_record, make_val_dataset=make_val_dataset,
                                 build_model_fn=build_model_fn)
    if not reload_result["passed"]:
        raise FrozenModelError("The reload check FAILED -- the model card was not written:\n  - " +
                               "\n  - ".join(reload_result["problems"]))
    print(f"    OK -- AUC-ROC {reload_result['auc_roc_recomputed']:.6f} (recorded {reload_result['auc_roc_recorded']:.6f}), "
          f"largest probability difference {reload_result['max_abs_probability_difference']:.2e}, "
          f"{reload_result['n_rows']} images, same ids and labels")

    card_path = final_dir / MODEL_CARD_FILENAME
    frozen_at = None
    if card_path.exists():                                   # a re-run of the same freeze keeps the original date
        try:
            with open(card_path, "r", encoding="utf-8") as f:
                old = json.load(f)
            if old.get("frozen_file", {}).get("sha256") == frozen["sha256"]:
                frozen_at = old.get("frozen_at")
        except (OSError, ValueError):
            pass
    card = build_model_card(config, selection, frozen, reload_result, encoder_record, frozen_at=frozen_at)
    _atomic_write_json(card_path, card)
    print(f"  MODEL CARD written: {card_path}")
    print("=" * 60)
    print("DONE. Next: run_final_gate() (10_02) checks the whole phase against the Checklist.")
    print("=" * 60)
    return {"selection": selection, "frozen": frozen, "reload_check": reload_result, "model_card_path": str(card_path)}


print("Select-and-freeze loaded.")
print("  select_and_freeze()   -- pick the best fold, freeze its weights, reload-check them, write the model card")
print(f"  frozen file: {FINAL_MODEL_DIR / FROZEN_MODEL_FILENAME}")
print(f"  model card : {FINAL_MODEL_DIR / MODEL_CARD_FILENAME}")

Select-and-freeze loaded.
  select_and_freeze()   -- pick the best fold, freeze its weights, reload-check them, write the model card
  frozen file: T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\final_model\phase3_final_multimodal_model.pt
  model card : T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\final_model\phase3_model_card.json


In [49]:
"""
10_02_final_gate.txt
Phase 3 -- Module 10: Results, Selection and Exit Gate -- the final gate (reusable)

Requires these cells already run: 01_00, 01_01, 02_00, 02_01, 03_00, 03_01, 03_02, 04_00, 04_01, 05_00, 06_00,
07_00, 07_01, 09_00, 10_00, 10_01   (after a kernel restart, run the earlier modules' reusable cells again first).

  run_final_gate()   checks the finished phase against the locked plan. Every check READS THE FILES ON DISK; nothing
                     is taken from memory or from earlier printed output. It writes
                         <FINAL_MODEL_DIR>/phase3_exit_gate.json
                     (every check with its status and detail, the BatchNorm evidence, the documented-deviations
                     list) and raises an error at the end if any check failed.

  The gate checks PROCESS, not performance: it does not ask whether the model is good enough. A weak fold or an
  unusual training curve cannot fail it; a missing file, a file that was changed, or a setting that drifted can.

CHECKS  (status: PASS / FAIL / NOTE = passes, with something to write up / SKIPPED = could not run)
  A  The run is complete and intact
     A1 run_config.json is readable (and in the format Module 09 wrote)
     A2 all 10 runs (5 folds x 2 models) are recorded, and run_config.json says complete
     A3 every run's resume file is readable, marked finished
     A4 every best-weights file matches the epoch and AUC-ROC recorded for it
     A5 every predictions file loads and passes Module 10_00's checks
     A6 every recorded metric reproduces from its predictions file
     A7 each model's validation folds together cover the CV pool exactly once (no image twice or missing)
     A8 the multimodal and ablation predictions of each fold are for the same images, same order, same labels
     A9 the mean +/- SD and the paired difference in run_config.json equal a fresh recomputation
  B  The settings were the locked ones
     B1 settings recorded in run_config.json equal the values locked in the Checklist / planning (LOCKED_SETTINGS)
     B2 the CV pool size recorded is the Phase 1 pool
     B3 maximum epochs: the Checklist's 50 passes; 100 (as Phase 2) is a NOTE; anything else FAILS
     B4 software environment unchanged during training (a change is a NOTE, recorded in the card)
  C  The selection and the frozen model
     C1 model card readable; C2 its five scores equal run_config's; C3 the selected fold is the best one
     C4 frozen file = card's SHA-256 = the selected fold's best weights; C5 its identity, epoch and AUC match
     C6 reload check passes now (fresh model + frozen file, selected fold's validation images)
     C7 the card's metadata encoding (age mean / std, feature names, categories) equals metadata_encoder_stats.json
     C8 the card records the class order, the image contract and the downstream rule "FROZEN ..."
  D  BatchNorm freeze evidence
     D1 live check with the real set_phase + train_one_epoch on small random inputs (CPU): in Phase A the backbone's
        BatchNorm statistics and weights do not move while the head group learns; in Phase B they move
     D2 each run's saved per-epoch history follows the schedule: epochs 1..n without gaps, 1-5 Phase A, 6+ Phase B
        (what the past run actually did; D1 shows the code is able to do it)

THE DOCUMENTED-DEVIATIONS LIST is printed and saved (DOCUMENTED_DEVIATIONS, plain text, edit freely): the places
where Phase 3 differs from the Checklist or from the source paper, for the write-up.

run_final_gate(bn_check=True, save=True, raise_on_failure=True) -- the other arguments exist for testing on stand-in data.
"""

import json
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

GATE_FILENAME = "phase3_exit_gate.json"
GATE_FORMAT_VERSION = 1

# The values locked in the Checklist and the Phase 3 planning document, written out HERE on purpose (not read from the
# live constants), so a constant that drifted since the plan was made is caught.  (label, path in settings, value)
LOCKED_SETTINGS = [
    ("random seed", ("seed",), 42),
    ("number of folds", ("data", "n_folds"), 5),
    ("classes", ("data", "classes"), ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]),
    ("metadata vector width", ("data", "metadata_dim"), 20),
    ("backbone feature channels", ("model", "feature_channels"), 1536),
    ("metadata branch", ("model", "metadata_branch"), "Linear(20 -> 32) -> ReLU -> Dropout(0.3)"),
    ("head dropout", ("model", "head_dropout"), 0.5),
    ("number of classes", ("model", "num_classes"), 7),
    ("batch size", ("training", "batch_size"), 32),
    ("Phase A epochs", ("training", "phase_a_epochs"), 5),
    ("patience", ("training", "patience"), 10),
    ("patience counted from epoch", ("training", "patience_counted_from_epoch"), 6),
    ("optimizer", ("training", "optimizer"), "AdamW"),
    ("backbone learning rate", ("training", "backbone_lr"), 1e-4),
    ("head learning rate", ("training", "head_lr"), 1e-3),
    ("weight decay", ("training", "weight_decay"), 1e-5),
    ("effective-number beta", ("training", "effective_number_beta"), 0.999),
    ("loss", ("training", "loss"), "CrossEntropyLoss with per-fold effective-number class weights"),
    ("sampler", ("training", "sampler"), "WeightedRandomSampler, balanced by class, with replacement"),
]
CHECKLIST_MAX_EPOCHS = 50
ALLOWED_MAX_EPOCHS = (50, 100)      # 50 = the Checklist; 100 = documented deviation (as Phase 2). Anything else fails B3.

DOCUMENTED_DEVIATIONS = [
    "Maximum epochs is 100, not the Checklist's 50 (as in Phase 2); early stopping (patience 10) decides when a fold ends.",
    "Patience counts non-improving epochs from epoch 6 (the first Phase B epoch) only; Phase A epochs never count, so the "
    "earliest possible early stop is epoch 15.",
    "Phase A epochs compete for the best weights: the best epoch is the one with the highest validation AUC-ROC over ALL "
    "epochs, so a fold's best weights can come from Phase A (frozen backbone).",
    "One Dropout(0.5) in front of the final linear layer in both models. The stock torchvision classifier (with its own "
    "Dropout(0.3)) is not used; the Phase 2 EfficientNet-B3 model carried both.",
    "The image-only ablation is trained exactly like the multimodal model (same two-phase schedule, optimizer, sampler, "
    "loss, head); it differs only by having no metadata branch and no MetaBlock.",
    "MetaBlock works on the unpooled 10x10 feature map and global average pooling comes AFTER the fusion, following the "
    "equations of Pacheco & Krohling (2021). The authors' released code pools first and reshapes the pooled vector.",
    "The 20-value metadata encoding and the training recipe (two phases, learning rates, sampler, class weights) are this "
    "thesis's own choices, not the paper's.",
    "Class balance uses the same balanced sampler and effective-number class weights as Phase 2 (the Checklist does not "
    "spell the sampler out).",
    "Validation folds decide both the best epoch and the reported scores (as in Phase 2), and the frozen model's fold is "
    "chosen on the same scores, so fold scores are optimistic for the chosen model; unbiased estimates must come from "
    "data not used for any selection (the Phase 1 holdout, external data).",
    "Per-fold results are reported as recorded. If a fold's training trajectory looks abnormal for either model "
    "(per-epoch histories are saved inside each fold_N_resume.pt), describe it in the write-up.",
]

BN_SCOPE_NOTE = ("D1 re-runs the freeze / unfreeze mechanism with the real set_phase and train_one_epoch on small random "
                 "inputs (CPU). It shows that the code in use freezes and releases the backbone BatchNorm as designed; it "
                 "cannot look inside the past run. The past run's phase schedule is checked separately (D2) from each "
                 "run's saved per-epoch history.")


# ---------------------------------------------------------------------------
# D1: the BatchNorm freeze mechanism, live
# ---------------------------------------------------------------------------

def _bn_state(layers):
    return [torch.cat([m.running_mean.flatten(), m.running_var.flatten(), m.num_batches_tracked.float().flatten()]).clone()
            for m in layers]


def _tensors_changed(before, after):
    return sum(0 if torch.equal(b, a) else 1 for b, a in zip(before, after))


def batchnorm_freeze_check(image_size=64, batch=4, batches_per_phase=2, seed=None):
    """Build both models (no pretrained weights), run a few real training steps in Phase A and then in Phase B with the
    real set_phase and train_one_epoch, and count what moved. Returns {"passed", "problems", "models": {kind: numbers}}."""
    seed = RANDOM_SEED if seed is None else seed
    cpu = torch.device("cpu")
    builders = {"multimodal": lambda: build_multimodal_model(pretrained=False),
                "ablation": lambda: build_image_only_model(pretrained=False)}
    problems, report = [], {}
    for kind, build in builders.items():
        set_all_seeds(seed)
        model = build().to(cpu)
        optimizer = build_optimizer(model)
        criterion = nn.CrossEntropyLoss()
        generator = torch.Generator().manual_seed(seed)

        def make_batches():
            return [(torch.randn(batch, 3, image_size, image_size, generator=generator),
                     torch.randn(batch, METADATA_DIM, generator=generator), torch.arange(batch) % 7)
                    for _ in range(batches_per_phase)]

        bn_layers = [m for m in model.backbone.modules() if isinstance(m, nn.BatchNorm2d)]
        head_bn = [m for m in model.modules() if isinstance(m, nn.BatchNorm1d)]
        backbone_params, head_params = model.backbone_parameters(), model.head_parameters()
        measured = {}
        for phase in ("A", "B"):
            set_phase(model, phase)
            in_train_mode = sum(m.training for m in bn_layers)
            requiring_grad = sum(p.requires_grad for p in backbone_params)
            bn_before, bb_before = _bn_state(bn_layers), [p.detach().clone() for p in backbone_params]
            hd_before, hbn_before = [p.detach().clone() for p in head_params], _bn_state(head_bn)
            train_one_epoch(model, make_batches(), criterion, optimizer, cpu, use_amp=False, epoch=1)
            measured[phase] = {
                "backbone_bn_layers": len(bn_layers),
                "backbone_bn_layers_in_train_mode": int(in_train_mode),
                "backbone_params": len(backbone_params),
                "backbone_params_requiring_grad": int(requiring_grad),
                "backbone_bn_layers_changed": _tensors_changed(bn_before, _bn_state(bn_layers)),
                "backbone_params_changed": _tensors_changed(bb_before, [p.detach() for p in backbone_params]),
                "head_params_changed": _tensors_changed(hd_before, [p.detach() for p in head_params]),
                "head_bn1d_layers": len(head_bn),
                "head_bn1d_layers_changed": _tensors_changed(hbn_before, _bn_state(head_bn)),
            }
        a, b = measured["A"], measured["B"]
        if a["backbone_bn_layers"] == 0:
            problems.append(f"{kind}: no backbone BatchNorm2d layers were found")
        if a["backbone_bn_layers_in_train_mode"] != 0:
            problems.append(f"{kind}, Phase A: {a['backbone_bn_layers_in_train_mode']} backbone BatchNorm layers are in train mode (must be 0)")
        if a["backbone_params_requiring_grad"] != 0:
            problems.append(f"{kind}, Phase A: {a['backbone_params_requiring_grad']} backbone parameters still require gradients")
        if a["backbone_bn_layers_changed"] != 0:
            problems.append(f"{kind}, Phase A: the running statistics of {a['backbone_bn_layers_changed']} backbone BatchNorm layers changed")
        if a["backbone_params_changed"] != 0:
            problems.append(f"{kind}, Phase A: {a['backbone_params_changed']} backbone weights changed")
        if a["head_params_changed"] == 0:
            problems.append(f"{kind}, Phase A: the head group did not learn (no parameter changed)")
        if a["head_bn1d_layers"] and a["head_bn1d_layers_changed"] == 0:
            problems.append(f"{kind}, Phase A: the head group's own BatchNorm1d statistics did not update")
        if b["backbone_bn_layers_in_train_mode"] != b["backbone_bn_layers"]:
            problems.append(f"{kind}, Phase B: only {b['backbone_bn_layers_in_train_mode']} of {b['backbone_bn_layers']} backbone BatchNorm layers are in train mode")
        if b["backbone_params_requiring_grad"] != b["backbone_params"]:
            problems.append(f"{kind}, Phase B: only {b['backbone_params_requiring_grad']} of {b['backbone_params']} backbone parameters require gradients")
        if b["backbone_bn_layers_changed"] == 0:
            problems.append(f"{kind}, Phase B: the backbone BatchNorm statistics did not update")
        if b["backbone_params_changed"] == 0:
            problems.append(f"{kind}, Phase B: no backbone weight changed")
        report[kind] = measured
    return {"passed": not problems, "problems": problems, "models": report}


# ---------------------------------------------------------------------------
# What the run left on disk, run by run
# ---------------------------------------------------------------------------

def _run_facts(training_dir, recorded):
    """{(fold, kind): facts} from each run's resume file and best-weights file. A damaged file is reported, never raised."""
    facts = {}
    for kind in MODEL_KINDS:
        for fold in range(N_FOLDS):
            entry = {"finished": False, "resume_problem": None, "best_problem": None, "schedule_problem": None,
                     "epochs_in_history": None}
            resume_path, best_path = checkpoint_paths(model_dir(training_dir, kind), fold)
            identity = make_identity(kind, fold)
            record = recorded.get(kind, {}).get(str(fold))
            try:
                state = load_resume_state(resume_path, identity)
                if state is None:
                    entry["resume_problem"] = "no resume file"
                else:
                    entry["finished"] = bool(state["finished"])
                    history = list(state["history"])
                    entry["epochs_in_history"] = len(history)
                    epochs = [h["epoch"] for h in history]
                    if not history or epochs != list(range(1, len(epochs) + 1)):
                        entry["schedule_problem"] = "history epochs are not 1..n without gaps"
                    else:
                        wrong = [h["epoch"] for h in history if h["phase"] != phase_for_epoch(h["epoch"])]
                        if wrong:
                            entry["schedule_problem"] = f"epochs {wrong[:5]} carry the wrong phase label"
                        elif record is not None and record["epochs_run"] != len(history):
                            entry["schedule_problem"] = f"history has {len(history)} epochs, run_config records {record['epochs_run']}"
                    state = None                                   # free the weights / optimizer state at once
            except CheckpointError as exc:
                entry["resume_problem"] = f"{type(exc).__name__}: {exc}"
            try:
                best = _load_checked(best_path, identity, "Best-weights")
                if best is None:
                    entry["best_problem"] = "no best-weights file"
                elif record is not None and (best["epoch"] != record["best_epoch"] or
                                             abs(float(best["val_auc"]) - float(record["best_val_auc_roc"])) > 1e-12):
                    entry["best_problem"] = (f"file holds epoch {best['epoch']} / AUC {best['val_auc']}, run_config records "
                                             f"epoch {record['best_epoch']} / AUC {record['best_val_auc_roc']}")
                best = None
            except CheckpointError as exc:
                entry["best_problem"] = f"{type(exc).__name__}: {exc}"
            facts[(fold, kind)] = entry
    return facts


class _Checks:
    def __init__(self):
        self.items = []

    def add(self, cid, name, ok, detail="", status=None):
        self.items.append({"id": cid, "section": cid[0], "name": name,
                           "status": status or ("PASS" if ok else "FAIL"), "detail": detail})

    def skip(self, cid, name, reason):
        self.add(cid, name, False, reason, status="SKIPPED")

    def ids(self, status):
        return [c["id"] for c in self.items if c["status"] == status]


def _label(fold, kind):
    return f"{kind} fold {fold}"


# ---------------------------------------------------------------------------
# Sections A and B
# ---------------------------------------------------------------------------

def _check_run(checks, ctx):
    config, training_dir, pool = ctx["config"], ctx["training_dir"], ctx["expected_pool_rows"]
    recorded = config["results"]["per_fold"]

    missing = [_label(f, k) for k in MODEL_KINDS for f in range(N_FOLDS) if str(f) not in recorded.get(k, {})]
    flag = config["results"].get("complete") is True
    checks.add("A2", "all 10 runs are recorded and run_config.json says complete", not missing and flag,
               "10 of 10 recorded" if not missing and flag else f"missing: {missing}; complete flag: {flag}")

    facts = _run_facts(training_dir, recorded)
    ctx["facts"] = facts
    bad = [f"{_label(f, k)} ({e['resume_problem'] or 'not finished'})" for (f, k), e in facts.items()
           if e["resume_problem"] or not e["finished"]]
    checks.add("A3", "every run's resume file is readable and marked finished", not bad,
               "10 of 10 finished" if not bad else "; ".join(bad))
    bad = [f"{_label(f, k)}: {e['best_problem']}" for (f, k), e in facts.items() if e["best_problem"]]
    checks.add("A4", "every best-weights file matches the epoch and AUC-ROC recorded for it", not bad,
               "10 of 10 match" if not bad else "; ".join(bad))

    try:
        _, predictions = load_run_results(training_dir)
    except ResultsError as exc:
        checks.add("A5", "every predictions file loads and passes its checks", False, str(exc))
        for cid, name in (("A6", "every recorded metric reproduces from its predictions file"),
                          ("A7", "each model's validation folds cover the CV pool exactly once"),
                          ("A8", "multimodal and ablation predictions are for the same images and labels"),
                          ("A9", "run_config.json's mean +/- SD and paired difference equal a recomputation")):
            checks.skip(cid, name, "the predictions files could not be loaded (A5)")
        return
    ctx["predictions"] = predictions
    checks.add("A5", "every predictions file loads and passes its checks", True,
               f"{sum(len(v) for v in predictions.values())} files")

    tables = build_tables(config, predictions)
    wrong = [_label(int(r["fold"]), r["model"]) for _, r in tables["per_fold"].iterrows() if not r["metrics_reproduce"]]
    checks.add("A6", "every recorded metric reproduces from its predictions file", not wrong,
               f"{len(tables['per_fold'])} runs" if not wrong else "do not reproduce: " + ", ".join(wrong))

    problems = []
    for kind in MODEL_KINDS:
        ids = [i for f in sorted(predictions[kind]) for i in predictions[kind][f]["image_ids"]]
        if len(ids) != pool or len(set(ids)) != pool:
            problems.append(f"{kind}: {len(ids)} validation images, {len(set(ids))} distinct (expected {pool})")
    checks.add("A7", "each model's validation folds cover the CV pool exactly once", not problems,
               f"{pool} images, each exactly once, per model" if not problems else "; ".join(problems))

    problems = []
    for f in sorted(set(predictions["multimodal"]) | set(predictions["ablation"])):
        mm, ab = predictions["multimodal"].get(f), predictions["ablation"].get(f)
        if mm is None or ab is None:
            problems.append(f"fold {f}: recorded for only one model")
        elif mm["image_ids"] != ab["image_ids"] or not np.array_equal(mm["labels"], ab["labels"]):
            problems.append(f"fold {f}: different images, order or labels")
    checks.add("A8", "multimodal and ablation predictions are for the same images and labels", not problems,
               "all folds matched" if not problems else "; ".join(problems))

    problems = []
    results = config["results"]
    for kind in MODEL_KINDS:
        folds = sorted(recorded.get(kind, {}), key=int)
        if len(folds) >= 2:
            fresh = summarize_across_folds([recorded[kind][f]["metrics"] for f in folds])
            stored = results["mean_sd"].get(kind, {})
            for key in METRIC_KEYS:
                if key not in stored or abs(stored[key]["mean"] - fresh[key]["mean"]) > 1e-9 \
                        or abs(stored[key]["sd"] - fresh[key]["sd"]) > 1e-9:
                    problems.append(f"{kind} {key}")
    diffs = {f: recorded["multimodal"][f]["metrics"]["auc_roc"] - recorded["ablation"][f]["metrics"]["auc_roc"]
             for f in recorded.get("multimodal", {}) if f in recorded.get("ablation", {})}
    stored_pair = results.get("paired_auc_roc_difference", {})
    if set(stored_pair.get("per_fold", {})) != set(diffs) or any(abs(stored_pair["per_fold"][f] - diffs[f]) > 1e-9 for f in diffs):
        problems.append("paired difference per fold")
    elif len(diffs) >= 2 and (abs(stored_pair.get("mean", np.nan) - np.mean(list(diffs.values()))) > 1e-9 or
                              abs(stored_pair.get("sd", np.nan) - np.std(list(diffs.values()), ddof=1)) > 1e-9):
        problems.append("paired difference mean / SD")
    checks.add("A9", "run_config.json's mean +/- SD and paired difference equal a recomputation", not problems,
               "match" if not problems else "differ: " + ", ".join(problems))


def _check_settings(checks, ctx):
    config = ctx["config"]
    settings = config["settings"]
    mismatches = []
    for label, path, want in ctx["locked_settings"]:
        node = settings
        try:
            for key in path:
                node = node[key]
        except (KeyError, TypeError):
            node = "<absent>"
        if node != want:
            mismatches.append(f"{label}: recorded {node!r}, locked {want!r}")
    checks.add("B1", "settings recorded in run_config.json equal the locked Checklist / planning values", not mismatches,
               f"{len(ctx['locked_settings'])} values match" if not mismatches else "; ".join(mismatches))

    recorded_pool = settings.get("data", {}).get("cv_pool_rows")
    checks.add("B2", "the CV pool size recorded is the Phase 1 pool", recorded_pool == ctx["expected_pool_rows"],
               f"{recorded_pool} rows" if recorded_pool == ctx["expected_pool_rows"]
               else f"recorded {recorded_pool!r}, Phase 1 pool {ctx['expected_pool_rows']}")

    max_epochs = settings.get("training", {}).get("max_epochs")
    if max_epochs == CHECKLIST_MAX_EPOCHS:
        checks.add("B3", "maximum epochs", True, f"{max_epochs} (the Checklist's value)")
    elif max_epochs in ctx["allowed_max_epochs"]:
        checks.add("B3", "maximum epochs", True,
                   f"{max_epochs}; the Checklist says {CHECKLIST_MAX_EPOCHS} -- documented deviation", status="NOTE")
    else:
        checks.add("B3", "maximum epochs", False,
                   f"{max_epochs!r} is neither the Checklist's {CHECKLIST_MAX_EPOCHS} nor a documented deviation "
                   f"{tuple(ctx['allowed_max_epochs'])}")

    changes = config.get("environment_changes", [])
    if not changes:
        checks.add("B4", "software environment unchanged during training", True, "no change recorded")
    else:
        names = sorted({key for c in changes for key in c["differences"]})
        checks.add("B4", "software environment unchanged during training", True,
                   f"{len(changes)} change(s) recorded, in: {', '.join(names)} -- mention in the write-up", status="NOTE")


# ---------------------------------------------------------------------------
# Section C
# ---------------------------------------------------------------------------

def _check_selection(checks, ctx):
    config, training_dir, final_dir = ctx["config"], ctx["training_dir"], ctx["final_dir"]
    recorded = config["results"]["per_fold"]
    names = {"C1": "model card is readable", "C2": "the card's five fold scores equal run_config.json's",
             "C3": "the selected fold is the best fold", "C4": "frozen file = card's SHA-256 = selected fold's best weights",
             "C5": "frozen file's identity, epoch and AUC-ROC match the record",
             "C6": "reload check passes now (fresh model + frozen file)",
             "C7": "card's metadata encoding equals metadata_encoder_stats.json",
             "C8": "card records class order, image contract and the downstream rule"}
    card_path, frozen_path = final_dir / MODEL_CARD_FILENAME, final_dir / FROZEN_MODEL_FILENAME
    try:
        with open(card_path, "r", encoding="utf-8") as f:
            card = json.load(f)
        needed = ("selection", "frozen_file", "input_contract", "downstream_rule", "model")
        absent = [k for k in needed if k not in card]
        if card.get("format_version") != MODEL_CARD_FORMAT_VERSION or absent:
            raise ValueError(f"wrong format version or missing parts {absent}")
    except (OSError, ValueError) as exc:
        checks.add("C1", names["C1"], False, f"{card_path}: {exc}")
        for cid in ("C2", "C3", "C4", "C5", "C6", "C7", "C8"):
            checks.skip(cid, names[cid], "the model card could not be read (C1)")
        return
    checks.add("C1", names["C1"], True, str(card_path.name))

    scores = {str(f): float(recorded["multimodal"][str(f)]["metrics"]["auc_roc"])
              for f in range(N_FOLDS) if str(f) in recorded.get("multimodal", {})}
    card_scores = card["selection"].get("all_fold_val_auc_roc", {})
    same = set(card_scores) == set(scores) and len(scores) == N_FOLDS and \
        all(abs(card_scores[f] - scores[f]) <= 1e-12 for f in scores)
    checks.add("C2", names["C2"], same, "5 scores equal" if same else f"card {card_scores}, run_config {scores}")

    try:
        selection = select_best_fold(config["results"])
        chosen = card["selection"].get("selected_fold")
        ok = chosen == selection["selected_fold"] and abs(card["selection"].get("val_auc_roc", -1) - selection["val_auc_roc"]) <= 1e-12
        checks.add("C3", names["C3"], ok, f"fold {chosen}, AUC-ROC {selection['val_auc_roc']:.4f}" if ok
                   else f"card says fold {chosen} (AUC {card['selection'].get('val_auc_roc')}), the best is fold "
                        f"{selection['selected_fold']} (AUC {selection['val_auc_roc']})")
    except ResultsError as exc:
        selection, chosen = None, card["selection"].get("selected_fold")
        checks.add("C3", names["C3"], False, str(exc))

    fold = chosen if isinstance(chosen, int) and not isinstance(chosen, bool) and 0 <= chosen < N_FOLDS else None
    if fold is None or str(fold) not in recorded.get("multimodal", {}):
        for cid in ("C4", "C5", "C6"):
            checks.skip(cid, names[cid], "the card's selected fold is not a recorded multimodal fold")
        record = None
    else:
        record = recorded["multimodal"][str(fold)]
        _, source = checkpoint_paths(model_dir(training_dir, "multimodal"), fold)
        problems = []
        if not frozen_path.exists():
            problems.append(f"frozen file missing: {frozen_path}")
        elif not os.path.exists(source):
            problems.append(f"the selected fold's best weights are missing: {source}")
        else:
            frozen_hash, source_hash = sha256_of_file(frozen_path), sha256_of_file(source)
            if frozen_hash != card["frozen_file"].get("sha256"):
                problems.append("the frozen file's SHA-256 differs from the card's (file changed after freezing)")
            if frozen_hash != source_hash:
                problems.append("the frozen file differs from the selected fold's best weights")
        checks.add("C4", names["C4"], not problems, "all three SHA-256 values equal" if not problems else "; ".join(problems))

        if frozen_path.exists():
            try:
                state = _load_checked(frozen_path, make_identity("multimodal", fold), "Frozen-model")
                ok = state["epoch"] == record["best_epoch"] and abs(float(state["val_auc"]) - float(record["best_val_auc_roc"])) <= 1e-12
                checks.add("C5", names["C5"], ok, f"multimodal fold {fold}, epoch {state['epoch']}" if ok
                           else f"file holds epoch {state['epoch']} / AUC {state['val_auc']}; run_config records "
                                f"epoch {record['best_epoch']} / AUC {record['best_val_auc_roc']}")
                state = None
            except CheckpointError as exc:
                checks.add("C5", names["C5"], False, str(exc))
        else:
            checks.skip("C5", names["C5"], "frozen file missing (C4)")

        try:
            result = reload_check(frozen_path, fold, record["metrics"],
                                  model_dir(training_dir, "multimodal") / record["predictions_file"],
                                  encoder_record=ctx["encoder_record"], make_val_dataset=ctx["make_val_dataset"],
                                  build_model_fn=ctx["build_model_fn"])
            ctx["reload_result"] = result
            checks.add("C6", names["C6"], result["passed"],
                       (f"AUC-ROC {result['auc_roc_recomputed']:.6f} vs recorded {result['auc_roc_recorded']:.6f}, largest "
                        f"probability difference {result['max_abs_probability_difference']:.2e}") if result["passed"]
                       else "; ".join(result["problems"]))
        except (ResultsError, CheckpointError) as exc:
            checks.add("C6", names["C6"], False, str(exc))

    problems = []
    if fold is not None:
        age = card["input_contract"].get("metadata", {}).get("age", {})
        try:
            mean, std = get_fold_age_stats(ctx["encoder_record"], fold)
            if age.get("mean") != mean or age.get("std") != std:
                problems.append(f"age mean/std in the card ({age.get('mean')}, {age.get('std')}) differ from the stats file ({mean}, {std})")
        except KeyError:
            problems.append(f"no statistics for fold {fold} in the encoder stats")
    meta = card["input_contract"].get("metadata", {})
    if meta.get("feature_names") != list(FEATURE_NAMES):
        problems.append("feature names differ from the live layout")
    if meta.get("sex_categories") != list(SEX_CATEGORIES) or meta.get("site_categories") != list(SITE_CATEGORIES):
        problems.append("sex / site categories differ from the live layout")
    checks.add("C7", names["C7"], not problems, "age mean/std, feature names and categories equal" if not problems else "; ".join(problems))

    problems = []
    if card["input_contract"].get("classes_output_order") != list(CLASSES):
        problems.append("class order in the card differs from CLASSES")
    if card["input_contract"].get("image", {}).get("size") != [300, 300]:
        problems.append("image size in the card is not 300x300")
    rule = card.get("downstream_rule", "")
    if not isinstance(rule, str) or "FROZEN" not in rule:
        problems.append("the downstream rule ('FROZEN ...') is missing")
    checks.add("C8", names["C8"], not problems, "recorded" if not problems else "; ".join(problems))


# ---------------------------------------------------------------------------
# Section D
# ---------------------------------------------------------------------------

def _check_batchnorm(checks, ctx):
    name1 = "BatchNorm freeze / unfreeze works (live check, real set_phase + train_one_epoch)"
    if not ctx["bn_check"]:
        checks.skip("D1", name1, "bn_check=False -- a gate without D1 is not a pass")
        ctx["bn_evidence"] = None
    else:
        result = batchnorm_freeze_check()
        ctx["bn_evidence"] = {"scope_note": BN_SCOPE_NOTE, "live_check": result}
        if result["passed"]:
            m = result["models"]["multimodal"]
            detail = (f"{m['A']['backbone_bn_layers']} backbone BatchNorm layers: Phase A {m['A']['backbone_bn_layers_changed']} "
                      f"changed / {m['A']['backbone_bn_layers_in_train_mode']} in train mode, weights moved: "
                      f"{m['A']['backbone_params_changed']}; Phase B {m['B']['backbone_bn_layers_changed']} changed, weights moved: "
                      f"{m['B']['backbone_params_changed']} (the ablation passes the same checks)")
        else:
            detail = "; ".join(result["problems"])
        checks.add("D1", name1, result["passed"], detail)

    name2 = "each run's saved history follows the schedule (epochs 1-5 Phase A, 6+ Phase B, no gaps)"
    facts = ctx["facts"]
    bad = [f"{_label(f, k)}: {e['schedule_problem'] or e['resume_problem']}" for (f, k), e in facts.items()
           if e["schedule_problem"] or e["resume_problem"]]
    checks.add("D2", name2, not bad, "10 of 10 runs" if not bad else "; ".join(bad))


# ---------------------------------------------------------------------------
# The gate
# ---------------------------------------------------------------------------

def _facts_from_run(config, facts):
    recorded = config["results"]["per_fold"]
    over_50, in_phase_a, stops = [], [], {}
    for kind in MODEL_KINDS:
        for f in sorted(recorded.get(kind, {}), key=int):
            rec = recorded[kind][f]
            if rec["best_epoch"] > CHECKLIST_MAX_EPOCHS:
                over_50.append(f"{kind} fold {f} (best epoch {rec['best_epoch']})")
            if rec["best_epoch_phase"] == "A":
                in_phase_a.append(f"{kind} fold {f} (epoch {rec['best_epoch']})")
            stops.setdefault(kind, {}).setdefault(rec["stop_reason"], 0)
            stops[kind][rec["stop_reason"]] += 1
    return {"runs_whose_best_epoch_is_above_the_checklist_limit_of_50": over_50,
            "runs_whose_best_weights_come_from_phase_a": in_phase_a, "stop_reasons": stops}


def run_final_gate(encoder_record=None, training_dir=None, final_dir=None, make_val_dataset=None, build_model_fn=None,
                   bn_check=True, save=True, raise_on_failure=True, locked_settings=None, allowed_max_epochs=None,
                   expected_pool_rows=None):
    """Run every check, print the report, save phase3_exit_gate.json, and raise RuntimeError if any check FAILED or
    was SKIPPED. Returns the report dict. (Arguments after `final_dir` exist so the gate can be tested on stand-in data.)"""
    training_dir = Path(training_dir) if training_dir else TRAINING_DIR
    final_dir = Path(final_dir) if final_dir else FINAL_MODEL_DIR
    ctx = {"training_dir": training_dir, "final_dir": final_dir,
           "encoder_record": encoder_record if encoder_record is not None else load_encoder_stats(),
           "make_val_dataset": make_val_dataset, "build_model_fn": build_model_fn, "bn_check": bn_check,
           "locked_settings": locked_settings if locked_settings is not None else LOCKED_SETTINGS,
           "allowed_max_epochs": allowed_max_epochs if allowed_max_epochs is not None else ALLOWED_MAX_EPOCHS,
           "expected_pool_rows": expected_pool_rows if expected_pool_rows is not None else EXPECTED_POOL_ROWS,
           "facts": {}, "bn_evidence": None}
    checks = _Checks()

    print("=" * 78)
    print("PHASE 3 EXIT GATE -- every check reads the files on disk")
    print("=" * 78)
    try:
        ctx["config"] = _read_config(run_config_path(training_dir))
        checks.add("A1", "run_config.json is readable and in the expected format", True, str(run_config_path(training_dir)))
        _check_run(checks, ctx)
        _check_settings(checks, ctx)
        _check_selection(checks, ctx)
        _check_batchnorm(checks, ctx)
    except RunConfigError as exc:
        checks.add("A1", "run_config.json is readable and in the expected format", False, str(exc))
        ctx["config"] = None

    counts = {s: len(checks.ids(s)) for s in ("PASS", "FAIL", "NOTE", "SKIPPED")}
    passed = counts["FAIL"] == 0 and counts["SKIPPED"] == 0 and counts["PASS"] + counts["NOTE"] > 0 and ctx["config"] is not None

    last_section = None
    for c in checks.items:
        if c["section"] != last_section:
            last_section = c["section"]
            print({"A": "\nA. The run is complete and intact", "B": "\nB. The settings were the locked ones",
                   "C": "\nC. The selection and the frozen model", "D": "\nD. BatchNorm freeze evidence"}[last_section])
        print(f"  [{c['status']:<7}] {c['id']}  {c['name']}" + (f"\n             {c['detail']}" if c["detail"] else ""))

    facts_run = _facts_from_run(ctx["config"], ctx["facts"]) if ctx["config"] is not None else {}
    print("\nDOCUMENTED DEVIATIONS (for the write-up)")
    for i, text in enumerate(DOCUMENTED_DEVIATIONS, 1):
        print(f"  {i:>2}. {text}")
    if facts_run:
        print("FACTS FROM THIS RUN (for the same list)")
        print(f"  best epoch above the Checklist's 50: {facts_run['runs_whose_best_epoch_is_above_the_checklist_limit_of_50'] or 'none'}")
        print(f"  best weights from Phase A: {facts_run['runs_whose_best_weights_come_from_phase_a'] or 'none'}")
        print(f"  stop reasons: {facts_run['stop_reasons']}")

    report = {"format_version": GATE_FORMAT_VERSION, "created_at": _now(), "passed": bool(passed), "counts": counts,
              "training_dir": str(training_dir), "final_dir": str(final_dir), "checks": checks.items,
              "batchnorm_evidence": ctx["bn_evidence"], "documented_deviations": list(DOCUMENTED_DEVIATIONS),
              "facts_from_this_run": facts_run}
    if save:
        _atomic_write_json(final_dir / GATE_FILENAME, report)
        print(f"\nGate report saved: {final_dir / GATE_FILENAME}")

    print("=" * 78)
    if passed:
        print(f"PHASE 3 EXIT GATE: PASSED -- {counts['PASS']} checks passed, {counts['NOTE']} note(s) to write up.")
        print("=" * 78)
    else:
        problems = [f"{c['id']} {c['name']}" for c in checks.items if c["status"] in ("FAIL", "SKIPPED")]
        print(f"PHASE 3 EXIT GATE: FAILED -- {len(problems)} check(s) not passed:")
        for p in problems:
            print(f"  - {p}")
        print("=" * 78)
        if raise_on_failure:
            raise RuntimeError(f"Phase 3 exit gate failed: {len(problems)} check(s) not passed ({', '.join(checks.ids('FAIL') + checks.ids('SKIPPED'))}).")
    return report


print("Final gate loaded.")
print("  run_final_gate()   -- checks the finished phase against the locked plan; writes phase3_exit_gate.json")
print("  batchnorm_freeze_check()   -- the live BatchNorm freeze / unfreeze check the gate uses (D1)")

Final gate loaded.
  run_final_gate()   -- checks the finished phase against the locked plan; writes phase3_exit_gate.json
  batchnorm_freeze_check()   -- the live BatchNorm freeze / unfreeze check the gate uses (D1)


In [50]:
"""
10_03_module10_verification.txt
Phase 3 -- Module 10: Results, Selection and Exit Gate -- Verification (run once)

Requires 01_00, 01_01, 02_00, 02_01, 03_00, 03_01, 03_02, 04_00, 04_01, 05_00, 06_00, 07_00, 07_01, 07_02, 08_00,
09_00, 09_01, 10_00, 10_01, 10_02 already run.
Builds a small but REAL 5-fold run (the real run_cv_training, on CPU, with a tiny stand-in model and synthetic data:
24 training and 28 validation "images" per fold, batch 8, 6 epochs), then runs the real Module 10 code on it, and on
damaged copies of it. No dataset, no pretrained weights, no GPU. To do that it temporarily swaps DEVICE, USE_AMP,
BATCH_SIZE, MAX_EPOCHS and EXPECTED_POOL_ROWS for small values and restores them afterwards. Everything goes to a
temporary folder that is deleted afterwards. Your real TRAINING_DIR, RESULTS_SUMMARY_DIR and FINAL_MODEL_DIR are never touched.
(One part uses the real EfficientNet-B3 + MetaBlock architecture, without pretrained weights, on 64x64 random images: the
BatchNorm freeze check. It takes a few seconds on CPU.)

All eight checks run as a full set (failures collected, one RuntimeError at the end).

Checks:
  1. Result tables     -- shapes; mean +/- SD and the paired difference equal a by-hand recomputation; per-class values
                          average back to each fold's macro metrics; a per-class AUC equals an independent sklearn call;
                          the CSV files read back equal; COLUMN_NAMES renames columns; partial runs work.
  2. Damaged files     -- a predictions file with a wrong shape / class order / label / probabilities / best epoch /
                          duplicate ids, a missing file, or a wrong file name in run_config.json is refused, naming the file.
  3. Selection         -- the highest multimodal AUC-ROC wins; a tie goes to the lower fold; an incomplete run, or a
                          missing fold, is refused.
  4. Freeze            -- the frozen file is byte-identical to the selected fold's best weights (SHA-256) and the card is
                          complete; a second call changes nothing; a different file already there is never overwritten; a
                          corrupted copy is caught; a source with the wrong identity, the wrong epoch, or from an
                          incomplete run is refused.
  5. Reload check      -- passes on the real freeze; fails for altered weights, a different image order, different
                          labels, probabilities far from the saved ones, weights that do not fit the model, and an
                          AUC that does not match the record.
  6. Gate, clean run   -- every check passes (one NOTE: maximum epochs); the report file is written; a recorded
                          environment change is a NOTE, not a failure; a gate without the BatchNorm check is not a pass.
  7. Gate, damage      -- 28 different kinds of damage (changed / missing frozen file, card edited, wrong scores,
                          missing / cut / reordered predictions, altered metric or summary, drifted setting, undocumented
                          maximum epochs, unfinished run, wrong phase history, altered best file, incomplete run, unreadable
                          run_config.json ...) each make the gate fail on the check meant to catch it.
  8. BatchNorm check   -- passes with the real set_phase; fails if Phase A forgets to put the backbone in eval mode, if
                          Phase A leaves the backbone weights trainable, or if Phase B never (or only partly) unfreezes.
"""

import contextlib
import io
import json
import os
import shutil
import tempfile
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score


class _V10TinyNet(nn.Module):
    """Stand-in with the real models' structure. `hidden` differs between the two 'models' so their results differ."""

    def __init__(self, hidden=6):
        super().__init__()
        self.backbone = nn.Sequential(nn.Conv2d(3, 4, 3, padding=1), nn.BatchNorm2d(4), nn.ReLU(),
                                      nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Sequential(nn.Linear(4, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.5),
                                  nn.Linear(hidden, 7))

    def forward(self, image, metadata):
        return self.head(self.backbone(image))

    def backbone_parameters(self):
        return list(self.backbone.parameters())

    def head_parameters(self):
        return list(self.head.parameters())


class _V10SynthDataset:
    def __init__(self, n, seed, prefix):
        g = torch.Generator()
        g.manual_seed(seed)
        self.images = torch.randn(n, 3, 8, 8, generator=g)
        self.meta = torch.randn(n, METADATA_DIM, generator=g)
        self.labels = np.arange(n) % 7
        self.image_ids = [f"{prefix}_{i:03d}" for i in range(n)]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, i):
        return self.images[i], self.meta[i], int(self.labels[i])

    def get_labels(self):
        return self.labels


class _V10ReversedView:
    """The same images in the opposite order (with their ids) -- a validation set that differs from the recorded one."""

    def __init__(self, ds):
        self.ds = ds
        self.image_ids = list(reversed(ds.image_ids))

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        return self.ds[len(self.ds) - 1 - i]


class _V10RelabelView:
    """The same images and ids, but every label shifted by one class -- a validation set with other labels."""

    def __init__(self, ds):
        self.ds = ds
        self.image_ids = list(ds.image_ids)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        image, meta, label = self.ds[i]
        return image, meta, (label + 1) % 7


@contextlib.contextmanager
def _v10_patch_globals(**overrides):
    g = globals()
    saved = {name: g[name] for name in overrides}
    g.update(overrides)
    try:
        yield
    finally:
        g.update(saved)


@contextlib.contextmanager
def _v10_patch_function(name, make_replacement):
    g = globals()
    real = g[name]
    g[name] = make_replacement(real)
    try:
        yield
    finally:
        g[name] = real


@contextlib.contextmanager
def _v10_patch_dict(target, changes):
    saved = dict(target)
    target.update(changes)
    try:
        yield
    finally:
        target.clear()
        target.update(saved)


def _v10_quiet(fn, *args, **kwargs):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)


def _v10_read(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _v10_write(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f)


def _v10_bytes(path):
    with open(path, "rb") as f:
        return f.read()


def _v10_edit_checkpoint(path, edit):
    state = torch.load(path, map_location="cpu", weights_only=False)
    edit(state)
    torch.save(state, path)


def _v10_edit_npz(path, edit):
    with np.load(path, allow_pickle=False) as data:
        content = {k: data[k] for k in data.files}
    edit(content)
    np.savez(path, **content)


def _v10_same_table(a, b):
    """Same columns, equal numbers (NaN == NaN) and equal text."""
    if list(a.columns) != list(b.columns) or len(a) != len(b):
        return False
    for col in a.columns:
        if pd.api.types.is_numeric_dtype(a[col]) and not pd.api.types.is_bool_dtype(a[col]):
            if not np.allclose(a[col].to_numpy(dtype=float), b[col].to_numpy(dtype=float), equal_nan=True, atol=1e-12):
                return False
        elif list(a[col].astype(str)) != list(b[col].astype(str)):
            return False
    return True


def run_module10_verification():
    failures = []
    print("=" * 60)
    print("MODULE 10 VERIFICATION -- results, selection, freeze, exit gate")
    print("=" * 60)
    print("  real run_cv_training + real Module 10 code, CPU, tiny stand-in models, synthetic data, temporary folder")

    cpu = torch.device("cpu")
    C = COLUMN_NAMES
    folds_all = list(range(N_FOLDS))
    enc = {"feature_names": list(FEATURE_NAMES), "age_std_ddof": 0, "age_missing_rule": "stand-in rule",
           "folds": {str(f): {"mean": 50.0 + f, "std": 15.0, "n_recorded": 100, "n_train_images": 110} for f in folds_all}}
    make_val = lambda fold: _V10SynthDataset(28, 200 + fold, f"va{fold}")
    make_model = lambda: _V10TinyNet(6)
    builders = {"multimodal": lambda: _V10TinyNet(6), "ablation": lambda: _V10TinyNet(5)}

    small = dict(DEVICE=cpu, USE_AMP=False, BATCH_SIZE=8, MAX_EPOCHS=6, EXPECTED_POOL_ROWS=140)
    with tempfile.TemporaryDirectory() as tmp, warnings.catch_warnings(), _v10_patch_globals(**small):
        warnings.filterwarnings("ignore", message="The given NumPy array is not writable")

        def d(name):
            return os.path.join(tmp, name)

        locked = [(label, path, BATCH_SIZE if path == ("training", "batch_size") else value)
                  for label, path, value in LOCKED_SETTINGS]          # the tiny run's batch size is 8, not 32

        def gate(td, fd, bn=False, **over):
            kw = dict(encoder_record=enc, training_dir=td, final_dir=fd, make_val_dataset=make_val,
                      build_model_fn=make_model, bn_check=bn, save=False, raise_on_failure=False,
                      locked_settings=locked, allowed_max_epochs=(MAX_EPOCHS,))
            kw.update(over)
            return _v10_quiet(run_final_gate, **kw)

        def failed(report):
            return {c["id"] for c in report["checks"] if c["status"] == "FAIL"}

        def status_of(report, cid):
            return next(c["status"] for c in report["checks"] if c["id"] == cid)

        # --- Setup: a real tiny run, frozen ---
        base_cv, base_final = d("base_cv"), d("base_final")
        try:
            _v10_quiet(run_cv_training, None, training_dir=base_cv,
                       make_datasets=lambda fold: (_V10SynthDataset(24, 100 + fold, f"tr{fold}"), make_val(fold)),
                       model_builders=builders, criterion_fn=lambda fold: nn.CrossEntropyLoss(), log_fn=lambda *_: None)
            _v10_quiet(select_and_freeze, encoder_record=enc, training_dir=base_cv, final_dir=base_final,
                       make_val_dataset=make_val, build_model_fn=make_model)
        except Exception as exc:                                       # noqa: BLE001 -- a setup failure ends the verification
            print(f"\nSETUP FAILED: {type(exc).__name__}: {exc}")
            raise RuntimeError(f"Module 10 verification could not build its stand-in run: {exc}") from exc

        base_results = _v10_read(run_config_path(base_cv))["results"]
        mm_auc = {f: base_results["per_fold"]["multimodal"][str(f)]["metrics"]["auc_roc"] for f in folds_all}
        sel = max(folds_all, key=lambda f: (mm_auc[f], -f))             # by hand: highest AUC-ROC, lower fold on a tie
        frozen_base = os.path.join(base_final, FROZEN_MODEL_FILENAME)
        source_of = lambda td, fold: checkpoint_paths(model_dir(td, "multimodal"), fold)[1]

        def fresh(name):
            td, fd = d(name + "_cv"), d(name + "_final")
            shutil.copytree(base_cv, td)
            shutil.copytree(base_final, fd)
            return td, fd

        # --- 1. Result tables ---
        try:
            print("\n[1/8] Result tables...")
            ok = True
            res = _v10_quiet(collect_results, training_dir=base_cv, out_dir=d("summary"), print_tables=False)
            want_rows = {"per_fold": 10, "mean_sd": 11, "paired": 5, "per_class": 70, "per_class_mean_sd": 42}
            for name, n in want_rows.items():
                if len(res[name]) != n:
                    failures.append(f"Result tables: '{name}' has {len(res[name])} rows, expected {n}"); ok = False
            if res["complete"] is not True:
                failures.append("Result tables: a complete run is not reported complete"); ok = False
            per_fold_cfg = base_results["per_fold"]
            ms = res["mean_sd"]
            for kind in MODEL_KINDS:
                for key in METRIC_KEYS:
                    values = [per_fold_cfg[kind][str(f)]["metrics"][key] for f in folds_all]
                    row = ms[(ms[C["model"]] == kind) & (ms[C["metric"]] == key)]
                    if len(row) != 1 or not (np.isclose(row[C["mean"]].iloc[0], np.mean(values), atol=1e-12) and
                                             np.isclose(row[C["sd"]].iloc[0], np.std(values, ddof=1), atol=1e-12)):
                        failures.append(f"Result tables: {kind} {key} mean +/- SD differs from a by-hand recomputation"); ok = False
            diffs = [per_fold_cfg["multimodal"][str(f)]["metrics"]["auc_roc"] - per_fold_cfg["ablation"][str(f)]["metrics"]["auc_roc"]
                     for f in folds_all]
            prow = ms[ms[C["model"]] == PAIRED_LABEL]
            if len(prow) != 1 or not (np.isclose(prow[C["mean"]].iloc[0], np.mean(diffs), atol=1e-12) and
                                      np.isclose(prow[C["sd"]].iloc[0], np.std(diffs, ddof=1), atol=1e-12)):
                failures.append("Result tables: the paired difference mean / SD is wrong"); ok = False
            if not np.allclose(res["paired"][C["difference"]].to_numpy(dtype=float), diffs, atol=1e-12):
                failures.append("Result tables: the per-fold paired differences are wrong"); ok = False
            if all(abs(v) < 1e-12 for v in diffs):
                failures.append("Result tables: the two stand-in models gave identical AUCs -- test setup is not discriminating"); ok = False

            pc, pf = res["per_class"], res["per_fold"]
            for _, r in pf.iterrows():
                part = pc[(pc[C["model"]] == r[C["model"]]) & (pc[C["fold"]] == r[C["fold"]])]
                for key in PER_CLASS_KEYS:
                    if not np.isclose(part[C[key]].mean(), r[C[key]], atol=1e-9):
                        failures.append(f"Result tables: per-class {key} of {r[C['model']]} fold {r[C['fold']]} does not average "
                                        f"back to the fold metric"); ok = False
                if int(part[C["support"]].sum()) != 28:
                    failures.append(f"Result tables: per-class supports of {r[C['model']]} fold {r[C['fold']]} do not add up to 28"); ok = False
            with np.load(os.path.join(model_dir(base_cv, "multimodal"), "fold_2_val_predictions.npz"), allow_pickle=False) as f:
                labels2, probs2 = f["labels"], f["probs"]
            mel = CLASS_TO_IDX["mel"]
            row = pc[(pc[C["model"]] == "multimodal") & (pc[C["fold"]] == 2) & (pc[C["class"]] == "mel")]
            if len(row) != 1 or not np.isclose(row[C["auc_roc"]].iloc[0], roc_auc_score((labels2 == mel).astype(int), probs2[:, mel]), atol=1e-12):
                failures.append("Result tables: a per-class AUC differs from an independent sklearn computation"); ok = False

            bcc_values = pc[(pc[C["model"]] == "multimodal") & (pc[C["class"]] == "bcc")].sort_values(C["fold"])[C["auc_roc"]].tolist()
            pcms = res["per_class_mean_sd"]
            row = pcms[(pcms[C["model"]] == "multimodal") & (pcms[C["class"]] == "bcc") & (pcms[C["metric"]] == "auc_roc")]
            if len(row) != 1 or not (np.isclose(row[C["mean"]].iloc[0], np.mean(bcc_values), atol=1e-12) and
                                     np.isclose(row[C["sd"]].iloc[0], np.std(bcc_values, ddof=1), atol=1e-12) and
                                     int(row[C["n_folds"]].iloc[0]) == N_FOLDS and int(row[C["support_total"]].iloc[0]) == 20):
                failures.append("Result tables: a per-class mean +/- SD (multimodal, bcc, AUC-ROC) differs from a by-hand recomputation"); ok = False

            for name, filename in (("per_fold", PER_FOLD_CSV), ("mean_sd", MEAN_SD_CSV), ("paired", PAIRED_CSV),
                                   ("per_class", PER_CLASS_CSV), ("per_class_mean_sd", PER_CLASS_MEAN_SD_CSV)):
                path = os.path.join(d("summary"), filename)
                if not os.path.exists(path):
                    failures.append(f"Result tables: {filename} was not written"); ok = False
                elif not _v10_same_table(pd.read_csv(path), res[name]):
                    failures.append(f"Result tables: {filename} read back differs from the returned table"); ok = False

            with _v10_patch_dict(COLUMN_NAMES, {"auc_roc": "AUC_renamed"}):
                res2 = _v10_quiet(collect_results, training_dir=base_cv, out_dir=d("summary2"), print_tables=False)
            header = open(os.path.join(d("summary2"), PER_FOLD_CSV), encoding="utf-8").readline()
            if "AUC_renamed" not in res2["per_fold"].columns or "AUC_renamed" not in header or "auc_roc" in header:
                failures.append("Result tables: renaming a column in COLUMN_NAMES did not reach the table and the CSV"); ok = False

            for keep, want_folds, want_mean_sd in ((["0", "1"], 2, 11), (["0"], 1, 0)):
                td = d("partial_" + "".join(keep))
                shutil.copytree(base_cv, td)
                cfg = _v10_read(run_config_path(td))
                for kind in MODEL_KINDS:
                    cfg["results"]["per_fold"][kind] = {k: v for k, v in cfg["results"]["per_fold"][kind].items() if k in keep}
                cfg["results"]["complete"] = False
                _v10_write(run_config_path(td), cfg)
                try:
                    part = _v10_quiet(collect_results, training_dir=td, save=False, print_tables=True)
                    if len(part["per_fold"]) != 2 * want_folds or len(part["mean_sd"]) != want_mean_sd or part["complete"] is not False:
                        failures.append(f"Result tables: a partial run with folds {keep} gave {len(part['per_fold'])} fold rows, "
                                        f"{len(part['mean_sd'])} mean +/- SD rows, complete={part['complete']}"); ok = False
                except Exception as exc:                                   # noqa: BLE001
                    failures.append(f"Result tables: a partial run with folds {keep} crashed -- {type(exc).__name__}: {exc}"); ok = False
            print("  OK -- row counts; mean +/- SD and paired difference match by-hand values; per-class values average back "
                  "to the fold metrics; CSVs read back equal; columns renameable; partial runs work" if ok else "  FAILED -- see summary")
        except Exception as exc:                                       # noqa: BLE001 -- a crash is a finding, not the end
            failures.append(f"Check 1: crashed -- {type(exc).__name__}: {exc}")
            print("  CRASHED -- see summary")

        # --- 2. Damaged files ---
        try:
            print("\n[2/8] Damaged result files are refused, naming the file...")
            ok = True
            target = "fold_2_val_predictions.npz"

            def set_probs_shape(c): c["probs"] = np.concatenate([c["probs"], np.zeros((len(c["probs"]), 1), dtype=c["probs"].dtype)], axis=1)
            def set_classes(c): c["class_names"] = np.array(["a", "bcc", "bkl", "df", "mel", "nv", "vasc"], dtype=str)
            def set_label(c): c["labels"] = c["labels"].copy(); c["labels"][0] = 9
            def set_probs(c): c["probs"] = c["probs"] * 0.5
            def set_best_epoch(c): c["best_epoch"] = np.int64(int(c["best_epoch"]) + 1)
            def set_ids(c): c["image_ids"] = np.array([c["image_ids"][0]] * len(c["image_ids"]), dtype=str)

            for label, edit in (("probs with the wrong shape", set_probs_shape), ("a wrong class order", set_classes),
                                ("a label outside 0-6", set_label), ("probabilities that do not sum to 1", set_probs),
                                ("a best epoch that differs from run_config.json", set_best_epoch), ("duplicate image ids", set_ids)):
                td = d("dmg2_" + label.split()[1])
                shutil.copytree(base_cv, td)
                _v10_edit_npz(os.path.join(model_dir(td, "multimodal"), target), edit)
                try:
                    _v10_quiet(collect_results, training_dir=td, save=False, print_tables=False)
                    failures.append(f"Damaged files: {label} was accepted"); ok = False
                except ResultsError as exc:
                    if "fold_2_val_predictions" not in str(exc):
                        failures.append(f"Damaged files: the refusal for {label} does not name the file: {exc}"); ok = False
            td = d("dmg2_missing")
            shutil.copytree(base_cv, td)
            os.remove(os.path.join(model_dir(td, "ablation"), target))
            try:
                _v10_quiet(collect_results, training_dir=td, save=False, print_tables=False)
                failures.append("Damaged files: a missing predictions file was accepted"); ok = False
            except ResultsError as exc:
                if "fold_2_val_predictions" not in str(exc):
                    failures.append(f"Damaged files: the refusal for a missing file does not name it: {exc}"); ok = False
            td = d("dmg2_name")
            shutil.copytree(base_cv, td)
            cfg = _v10_read(run_config_path(td))
            cfg["results"]["per_fold"]["multimodal"]["2"]["predictions_file"] = "fold_3_val_predictions.npz"
            _v10_write(run_config_path(td), cfg)
            try:
                _v10_quiet(collect_results, training_dir=td, save=False, print_tables=False)
                failures.append("Damaged files: a recorded predictions_file that belongs to another fold was accepted"); ok = False
            except ResultsError as exc:
                if "predictions_file" not in str(exc):
                    failures.append(f"Damaged files: the refusal for a wrong file name does not say what is wrong: {exc}"); ok = False
            print("  OK -- each kind of damage refused, with the file named" if ok else "  FAILED -- see summary")
        except Exception as exc:                                       # noqa: BLE001 -- a crash is a finding, not the end
            failures.append(f"Check 2: crashed -- {type(exc).__name__}: {exc}")
            print("  CRASHED -- see summary")

        # --- 3. Selection ---
        try:
            print("\n[3/8] Selection...")
            ok = True
            chosen = select_best_fold(base_results)
            second = sorted((f for f in folds_all if f != sel), key=lambda f: (-mm_auc[f], f))[0]
            if chosen["selected_fold"] != sel or chosen["val_auc_roc"] != mm_auc[sel]:
                failures.append(f"Selection: chose fold {chosen['selected_fold']}, the best fold by hand is {sel}"); ok = False
            if chosen["runner_up_fold"] != second or abs(chosen["margin_to_runner_up"] - (mm_auc[sel] - mm_auc[second])) > 1e-12:
                failures.append("Selection: runner-up fold or margin is wrong"); ok = False
            if chosen["all_fold_val_auc_roc"] != {str(f): mm_auc[f] for f in folds_all}:
                failures.append("Selection: the five recorded scores are not the run's scores"); ok = False

            forced = json.loads(json.dumps(base_results))
            forced["per_fold"]["multimodal"]["2"]["metrics"]["auc_roc"] = 0.99
            if select_best_fold(forced)["selected_fold"] != 2:
                failures.append("Selection: a fold forced to the highest AUC-ROC was not chosen"); ok = False
            tie = json.loads(json.dumps(base_results))
            for f in (1, 4):
                tie["per_fold"]["multimodal"][str(f)]["metrics"]["auc_roc"] = 0.995
            tied = select_best_fold(tie)
            if tied["selected_fold"] != 1 or tied["tied_folds"] != [1, 4]:
                failures.append(f"Selection: an exact tie between folds 1 and 4 chose fold {tied['selected_fold']} (tied {tied['tied_folds']})"); ok = False
            unfinished = json.loads(json.dumps(base_results))
            unfinished["complete"] = False
            try:
                select_best_fold(unfinished)
                failures.append("Selection: an incomplete run was accepted"); ok = False
            except ResultsError:
                pass
            gap = json.loads(json.dumps(base_results))
            del gap["per_fold"]["multimodal"]["3"]
            try:
                select_best_fold(gap)
                failures.append("Selection: a run with a missing multimodal fold was accepted"); ok = False
            except ResultsError:
                pass
            print(f"  OK -- best fold ({sel}) chosen with runner-up and margin; a tie goes to the lower fold; an incomplete run "
                  f"or a missing fold is refused" if ok else "  FAILED -- see summary")
        except Exception as exc:                                       # noqa: BLE001 -- a crash is a finding, not the end
            failures.append(f"Check 3: crashed -- {type(exc).__name__}: {exc}")
            print("  CRASHED -- see summary")

        # --- 4. Freeze ---
        try:
            print("\n[4/8] Freeze...")
            ok = True
            final4 = d("freeze4_final")
            out = _v10_quiet(select_and_freeze, encoder_record=enc, training_dir=base_cv, final_dir=final4,
                             make_val_dataset=make_val, build_model_fn=make_model)
            frozen4 = os.path.join(final4, FROZEN_MODEL_FILENAME)
            if not os.path.exists(frozen4) or sha256_of_file(frozen4) != sha256_of_file(source_of(base_cv, sel)) \
                    or _v10_bytes(frozen4) != _v10_bytes(source_of(base_cv, sel)):
                failures.append("Freeze: the frozen file is not byte-identical to the selected fold's best weights"); ok = False
            card = _v10_read(os.path.join(final4, MODEL_CARD_FILENAME))
            for part in ("format_version", "frozen_at", "last_checked_at", "description", "model", "selection", "frozen_file",
                         "input_contract", "cross_fold_summary", "training_settings", "environment", "reload_check", "downstream_rule"):
                if part not in card:
                    failures.append(f"Freeze: the model card has no '{part}'"); ok = False
            if card.get("selection", {}).get("selected_fold") != sel or \
                    card.get("frozen_file", {}).get("sha256") != sha256_of_file(frozen4):
                failures.append("Freeze: the card's selected fold or SHA-256 is wrong"); ok = False
            meta = card.get("input_contract", {}).get("metadata", {})
            if meta.get("age", {}).get("mean") != enc["folds"][str(sel)]["mean"] or meta.get("age", {}).get("std") != enc["folds"][str(sel)]["std"]:
                failures.append("Freeze: the card's age mean / std are not the selected fold's"); ok = False
            if meta.get("feature_names") != list(FEATURE_NAMES) or card.get("model", {}).get("grad_cam_layer") != "model.backbone[-1]":
                failures.append("Freeze: the card's feature names or Grad-CAM layer are wrong"); ok = False
            if "FROZEN" not in card.get("downstream_rule", "") or card.get("reload_check", {}).get("passed") is not True:
                failures.append("Freeze: the card lacks the downstream rule or a passed reload check"); ok = False

            before_bytes, before_card = _v10_bytes(frozen4), card
            time.sleep(1.1)                                                # so a re-written freeze date could not look the same
            again = _v10_quiet(select_and_freeze, encoder_record=enc, training_dir=base_cv, final_dir=final4,
                               make_val_dataset=make_val, build_model_fn=make_model)
            if _v10_bytes(frozen4) != before_bytes or "already frozen" not in again["frozen"]["status"] \
                    or _v10_read(os.path.join(final4, MODEL_CARD_FILENAME))["frozen_at"] != before_card["frozen_at"]:
                failures.append("Freeze: a second call changed the frozen file, did not recognise it, or changed the freeze date"); ok = False

            final4b = d("freeze4b_final")
            os.makedirs(final4b)
            stale = os.path.join(final4b, FROZEN_MODEL_FILENAME)
            shutil.copy(frozen4, stale)
            with open(stale, "ab") as f:
                f.write(b"x")
            stale_bytes = _v10_bytes(stale)
            try:
                _v10_quiet(select_and_freeze, encoder_record=enc, training_dir=base_cv, final_dir=final4b,
                           make_val_dataset=make_val, build_model_fn=make_model)
                failures.append("Freeze: a different file already frozen was overwritten or accepted"); ok = False
            except FrozenModelError:
                if _v10_bytes(stale) != stale_bytes or os.path.exists(os.path.join(final4b, MODEL_CARD_FILENAME)):
                    failures.append("Freeze: the refusal still touched the existing file or wrote a card"); ok = False

            def corrupt_copy(real):
                def wrapped(source, destination):
                    real(source, destination)
                    with open(destination, "ab") as f:
                        f.write(b"x")                                      # a copy that does not match its source
                return wrapped

            final4c = d("freeze4c_final")
            with _v10_patch_function("_copy_atomic", corrupt_copy):
                try:
                    _v10_quiet(select_and_freeze, encoder_record=enc, training_dir=base_cv, final_dir=final4c,
                               make_val_dataset=make_val, build_model_fn=make_model)
                    failures.append("Freeze: a copy that does not match its source was accepted"); ok = False
                except FrozenModelError:
                    if os.path.exists(os.path.join(final4c, FROZEN_MODEL_FILENAME)):
                        failures.append("Freeze: a copy that does not match its source was left in place"); ok = False

            for label, edit, expect in (("the wrong identity", lambda s: s["identity"].update(fold=(sel + 1) % N_FOLDS), (ResultsError, CheckpointError)),
                                        ("the wrong epoch", lambda s: s.update(epoch=s["epoch"] + 1), (FrozenModelError,))):
                td, fd = d("freeze4_" + label.split()[-1] + "_cv"), d("freeze4_" + label.split()[-1] + "_final")
                shutil.copytree(base_cv, td)
                _v10_edit_checkpoint(source_of(td, sel), edit)
                try:
                    _v10_quiet(select_and_freeze, encoder_record=enc, training_dir=td, final_dir=fd,
                               make_val_dataset=make_val, build_model_fn=make_model)
                    failures.append(f"Freeze: a source file with {label} was accepted"); ok = False
                except expect:
                    if os.path.exists(os.path.join(fd, FROZEN_MODEL_FILENAME)):
                        failures.append(f"Freeze: a frozen file was written from a source with {label}"); ok = False
            td, fd = d("freeze4_inc_cv"), d("freeze4_inc_final")
            shutil.copytree(base_cv, td)
            cfg = _v10_read(run_config_path(td))
            del cfg["results"]["per_fold"]["ablation"]["4"]
            cfg["results"]["complete"] = False
            _v10_write(run_config_path(td), cfg)
            try:
                _v10_quiet(select_and_freeze, encoder_record=enc, training_dir=td, final_dir=fd,
                           make_val_dataset=make_val, build_model_fn=make_model)
                failures.append("Freeze: an incomplete run was frozen"); ok = False
            except ResultsError:
                if os.path.exists(fd) and os.listdir(fd):
                    failures.append("Freeze: files were written for an incomplete run"); ok = False
            print("  OK -- byte-identical frozen file + complete card; a second call changes nothing; a different existing "
                  "file is never overwritten; a corrupted copy is caught and removed; wrong identity / wrong epoch / incomplete run "
                  "refused, nothing written" if ok else "  FAILED -- see summary")
        except Exception as exc:                                       # noqa: BLE001 -- a crash is a finding, not the end
            failures.append(f"Check 4: crashed -- {type(exc).__name__}: {exc}")
            print("  CRASHED -- see summary")

        # --- 5. Reload check ---
        try:
            print("\n[5/8] Reload check...")
            ok = True
            record = base_results["per_fold"]["multimodal"][str(sel)]
            pred_path = os.path.join(model_dir(base_cv, "multimodal"), record["predictions_file"])

            def reload(frozen=frozen_base, metrics=record["metrics"], **kw):
                args = dict(encoder_record=enc, make_val_dataset=make_val, build_model_fn=make_model)
                args.update(kw)
                return _v10_quiet(reload_check, frozen, sel, metrics, pred_path, **args)

            clean = reload()
            if not clean["passed"] or clean["max_abs_probability_difference"] > 1e-4 or not (clean["image_ids_match"] and clean["labels_match"]):
                failures.append(f"Reload check: the real freeze did not pass cleanly ({clean['problems']})"); ok = False
            altered = d("altered.pt")
            shutil.copy(frozen_base, altered)
            _v10_edit_checkpoint(altered, lambda s: s["model_state_dict"].update(
                {k: v + 0.5 for k, v in s["model_state_dict"].items() if v.dim() == 2 and v.dtype.is_floating_point}))
            if reload(frozen=altered)["passed"]:
                failures.append("Reload check: altered weights passed"); ok = False
            res_rev = reload(make_val_dataset=lambda f: _V10ReversedView(make_val(f)))
            if res_rev["passed"] or not any("order" in p for p in res_rev["problems"]):
                failures.append("Reload check: a different validation order passed or was not described"); ok = False
            res_fit = reload(build_model_fn=lambda: _V10TinyNet(5))
            if res_fit["passed"] or not any("do not fit" in p for p in res_fit["problems"]):
                failures.append("Reload check: weights that do not fit the model passed or were not described"); ok = False
            res_lab = reload(make_val_dataset=lambda f: _V10RelabelView(make_val(f)))
            if res_lab["passed"] or not any("labels" in p for p in res_lab["problems"]):
                failures.append("Reload check: different validation labels passed or were not described"); ok = False
            sharpened = d("sharpened_predictions.npz")
            shutil.copy(pred_path, sharpened)

            def sharpen(c):                                                # rows still sum to 1; only the probabilities move
                p = c["probs"].astype(np.float64)
                c["probs"] = (0.7 * p + 0.3 * np.eye(p.shape[1])[p.argmax(axis=1)]).astype(np.float32)

            _v10_edit_npz(sharpened, sharpen)
            res_prob = _v10_quiet(reload_check, frozen_base, sel, record["metrics"], sharpened, encoder_record=enc,
                                  make_val_dataset=make_val, build_model_fn=make_model)
            if res_prob["passed"] or not any("probability" in p for p in res_prob["problems"]):
                failures.append("Reload check: probabilities that differ from the saved ones by up to 0.3 passed or were not described"); ok = False
            off = dict(record["metrics"], auc_roc=record["metrics"]["auc_roc"] + 0.01)
            res_auc = reload(metrics=off)
            if res_auc["passed"] or not any("AUC-ROC" in p for p in res_auc["problems"]):
                failures.append("Reload check: an AUC-ROC that differs from the record by 0.01 passed or was not described"); ok = False
            print("  OK -- passes on the real freeze; fails for altered weights, a different image order, weights that do not "
                  "fit, different labels, probabilities far from the saved ones, and an AUC that does not match" if ok else "  FAILED -- see summary")
        except Exception as exc:                                       # noqa: BLE001 -- a crash is a finding, not the end
            failures.append(f"Check 5: crashed -- {type(exc).__name__}: {exc}")
            print("  CRASHED -- see summary")

        # --- 6. Gate, clean run ---
        try:
            print("\n[6/8] Exit gate on the clean run (with the live BatchNorm check)...")
            ok = True
            td, fd = fresh("clean")
            report = _v10_quiet(run_final_gate, encoder_record=enc, training_dir=td, final_dir=fd, make_val_dataset=make_val,
                                build_model_fn=make_model, bn_check=True, save=True, raise_on_failure=False,
                                locked_settings=locked, allowed_max_epochs=(MAX_EPOCHS,))
            if not report["passed"] or report["counts"]["FAIL"] or report["counts"]["SKIPPED"]:
                failures.append(f"Gate: the clean run did not pass: {[(c['id'], c['status'], c['detail']) for c in report['checks'] if c['status'] not in ('PASS', 'NOTE')]}"); ok = False
            if status_of(report, "B3") != "NOTE":
                failures.append("Gate: a maximum-epochs value that differs from the Checklist was not reported as a NOTE"); ok = False
            expected_ids = {"A1", "A2", "A3", "A4", "A5", "A6", "A7", "A8", "A9", "B1", "B2", "B3", "B4",
                            "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "D1", "D2"}
            if {c["id"] for c in report["checks"]} != expected_ids:
                failures.append("Gate: the set of checks that ran is not A1-A9, B1-B4, C1-C8, D1-D2"); ok = False
            saved = _v10_read(os.path.join(fd, GATE_FILENAME)) if os.path.exists(os.path.join(fd, GATE_FILENAME)) else {}
            if saved.get("passed") is not True or len(saved.get("documented_deviations", [])) != len(DOCUMENTED_DEVIATIONS) \
                    or not (saved.get("batchnorm_evidence") or {}).get("live_check", {}).get("passed"):
                failures.append("Gate: the saved report is missing, or lacks the verdict, the deviations list or the BatchNorm evidence"); ok = False

            cfg = _v10_read(run_config_path(td))
            cfg["environment_changes"] = [{"at": "now", "differences": {"torch": {"was": "a", "now": "b"}}}]
            _v10_write(run_config_path(td), cfg)
            noted = gate(td, fd)
            if status_of(noted, "B4") != "NOTE" or "B4" in failed(noted):
                failures.append("Gate: a recorded environment change was not a NOTE (or made the gate fail)"); ok = False
            skipped = gate(td, fd, bn=False)
            if skipped["passed"] or status_of(skipped, "D1") != "SKIPPED":
                failures.append("Gate: a gate run without the BatchNorm check counted as a pass"); ok = False
            try:
                gate(td, fd, bn=False, raise_on_failure=True)
                failures.append("Gate: a gate that did not pass did not raise"); ok = False
            except RuntimeError:
                pass
            print("  OK -- 23 checks ran, all pass (one NOTE: maximum epochs); report file written; an environment change is "
                  "a NOTE; a gate without the BatchNorm check is not a pass and raises" if ok else "  FAILED -- see summary")
        except Exception as exc:                                       # noqa: BLE001 -- a crash is a finding, not the end
            failures.append(f"Check 6: crashed -- {type(exc).__name__}: {exc}")
            print("  CRASHED -- see summary")

        # --- 7. Gate, damage ---
        try:
            print("\n[7/8] Exit gate on damaged copies of the run...")
            ok = True
            other = (sel + 1) % N_FOLDS

            def m_tamper(td, fd):
                with open(os.path.join(fd, FROZEN_MODEL_FILENAME), "ab") as f:
                    f.write(b"\0")

            def m_frozen_missing(td, fd):
                os.remove(os.path.join(fd, FROZEN_MODEL_FILENAME))

            def m_card_missing(td, fd):
                os.remove(os.path.join(fd, MODEL_CARD_FILENAME))

            def edit_card(fd, edit):
                path = os.path.join(fd, MODEL_CARD_FILENAME)
                card = _v10_read(path)
                edit(card)
                _v10_write(path, card)

            def m_card_fold(td, fd):
                edit_card(fd, lambda c: c["selection"].update(selected_fold=other))

            def m_card_scores(td, fd):
                edit_card(fd, lambda c: c["selection"]["all_fold_val_auc_roc"].update({"1": c["selection"]["all_fold_val_auc_roc"]["1"] + 0.01}))

            def m_encoder(td, fd):
                changed = json.loads(json.dumps(enc))
                changed["folds"][str(sel)]["mean"] += 1.0
                return {"encoder_record": changed}

            def m_pred_missing(td, fd):
                os.remove(os.path.join(model_dir(td, "multimodal"), "fold_1_val_predictions.npz"))

            def edit_config(td, edit):
                cfg = _v10_read(run_config_path(td))
                edit(cfg)
                _v10_write(run_config_path(td), cfg)

            def m_metric(td, fd):
                edit_config(td, lambda c: c["results"]["per_fold"]["ablation"]["2"]["metrics"].update(
                    f1=c["results"]["per_fold"]["ablation"]["2"]["metrics"]["f1"] + 0.01))

            def m_ids_swapped(td, fd):
                def swap(c):
                    ids = c["image_ids"].copy()
                    ids[0], ids[1] = ids[1], ids[0]
                    c["image_ids"] = ids
                _v10_edit_npz(os.path.join(model_dir(td, "ablation"), "fold_0_val_predictions.npz"), swap)

            def m_rows_cut(td, fd):
                def cut(c):
                    for key in ("image_ids", "labels", "probs"):
                        c[key] = c[key][:-1]
                _v10_edit_npz(os.path.join(model_dir(td, "multimodal"), "fold_1_val_predictions.npz"), cut)

            def m_max_epochs(td, fd):
                return {"allowed_max_epochs": (50, 100)}

            def m_setting(td, fd):
                edit_config(td, lambda c: c["settings"]["training"].update(backbone_lr=5e-4))

            def m_unfinished(td, fd):
                _v10_edit_checkpoint(checkpoint_paths(model_dir(td, "ablation"), 4)[0], lambda s: s.update(finished=False))

            def m_phase(td, fd):
                _v10_edit_checkpoint(checkpoint_paths(model_dir(td, "multimodal"), 0)[0],
                                     lambda s: s["history"][0].update(phase="B"))

            def m_best_epoch(td, fd):
                _v10_edit_checkpoint(checkpoint_paths(model_dir(td, "ablation"), 1)[1], lambda s: s.update(epoch=s["epoch"] + 1))

            def m_best_missing(td, fd):
                os.remove(checkpoint_paths(model_dir(td, "multimodal"), 0)[1])

            def m_incomplete(td, fd):
                def cut(c):
                    del c["results"]["per_fold"]["ablation"]["4"]
                    c["results"]["complete"] = False
                edit_config(td, cut)

            def m_ids_overlap(td, fd):
                def overlap(c):
                    ids = c["image_ids"].copy()
                    ids[0] = "va0_000"                                     # an image that fold 0 already validated
                    c["image_ids"] = ids
                _v10_edit_npz(os.path.join(model_dir(td, "multimodal"), "fold_1_val_predictions.npz"), overlap)

            def m_labels_differ(td, fd):
                def swap(c):
                    labels = c["labels"].copy()
                    labels[0], labels[1] = labels[1], labels[0]
                    c["labels"] = labels
                _v10_edit_npz(os.path.join(model_dir(td, "ablation"), "fold_0_val_predictions.npz"), swap)

            def m_stored_summary(td, fd):
                def bump(c):
                    entry = c["results"]["mean_sd"]["multimodal"]["auc_roc"]
                    entry["mean"] = entry["mean"] + 0.01
                edit_config(td, bump)

            def resave_frozen(fd, edit):
                frozen = os.path.join(fd, FROZEN_MODEL_FILENAME)
                _v10_edit_checkpoint(frozen, edit)
                edit_card(fd, lambda c: c["frozen_file"].update(sha256=sha256_of_file(frozen)))   # the card follows the file

            def m_resaved(td, fd):
                resave_frozen(fd, lambda s: s.update(note="re-saved"))

            def m_frozen_epoch(td, fd):
                resave_frozen(fd, lambda s: s.update(epoch=s["epoch"] + 1))

            def m_weights_altered(td, fd):
                resave_frozen(fd, lambda s: s["model_state_dict"].update(
                    {k: v + 0.5 for k, v in s["model_state_dict"].items() if v.dim() == 2 and v.dtype.is_floating_point}))

            def m_card_rule(td, fd):
                edit_card(fd, lambda c: c.update(downstream_rule=""))

            def m_epochs_run(td, fd):
                edit_config(td, lambda c: c["results"]["per_fold"]["ablation"]["3"].update(epochs_run=5))

            def m_pool(td, fd):
                edit_config(td, lambda c: c["settings"]["data"].update(cv_pool_rows=141))

            def m_card_hash(td, fd):
                edit_card(fd, lambda c: c["frozen_file"].update(sha256="0" * 64))

            def m_config_unreadable(td, fd):
                with open(run_config_path(td), "w") as f:
                    f.write("{not json")

            cases = [
                ("the frozen file changed after freezing", {"C4"}, m_tamper),
                ("the frozen file is missing", {"C4"}, m_frozen_missing),
                ("the model card is missing", {"C1"}, m_card_missing),
                ("the card names a different fold", {"C3"}, m_card_fold),
                ("the card's fold scores were edited", {"C2"}, m_card_scores),
                ("the metadata statistics differ from the card's", {"C7"}, m_encoder),
                ("a predictions file is missing", {"A5"}, m_pred_missing),
                ("a recorded metric was altered", {"A6"}, m_metric),
                ("two image ids swapped in one model's predictions", {"A8"}, m_ids_swapped),
                ("a validation row is missing from one fold", {"A7"}, m_rows_cut),
                ("undocumented maximum epochs", {"B3"}, m_max_epochs),
                ("a locked setting drifted (backbone learning rate)", {"B1"}, m_setting),
                ("a run is not marked finished", {"A3"}, m_unfinished),
                ("a wrong phase in a run's history", {"D2"}, m_phase),
                ("a best-weights file holds another epoch", {"A4"}, m_best_epoch),
                ("a best-weights file is missing", {"A4"}, m_best_missing),
                ("a finished fold is missing from run_config.json", {"A2"}, m_incomplete),
                ("run_config.json is unreadable", {"A1"}, m_config_unreadable),
                ("an image validated in two folds", {"A7"}, m_ids_overlap),
                ("the two models' labels differ in a fold", {"A8"}, m_labels_differ),
                ("run_config.json's stored mean +/- SD was edited", {"A9"}, m_stored_summary),
                ("the frozen file is a different file than the selected weights (card updated to match)", {"C4"}, m_resaved),
                ("the frozen file holds another epoch (card updated to match)", {"C4", "C5"}, m_frozen_epoch),
                ("the frozen file's weights differ (card updated to match)", {"C4", "C6"}, m_weights_altered),
                ("the card's downstream rule is empty", {"C8"}, m_card_rule),
                ("a run's epoch count in run_config.json differs from its history", {"D2"}, m_epochs_run),
                ("the CV pool size recorded is not the Phase 1 pool", {"B2"}, m_pool),
                ("the card's SHA-256 is not the frozen file's", {"C4"}, m_card_hash),
            ]
            for i, (label, expect, mutate) in enumerate(cases):
                td, fd = fresh(f"dmg7_{i}")
                try:
                    over = mutate(td, fd) or {}
                    rep = gate(td, fd, **over)
                except Exception as exc:                                   # noqa: BLE001 -- the gate must report damage, never crash on it
                    failures.append(f"Gate damage: '{label}' made the gate crash -- {type(exc).__name__}: {exc}"); ok = False
                    continue
                if rep["passed"] or not expect <= failed(rep):
                    failures.append(f"Gate damage: '{label}' -- expected {sorted(expect)} to fail, got {sorted(failed(rep))} "
                                    f"(passed={rep['passed']})"); ok = False
            print(f"  OK -- all {len(cases)} kinds of damage make the gate fail on the check meant to catch them" if ok else "  FAILED -- see summary")
        except Exception as exc:                                       # noqa: BLE001 -- a crash is a finding, not the end
            failures.append(f"Check 7: crashed -- {type(exc).__name__}: {exc}")
            print("  CRASHED -- see summary")

        # --- 8. BatchNorm check ---
        try:
            print("\n[8/8] BatchNorm freeze check (real architecture, 64x64 random images)...")
            ok = True
            live = batchnorm_freeze_check()
            a = live["models"]["multimodal"]["A"]
            if not live["passed"] or a["backbone_bn_layers"] <= 0 or a["backbone_bn_layers_changed"] != 0 or a["backbone_params_changed"] != 0:
                failures.append(f"BatchNorm check: the real set_phase did not pass cleanly ({live['problems']})"); ok = False

            def forgets_eval(real):
                def wrapped(model, phase):
                    real(model, phase)
                    if phase == "A":
                        model.backbone.train()                             # the trap: BatchNorm statistics keep updating
                return wrapped

            def leaves_trainable(real):
                def wrapped(model, phase):
                    real(model, phase)
                    if phase == "A":
                        for p in model.backbone.parameters():
                            p.requires_grad = True                         # backbone weights not frozen
                return wrapped

            def never_unfreezes(real):
                def wrapped(model, phase):
                    real(model, phase)
                    if phase == "B":
                        for p in model.backbone.parameters():
                            p.requires_grad = False                        # Phase B never releases the backbone
                return wrapped

            def partly_unfreezes(real):
                def wrapped(model, phase):
                    real(model, phase)
                    if phase == "B":
                        for p in list(model.backbone.parameters())[::2]:
                            p.requires_grad = False                        # only half of the backbone is released
                return wrapped

            for label, make, word in (("Phase A forgetting backbone.eval()", forgets_eval, "Phase A"),
                                      ("Phase A leaving the backbone weights trainable", leaves_trainable, "Phase A"),
                                      ("Phase B never unfreezing", never_unfreezes, "Phase B"),
                                      ("Phase B releasing only part of the backbone", partly_unfreezes, "Phase B")):
                with _v10_patch_function("set_phase", make):
                    broken = batchnorm_freeze_check()
                if broken["passed"] or not any(word in p for p in broken["problems"]):
                    failures.append(f"BatchNorm check: {label} was not detected ({broken['problems']})"); ok = False
            td, fd = fresh("bn8")
            with _v10_patch_function("set_phase", forgets_eval):
                rep = gate(td, fd, bn=True)
            if rep["passed"] or "D1" not in failed(rep):
                failures.append("BatchNorm check: the gate did not fail on D1 when the freeze was broken"); ok = False
            print("  OK -- the real set_phase passes; a forgotten eval(), a trainable frozen backbone and a Phase B that never "
                  "unfreezes or only partly unfreezes are each detected, also through the gate" if ok else "  FAILED -- see summary")
        except Exception as exc:                                       # noqa: BLE001 -- a crash is a finding, not the end
            failures.append(f"Check 8: crashed -- {type(exc).__name__}: {exc}")
            print("  CRASHED -- see summary")

    print("\n" + "=" * 60)
    if failures:
        print(f"VERIFICATION FAILED -- {len(failures)} issue(s):")
        for msg in failures:
            print(f"  - {msg}")
        print("=" * 60)
        raise RuntimeError(f"Module 10 verification failed with {len(failures)} issue(s).")
    print("VERIFICATION PASSED -- all checks OK. Module 10 is safe to run on the finished training.")
    print("=" * 60)


# --- Usage ---
# run_module10_verification()

In [51]:
run_module10_verification()

MODULE 10 VERIFICATION -- results, selection, freeze, exit gate
  real run_cv_training + real Module 10 code, CPU, tiny stand-in models, synthetic data, temporary folder

[1/8] Result tables...
  OK -- row counts; mean +/- SD and paired difference match by-hand values; per-class values average back to the fold metrics; CSVs read back equal; columns renameable; partial runs work

[2/8] Damaged result files are refused, naming the file...
  OK -- each kind of damage refused, with the file named

[3/8] Selection...
  OK -- best fold (3) chosen with runner-up and margin; a tie goes to the lower fold; an incomplete run or a missing fold is refused

[4/8] Freeze...
  OK -- byte-identical frozen file + complete card; a second call changes nothing; a different existing file is never overwritten; a corrupted copy is caught and removed; wrong identity / wrong epoch / incomplete run refused, nothing written

[5/8] Reload check...
  OK -- passes on the real freeze; fails for altered weights, a dif

In [52]:
collect_results()


PER-FOLD RESULTS (validation folds, best-epoch weights)
  model      fold   AUC-ROC      F1    Sens    Spec     Acc   best ep  epochs  stop
  multimodal    0    0.9266  0.6248  0.6936  0.9473  0.6896     26(B)      36  early_stop
  multimodal    1    0.9444  0.7036  0.7278  0.9621  0.7990     61(B)      71  early_stop
  multimodal    2    0.9264  0.5992  0.6609  0.9494  0.7028     28(B)      38  early_stop
  multimodal    3    0.9481  0.6898  0.7339  0.9586  0.7666     51(B)      61  early_stop
  multimodal    4    0.9318  0.6473  0.6469  0.9466  0.7832     55(B)      65  early_stop
  ablation      0    0.9030  0.5325  0.6795  0.9413  0.6453     11(B)      21  early_stop
  ablation      1    0.9385  0.6091  0.6425  0.9519  0.7383     28(B)      38  early_stop
  ablation      2    0.8628  0.3071  0.5023  0.9040  0.3753      5(A)      15  early_stop
  ablation      3    0.9530  0.6688  0.7038  0.9583  0.7829     43(B)      53  early_stop
  ablation      4    0.9596  0.6844  0.7417  0.95

{'per_fold':         model  fold   auc_roc        f1  sensitivity  specificity  accuracy  \
 0  multimodal     0  0.926623  0.624790     0.693611     0.947257  0.689575   
 1  multimodal     1  0.944388  0.703612     0.727752     0.962108  0.798951   
 2  multimodal     2  0.926434  0.599210     0.660882     0.949407  0.702797   
 3  multimodal     3  0.948123  0.689754     0.733897     0.958564  0.766589   
 4  multimodal     4  0.931775  0.647300     0.646927     0.946566  0.783217   
 5    ablation     0  0.903007  0.532478     0.679532     0.941315  0.645312   
 6    ablation     1  0.938525  0.609130     0.642527     0.951895  0.738345   
 7    ablation     2  0.862776  0.307148     0.502271     0.904027  0.375291   
 8    ablation     3  0.953013  0.668763     0.703776     0.958299  0.782887   
 9    ablation     4  0.959649  0.684438     0.741665     0.957750  0.791958   
 
    best_epoch best_epoch_phase  epochs_run stop_reason  metrics_reproduce  
 0          26               

In [53]:
select_and_freeze()

SELECT AND FREEZE THE FINAL MODEL
  validation AUC-ROC per fold (multimodal): fold 0: 0.9266  fold 1: 0.9444  fold 2: 0.9264  fold 3: 0.9481  fold 4: 0.9318
  SELECTED: fold 3  (AUC-ROC 0.9481, best epoch 51 [phase B]; runner-up fold 1, margin 0.0037)
  FROZEN: copied
    file    T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\final_model\phase3_final_multimodal_model.pt  (43.8 MB)
    SHA-256 d4e369410e325820ce28833d98200ba4f3243c2be410e9a91efee8ab6709d0ed
  Reload check (fresh model + the frozen file, predicting the selected fold's validation images)...
    OK -- AUC-ROC 0.948123 (recorded 0.948123), largest probability difference 0.00e+00, 1718 images, same ids and labels
  MODEL CARD written: T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\final_model\phase3_model_card.json
DONE. Next: run_final_gate() (10_02) checks the whole phase against the Checklist.


{'selection': {'rule': "The single fold whose multimodal model has the highest validation AUC-ROC (best-epoch weights). One fold's model, not an ensemble of folds.",
  'tie_rule': 'exact tie in validation AUC-ROC -> the lower fold number',
  'selected_fold': 3,
  'val_auc_roc': 0.9481232283374819,
  'best_epoch': 51,
  'best_epoch_phase': 'B',
  'all_fold_val_auc_roc': {'0': 0.9266232633190009,
   '1': 0.9443881461545383,
   '2': 0.9264344588313859,
   '3': 0.9481232283374819,
   '4': 0.9317753861584782},
  'tied_folds': [3],
  'runner_up_fold': 1,
  'margin_to_runner_up': 0.0037350821829436054},
 'frozen': {'name': 'phase3_final_multimodal_model.pt',
  'path': 'T:\\Thesis - Development\\Resources\\Artifacts\\Phase 3 - P1\\final_model\\phase3_final_multimodal_model.pt',
  'sha256': 'd4e369410e325820ce28833d98200ba4f3243c2be410e9a91efee8ab6709d0ed',
  'bytes': 43846307,
  'copied_from': 'T:\\Thesis - Development\\Resources\\Artifacts\\Phase 3 - P1\\cv_training\\multimodal\\fold_3_best.p

In [54]:
run_final_gate()

PHASE 3 EXIT GATE -- every check reads the files on disk

A. The run is complete and intact
  [PASS   ] A1  run_config.json is readable and in the expected format
             T:\Thesis - Development\Resources\Artifacts\Phase 3 - P1\cv_training\run_config.json
  [PASS   ] A2  all 10 runs are recorded and run_config.json says complete
             10 of 10 recorded
  [PASS   ] A3  every run's resume file is readable and marked finished
             10 of 10 finished
  [PASS   ] A4  every best-weights file matches the epoch and AUC-ROC recorded for it
             10 of 10 match
  [PASS   ] A5  every predictions file loads and passes its checks
             10 files
  [PASS   ] A6  every recorded metric reproduces from its predictions file
             10 runs
  [PASS   ] A7  each model's validation folds cover the CV pool exactly once
             8583 images, each exactly once, per model
  [PASS   ] A8  multimodal and ablation predictions are for the same images and labels
            

{'format_version': 1,
 'created_at': '2026-10-01T03:22:06+06:00',
 'passed': True,
 'counts': {'PASS': 22, 'FAIL': 0, 'NOTE': 1, 'SKIPPED': 0},
 'training_dir': 'T:\\Thesis - Development\\Resources\\Artifacts\\Phase 3 - P1\\cv_training',
 'final_dir': 'T:\\Thesis - Development\\Resources\\Artifacts\\Phase 3 - P1\\final_model',
 'checks': [{'id': 'A1',
   'section': 'A',
   'name': 'run_config.json is readable and in the expected format',
   'status': 'PASS',
   'detail': 'T:\\Thesis - Development\\Resources\\Artifacts\\Phase 3 - P1\\cv_training\\run_config.json'},
  {'id': 'A2',
   'section': 'A',
   'name': 'all 10 runs are recorded and run_config.json says complete',
   'status': 'PASS',
   'detail': '10 of 10 recorded'},
  {'id': 'A3',
   'section': 'A',
   'name': "every run's resume file is readable and marked finished",
   'status': 'PASS',
   'detail': '10 of 10 finished'},
  {'id': 'A4',
   'section': 'A',
   'name': 'every best-weights file matches the epoch and AUC-ROC record